<a href="https://colab.research.google.com/github/amitkumarbharti-1234/Amitkumarportfolio/blob/main/openLearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# OpenLearn Baby - paste this WHOLE file into ONE empty Colab cell (select all, delete the old code first), then run it.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio", "pypdf", "transformers", "accelerate", "yt-dlp"], check=False)

import re, json, math, random, ast, operator, time, sqlite3, hashlib, threading, colorsys, inspect, shutil
import urllib.parse
import html as _html
from pathlib import Path
from datetime import datetime
from collections import Counter, deque
import numpy as np
from PIL import Image
import gradio as gr
from pypdf import PdfReader

# ===================== CONFIG =====================
USE_DRIVE = False   # True = keep the brain on Google Drive (survives runtime resets)
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = Path("/content/drive/MyDrive/openlearn_agent")
    except Exception as e:
        print("Drive not available, using local disk:", e)
        USE_DRIVE = False
        DATA_DIR = Path("/content/openlearn_agent")
else:
    DATA_DIR = Path("/content/openlearn_agent")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "brain.db"
LEGACY_PATH = DATA_DIR / "memory.json"
FEEDBACK_PATH = DATA_DIR / "feedback.json"

ALPHA = 0.4
RL_WEIGHT = 0.5
MIN_COV = 0.34
TOP_K = 3
DIM = 152
VIS_MIN_SIM = 0.80
DEDUP_SIM = 0.985
SELF_SIM = 0.95
MAX_PER_LABEL = 300
AUTO_EVERY = 0.4
IDLE_CAP = "Hi! Teach me something."
RESET_AFTER = 3.5
K = 72                       # points used for simulation targets

# ---- learning drop-down (singing / dancing are NAMES ONLY for now) ----
MODE_TEXT  = "💬 Text & documents"
MODE_SEE   = "👁 Seeing objects"
MODE_DRAW  = "✏️ Drawing (3D studio)"
MODE_SIM   = "🧪 Simulation world (3D)"
MODE_SING  = "🎤 Singing (coming soon)"
MODE_DANCE = "💃 Dancing (coming soon)"
MODE_FIGHT = "🥊 Fighting ring (3D)"
MODES = [MODE_TEXT, MODE_SEE, MODE_DRAW, MODE_SIM, MODE_SING, MODE_DANCE, MODE_FIGHT]
SOON = (MODE_SING, MODE_DANCE)

STOP = set("a an the is are was were be been am of to in on at for and or but with by from as it this that "
           "these those what who whom which when where why how do does did can could should would will "
           "i you he she we they me my your our their its if then so than too very about into over under "
           "has have had his her".split())

def load_json(path, default):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return default

def save_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False), encoding="utf-8")

feedback = load_json(FEEDBACK_PATH, {"good": 0, "bad": 0})
last = {"query": "", "used": []}
last_vis = {"label": None, "used": [], "feat": None}
live = {"hist": deque(maxlen=6), "prev_g": None, "last_key": None, "last_save": 0.0, "last_auto": 0.0,
        "last_flush": time.time(), "noted": set(), "prev_label": None, "surprise_until": 0.0, "feat": None,
        "arr": None, "mood": "idle", "mood_at": 0.0, "last_live": 0.0, "kind": "text", "learn_name": None,
        "learn_n": 0, "last_say": 0.0, "announced": None, "busy": False, "hold_until": 0.0, "draw_label": None}

def skip():
    return gr.skip() if hasattr(gr, "skip") else gr.update()

_nonce = [0]
def V(kind, text=""):
    """Voice/sound event for the browser: 'kind||text||counter' (counter makes every event unique)."""
    _nonce[0] += 1
    return f"{kind}||{text}||{_nonce[0]}"

def say_clean(s, n=240):
    s = re.sub(r"[*_`#>|]|\[|\]", " ", s or "")
    return re.sub(r"\s+", " ", s).strip()[:n]


# ===================== QWEN: WORDING + VOICE ONLY =====================
# Qwen never learns and never supplies knowledge. It only rewords what the robot's OWN memory found.
# If memory has nothing, the robot says it doesn't know (Qwen is not asked).
LLM = {"tok": None, "model": None, "dev": "cpu", "state": "off", "err": "", "id": ""}
USE_QWEN = [True]
LLM_LOCK = threading.Lock()

def llm_load():
    if LLM["state"] in ("loading", "ready"):
        return
    LLM["state"] = "loading"
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        mid = "Qwen/Qwen2.5-1.5B-Instruct" if dev == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
        tok = AutoTokenizer.from_pretrained(mid)
        model = AutoModelForCausalLM.from_pretrained(
            mid, torch_dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev)
        model.eval()
        LLM.update(tok=tok, model=model, dev=dev, id=mid, state="ready")
    except Exception as e:
        LLM.update(state="error", err=str(e)[:200])

_qt = threading.Timer(45.0, llm_load); _qt.daemon = True; _qt.start()     # loads quietly 45 s after start (or on first use)

QWEN_RULES = ("You are only the speaking voice of a small robot. You have NO knowledge of your own. "
              "Answer the user's message using ONLY the FACTS given. Reply in one or two short, friendly sentences. "
              "Never add, guess, explain or correct anything that is not written in the FACTS.")

CHAT_RULES = ("You are a friendly little learning robot chatting by voice with your teacher. Reply in one or two SHORT, warm sentences "
              "and usually end with a short friendly question that continues the conversation. Talk about feelings, greetings, jokes, "
              "the user's plans and your abilities (you learn from text, listening, pictures, drawings and fighting moves). "
              "NEVER state facts about the world. If the user asks a factual question you do not have in FACTS, say you do not know yet "
              "and invite them to teach you by saying: remember this, and then the fact.")

def _llm_generate(msgs, max_new=70):
    import torch
    tok, model = LLM["tok"], LLM["model"]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(LLM["dev"])
    with LLM_LOCK, torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new, do_sample=True, temperature=0.6, top_p=0.9, repetition_penalty=1.1)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def chat_llm(message, history, facts=None):
    """Open conversation (feelings, small talk, follow-up questions). Returns text or None when Qwen is not ready."""
    if not USE_QWEN[0]:
        return None
    if LLM["state"] == "off":
        threading.Thread(target=llm_load, daemon=True).start()
        return None
    if LLM["state"] != "ready":
        return None
    sysmsg = CHAT_RULES + (("\nFACTS you may use:\n- " + "\n- ".join(facts)) if facts else "")
    msgs = [{"role": "system", "content": sysmsg}]
    for h in (history or [])[-8:]:
        c = re.sub(r"[*_`#>|]", " ", str(h.get("content", ""))); c = re.sub(r"\s+", " ", c).strip()[:200]
        if c and h.get("role") in ("user", "assistant"):
            msgs.append({"role": h["role"], "content": c})
    msgs.append({"role": "user", "content": message})
    try:
        out = _llm_generate(msgs)
    except Exception:
        return None
    out = re.sub(r"\s+", " ", out).strip()
    out = " ".join(re.split(r"(?<=[.!?])\s+", out)[:2])[:260]
    if not out or re.search(r"as an ai|language model|i am qwen|alibaba|i'm qwen", out, re.I):
        return None
    return out

def phrase(question, facts):
    """Reword the memory facts into a natural reply (used for both the text and the voice)."""
    raw = facts[0] if len(facts) == 1 else facts[0] + "\n\n" + "\n".join("· " + f for f in facts[1:])
    if LLM["state"] == "off":
        threading.Thread(target=llm_load, daemon=True).start()
    if not USE_QWEN[0] or LLM["state"] != "ready":
        return raw
    try:
        import torch
        tok, model = LLM["tok"], LLM["model"]
        msgs = [{"role": "system", "content": QWEN_RULES},
                {"role": "user", "content": "FACTS:\n" + "\n".join("- " + f for f in facts[:3])
                                            + "\n\nUSER MESSAGE: " + question}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors="pt").to(LLM["dev"])
        with LLM_LOCK, torch.no_grad():
            out = model.generate(**inp, max_new_tokens=90, do_sample=False, repetition_penalty=1.05)
        res = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        # guard: if Qwen brought in many words that are not in the facts/question, ignore it
        got = token_set(res)
        allowed = token_set(" ".join(facts) + " " + question)
        if not res or (got and len(got - allowed) / len(got) > 0.5):
            return raw
        return res
    except Exception:
        return raw

def qwen_toggle(v):
    USE_QWEN[0] = bool(v)
    if not v:
        return "Qwen off — I answer with my raw memory."
    st = LLM["state"]
    if st == "ready":
        return f"🤖 Qwen ready ({LLM['id']}) — it only words my answers, it teaches me nothing."
    if st == "error":
        return f"Qwen failed to load: {LLM['err']}"
    return "🤖 Qwen is still loading… I use raw memory until it is ready."

# ===================== DATABASE =====================
LOCK = threading.RLock()
DB = None
HAS_FTS = True
_kc = {}

def ensure_schema():
    global HAS_FTS
    DB.executescript("""
    CREATE TABLE IF NOT EXISTS facts(id INTEGER PRIMARY KEY, text TEXT, source TEXT, h TEXT UNIQUE, updated TEXT);
    CREATE TABLE IF NOT EXISTS q(k TEXT PRIMARY KEY, v REAL);
    CREATE TABLE IF NOT EXISTS visual(id INTEGER PRIMARY KEY, label TEXT, feat BLOB, q REAL DEFAULT 0,
                                      hits INTEGER DEFAULT 0, updated TEXT);
    CREATE INDEX IF NOT EXISTS visual_label ON visual(label);
    CREATE TABLE IF NOT EXISTS skills(name TEXT PRIMARY KEY, v REAL DEFAULT 0, eps INTEGER DEFAULT 0, w TEXT);
    CREATE TABLE IF NOT EXISTS sketches(name TEXT PRIMARY KEY, data TEXT, n INTEGER DEFAULT 0);
    CREATE TABLE IF NOT EXISTS moves3d(name TEXT PRIMARY KEY, data TEXT, n INTEGER DEFAULT 0, dur REAL, prev TEXT, miss TEXT, score REAL DEFAULT 0, src TEXT);
    """)
    try:
        DB.execute("CREATE VIRTUAL TABLE IF NOT EXISTS facts_fts USING fts5(text, tokenize='porter unicode61')")
        HAS_FTS = True
    except sqlite3.OperationalError:
        HAS_FTS = False
    DB.commit()

def connect_db():
    global DB
    DB = sqlite3.connect(str(DB_PATH), check_same_thread=False)
    if not USE_DRIVE:
        DB.execute("PRAGMA journal_mode=WAL")
    DB.execute("PRAGMA synchronous=NORMAL")
    ensure_schema()

connect_db()

# ===================== TEXT MEMORY =====================
def stem(w):
    if len(w) > 3 and w.endswith("s") and not w.endswith("ss"):
        return w[:-1]
    return w

def token_set(s):
    return {stem(w) for w in re.findall(r"[a-z0-9_]+", (s or "").lower()) if w not in STOP}

def norm_key(s):
    return hashlib.md5(" ".join(re.findall(r"[a-z0-9]+", s.lower())).encode()).hexdigest()

def chunk_text(text, max_len=0):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text or "") if len(s.strip()) > 2]
    chunks, cur = [], ""
    for s in sents:
        if cur and len(cur) + len(s) + 1 > max_len:
            chunks.append(cur); cur = s
        else:
            cur = (cur + " " + s).strip()
    if cur:
        chunks.append(cur)
    return chunks

def _add_pieces(pieces, source):
    now = datetime.now().isoformat(timespec="seconds")
    ids, new = [], 0
    with LOCK:
        for p in pieces:
            h = norm_key(p)
            row = DB.execute("SELECT id FROM facts WHERE h=?", (h,)).fetchone()
            if row:
                fid = row[0]
                DB.execute("UPDATE facts SET text=?, source=?, updated=? WHERE id=?", (p, source, now, fid))
                if HAS_FTS:
                    DB.execute("UPDATE facts_fts SET text=? WHERE rowid=?", (p, fid))
            else:
                fid = DB.execute("INSERT INTO facts(text,source,h,updated) VALUES(?,?,?,?)",
                                 (p, source, h, now)).lastrowid
                if HAS_FTS:
                    DB.execute("INSERT INTO facts_fts(rowid,text) VALUES(?,?)", (fid, p))
                new += 1
            ids.append(fid)
        DB.commit()
    _kc.clear()
    return ids, new

def count_facts():
    with LOCK:
        return DB.execute("SELECT COUNT(*) FROM facts").fetchone()[0]

def add_fact(text, source="user", return_ids=False, max_len=0):
    text = (text or "").strip()
    pieces = chunk_text(text, max_len) or ([text] if text else [])
    if not pieces:
        msg = "Nothing to learn."
        return (msg, []) if return_ids else msg
    ids, new = _add_pieces(pieces, source)
    msg = f"🎓 Learned ({new} new) · {count_facts():,} memories"
    return (msg, ids) if return_ids else msg

def _text_bonus(fid, inter):
    if not inter:
        return 0.0
    keys = [f"{w}|{fid}" for w in inter]
    rows = DB.execute(f"SELECT v FROM q WHERE k IN ({','.join('?' * len(keys))})", keys).fetchall()
    return sum(r[0] for r in rows) / len(keys)

def rank(query, limit=20):
    qset = token_set(query)
    if not qset:
        return []
    with LOCK:
        if HAS_FTS:
            match = " OR ".join(f'"{t}"' for t in sorted(qset))
            try:
                rows = DB.execute(
                    "SELECT f.id, f.text, f.source, facts_fts.rank FROM facts_fts "
                    "JOIN facts f ON f.id = facts_fts.rowid WHERE facts_fts MATCH ? "
                    "ORDER BY facts_fts.rank LIMIT 60", (match,)).fetchall()
                rows = [(a, b, c, -d) for a, b, c, d in rows]
            except sqlite3.OperationalError:
                rows = []
        else:
            cond = " OR ".join("text LIKE ?" for _ in qset)
            rows = [(a, b, c, 1.0) for a, b, c in DB.execute(
                f"SELECT id,text,source FROM facts WHERE {cond} LIMIT 200",
                [f"%{w}%" for w in qset]).fetchall()]
        if not rows:
            return []
        maxb = max(r[3] for r in rows)
        out = []
        for fid, text, src, b in rows:
            inter = qset & token_set(text)
            cov = len(inter) / len(qset)
            if cov < MIN_COV:
                continue
            sim = 0.6 * cov + 0.4 * (b / maxb if maxb > 0 else 0.0)
            out.append((sim + RL_WEIGHT * _text_bonus(fid, inter), sim, {"id": fid, "text": text, "source": src}))
    out.sort(key=lambda x: x[0], reverse=True)
    return out[:limit]

def epsilon():
    total = feedback["good"] + feedback["bad"]
    return max(0.05, 0.30 * (0.95 ** total))

def rl_update(query, used, reward):
    with LOCK:
        for w in token_set(query):
            for fid, weight in used:
                k = f"{w}|{fid}"
                row = DB.execute("SELECT v FROM q WHERE k=?", (k,)).fetchone()
                old = row[0] if row else 0.0
                new = max(-1.0, min(1.0, old + ALPHA * (reward * weight - old)))
                DB.execute("INSERT OR REPLACE INTO q(k,v) VALUES(?,?)", (k, new))
        DB.commit()

def visual_line():
    with LOCK:
        nd = DB.execute("SELECT COUNT(*) FROM sketches").fetchone()[0]
        nm = DB.execute("SELECT COUNT(*) FROM moves3d").fetchone()[0]
        return f"Camera: {idx.n:,} views of {len(idx.by_label):,} things · {nd} drawings · {nm} moves"

def memory_list(search=""):
    with LOCK:
        total = count_facts()
        if (search or "").strip():
            rows = [(r[2]["id"], r[2]["source"], r[2]["text"]) for r in rank(search, 50)]
            head = f"Results ({len(rows)}) | {total:,} memories | {visual_line()}"
        else:
            rows = DB.execute("SELECT id, source, text FROM facts ORDER BY id DESC LIMIT 100").fetchall()
            head = f"Latest {len(rows)} of {total:,} memories | {visual_line()}"
        if not rows:
            return head + "\n\n(empty)"
        lines = [head, ""]
        for fid, src, txt in rows:
            avg = DB.execute("SELECT AVG(v) FROM q WHERE k LIKE ?", (f"%|{fid}",)).fetchone()[0]
            score = f" ({avg:+.2f})" if avg is not None else ""
            lines.append(f"#{fid} [{src}] {txt}{score}")
    return "\n".join(lines)

def forget_all():
    with LOCK:
        DB.execute("DELETE FROM facts"); DB.execute("DELETE FROM q")
        if HAS_FTS:
            DB.execute("DELETE FROM facts_fts")
        DB.commit()
    _kc.clear()
    return "All text memories cleared.", memory_list(), F("sad", "I forgot everything…")

# ===================== ROBOT CAPTION + COMMANDS TO THE 3D STAGE =====================
# The 3D robot lives in the browser (Three.js). Python only sends a small caption element that carries
# the mood and, when needed, a command (data-cmd) such as "switch world", "draw this" or "play this move".
ACC = {"idle": "#22d3ee", "thinking": "#a78bfa", "happy": "#34d399", "learning": "#fbbf24",
       "confused": "#fb923c", "surprised": "#f472b6", "sad": "#60a5fa", "drawing": "#f59e0b", "move": "#fbbf24"}
_cn = [0]
EMO = {"idle": "🙂", "thinking": "🤔", "happy": "😄", "learning": "🤩", "confused": "😕", "surprised": "😮",
       "sad": "😢", "drawing": "🎨", "move": "🥊"}

def face_html(mood, caption, cmd=None):
    col = ACC.get(mood, ACC["idle"])
    _cn[0] += 1
    c = ""
    if cmd is not None:
        c = f' data-cmd="{_html.escape(json.dumps(cmd, separators=(",", ":")), quote=True)}"'
    emo = EMO.get(mood, "🙂")
    return (f'<div class="facebox" data-mood="{mood}" data-n="{_cn[0]}"{c} style="--acc:{col}">'
            f'<div class="cap"><span class="emo">{emo}</span> {_html.escape(caption)}</div></div>')

def F(mood, caption="", cmd=None):
    live["last_key"] = (mood, caption)
    live["mood"] = mood
    live["mood_at"] = time.time()
    return face_html(mood, caption, cmd)

def face_update(mood, caption):
    return skip() if live["last_key"] == (mood, caption) else F(mood, caption)

def idle_tick():
    """Every second: expressions return to the normal face after a few seconds."""
    t = time.time()
    if (live["mood"] != "idle" and not live["busy"] and t > live["hold_until"]
            and t - live["mood_at"] > RESET_AFTER and t - live["last_live"] > 1.5):
        return F("idle", IDLE_CAP)
    return skip()

# ===================== CAMERA: FEATURES =====================
def to_arr(img):
    im = Image.fromarray(np.asarray(img).astype("uint8"))
    if im.mode != "RGB":
        im = im.convert("RGB")
    return np.asarray(im.resize((64, 64)), dtype=np.float32) / 255.0

def _unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 1e-8 else v

def _damp(v, eps):
    return v / np.sqrt(float((v * v).sum()) + eps)

def extract_features(a):
    q = np.minimum((a * 4).astype(int), 3)
    hist = np.sqrt(np.bincount((q[..., 0] * 16 + q[..., 1] * 4 + q[..., 2]).ravel(), minlength=64).astype(np.float32))
    lay = a.reshape(4, 16, 4, 16, 3).mean(axis=(1, 3)).reshape(-1, 3)
    lay = (lay - lay.mean(axis=0)).ravel()
    g = a.mean(axis=2).reshape(32, 2, 32, 2).mean(axis=(1, 3))
    gx = np.zeros_like(g); gy = np.zeros_like(g)
    gx[:, 1:-1] = g[:, 2:] - g[:, :-2]
    gy[1:-1, :] = g[2:, :] - g[:-2, :]
    mag = np.maximum(np.hypot(gx, gy) - 0.05, 0)
    b = np.minimum((np.arctan2(gy, gx) % np.pi / np.pi * 8).astype(int), 7)
    cells = [np.bincount(b.ravel(), weights=mag.ravel(), minlength=8)]
    for i in range(2):
        for j in range(2):
            sl = (slice(i * 16, (i + 1) * 16), slice(j * 16, (j + 1) * 16))
            cells.append(np.bincount(b[sl].ravel(), weights=mag[sl].ravel(), minlength=8))
    hog = np.concatenate(cells).astype(np.float32)
    f = np.concatenate([_unit(hist), _damp(lay, 0.01), 0.5 * _damp(hog, 0.05)])
    return _unit(f).astype(np.float32)

def color_name(rgb):
    h, s, v = colorsys.rgb_to_hsv(*rgb)
    h *= 360
    if v < 0.2: return "black"
    if s < 0.15: return "white" if v > 0.8 else "gray"
    if h < 15 or h >= 345: return "red"
    if h < 45: return "brown" if v < 0.55 else "orange"
    if h < 70: return "yellow"
    if h < 165: return "green"
    if h < 200: return "teal"
    if h < 260: return "blue"
    if h < 290: return "purple"
    return "pink"

def observe(a):
    q = np.minimum((a * 4).astype(int), 3)
    top = int(np.bincount((q[..., 0] * 16 + q[..., 1] * 4 + q[..., 2]).ravel(), minlength=64).argmax())
    rgb = (((top // 16) + 0.5) / 4, (((top // 4) % 4) + 0.5) / 4, ((top % 4) + 0.5) / 4)
    br = float(a.mean())
    light = "dark" if br < 0.25 else "dim" if br < 0.45 else "well-lit" if br < 0.7 else "bright"
    g = a.mean(axis=2)
    edge = float(np.abs(np.diff(g, axis=1)).mean() + np.abs(np.diff(g, axis=0)).mean())
    detail = "smooth" if edge < 0.03 else "textured" if edge < 0.08 else "detailed"
    return color_name(rgb), light, detail

# ===================== CAMERA: VISUAL INDEX =====================
class VisIndex:
    def __init__(self):
        self.reset()

    def reset(self):
        self.cap = 1024
        self.M = np.zeros((self.cap, DIM), np.float32)
        self.ids = np.zeros(self.cap, np.int64)
        self.q = np.zeros(self.cap, np.float32)
        self.hits = np.zeros(self.cap, np.int32)
        self.n = 0
        self.labels, self.by_label, self.row_of, self.dirty = [], {}, {}, set()

    def _grow(self):
        e = self.cap
        self.M = np.vstack([self.M, np.zeros((e, DIM), np.float32)])
        self.ids = np.concatenate([self.ids, np.zeros(e, np.int64)])
        self.q = np.concatenate([self.q, np.zeros(e, np.float32)])
        self.hits = np.concatenate([self.hits, np.zeros(e, np.int32)])
        self.cap += e

    def _append(self, sid, label, f, q=0.0, hits=0):
        if self.n >= self.cap:
            self._grow()
        r = self.n
        self.M[r] = f; self.ids[r] = sid; self.q[r] = q; self.hits[r] = hits
        self.labels.append(label); self.by_label.setdefault(label, []).append(r); self.row_of[sid] = r
        self.n += 1
        return r

    def load(self):
        with LOCK:
            self.reset()
            for sid, label, blob, q, hits in DB.execute("SELECT id,label,feat,q,hits FROM visual ORDER BY id"):
                f = np.frombuffer(blob, dtype=np.float16).astype(np.float32)
                if f.shape[0] == DIM:
                    self._append(sid, label, f, q or 0.0, hits or 0)

    def count(self, label):
        return len(self.by_label.get(label, []))

    def add(self, feat, label):
        now = datetime.now().isoformat(timespec="seconds")
        with LOCK:
            rows = self.by_label.get(label, [])
            if rows:
                sims = self.M[rows] @ feat
                j = int(np.argmax(sims))
                if sims[j] >= DEDUP_SIM:
                    r = rows[j]; self.hits[r] += 1; self.dirty.add(int(self.ids[r]))
                    return int(self.ids[r]), "duplicate"
            blob = feat.astype(np.float16).tobytes()
            if len(rows) >= MAX_PER_LABEL:
                vals = self.hits[rows] + 3 * self.q[rows]
                r = rows[int(np.argmin(vals))]; sid = int(self.ids[r])
                self.M[r] = feat; self.q[r] = 0; self.hits[r] = 0
                DB.execute("UPDATE visual SET feat=?, q=0, hits=0, updated=? WHERE id=?", (blob, now, sid))
                DB.commit()
                return sid, "replaced"
            sid = DB.execute("INSERT INTO visual(label,feat,q,hits,updated) VALUES(?,?,?,?,?)",
                             (label, blob, 0.0, 0, now)).lastrowid
            DB.commit()
            self._append(sid, label, feat)
            return sid, "new"

    def search(self, feat):
        with LOCK:
            if self.n == 0:
                return []
            sims = self.M[:self.n] @ feat
            cand = np.nonzero(sims >= VIS_MIN_SIM)[0]
            if len(cand) > 500:
                cand = cand[np.argsort(-sims[cand])[:500]]
            best = {}
            for r in cand:
                r = int(r); lab = self.labels[r]
                score = float(sims[r]) + RL_WEIGHT * float(self.q[r])
                if lab not in best or score > best[lab][0]:
                    best[lab] = (score, float(sims[r]), r, lab)
            return sorted(best.values(), key=lambda x: x[0], reverse=True)

    def reward(self, sid, reward, weight=1.0):
        with LOCK:
            r = self.row_of.get(int(sid))
            if r is None:
                return
            self.q[r] = max(-1.0, min(1.0, self.q[r] + ALPHA * (reward * weight - self.q[r])))
            self.dirty.add(int(sid))

    def touch(self, row):
        with LOCK:
            self.hits[row] += 1; self.dirty.add(int(self.ids[row]))
            if len(self.dirty) > 200:
                self.flush()

    def flush(self):
        with LOCK:
            rows = [(float(self.q[self.row_of[s]]), int(self.hits[self.row_of[s]]), s)
                    for s in list(self.dirty) if s in self.row_of]
            if rows:
                DB.executemany("UPDATE visual SET q=?, hits=? WHERE id=?", rows)
                DB.commit()
            self.dirty.clear()

    def clear(self):
        with LOCK:
            DB.execute("DELETE FROM visual"); DB.commit()
            self.reset()

idx = VisIndex()
idx.load()

def migrate_legacy():
    old = load_json(LEGACY_PATH, None)
    if not old or count_facts() or idx.n:
        return
    for f in old.get("facts", []):
        add_fact(f.get("text", ""), f.get("source", "user"))
    for s in old.get("visual", []):
        feat = np.array(s.get("feat", []), np.float32)
        if feat.shape[0] == DIM:
            idx.add(feat, s.get("label", "?"))
    try:
        LEGACY_PATH.rename(LEGACY_PATH.with_suffix(".json.migrated"))
    except Exception:
        pass

migrate_legacy()

def teach_feat(feat, label, obs=None, reward=True):
    label = (label or "").strip().lower()
    sid, st = idx.add(feat, label)
    if reward and st != "duplicate":
        idx.reward(sid, +1.0)
    if obs and label not in live["noted"]:
        live["noted"].add(label)
        add_fact(f"{label} usually looks mostly {obs[0]}, {obs[1]} and {obs[2]} on camera.", "camera-observed")
    return sid, st

def know_text(label):
    t = time.time(); c = _kc.get(label)
    if c and t - c[0] < 15:
        return c[1]
    facts = [f for f in rank(label, 30) if label.lower() in f[2]["text"].lower()][:1]
    out = (" — " + facts[0][2]["text"][:140]) if facts else ""
    _kc[label] = (t, out)
    return out

def recognize_feat(feat, explore=True):
    last_vis["feat"] = feat
    m = idx.search(feat)
    if not m:
        last_vis.update(label=None, used=[])
        return None
    cands = m[:TOP_K]
    explored = explore and len(cands) > 1 and random.random() < epsilon()
    chosen = random.choice(cands[1:]) if explored else cands[0]
    last_vis.update(label=chosen[3], used=[(int(idx.ids[chosen[2]]), 1.0)])
    idx.touch(chosen[2])
    return dict(label=chosen[3], sim=chosen[1], explored=explored)

# ===================== CANVAS DRAWING LEARNING (drawn by the robot in the 3D studio) =====================
def sketch_norm(strokes):
    pts = [p for s in strokes for p in s]
    if len(pts) < 4:
        return None
    a = np.array(pts, float); lo, hi = a.min(0), a.max(0)
    size = max(float((hi - lo).max()), 1.0); c = (lo + hi) / 2
    out = []
    for s in strokes:
        s = np.array(s, float)
        if len(s) < 2:
            continue
        ids = list(range(0, len(s), 2))
        if ids[-1] != len(s) - 1:
            ids.append(len(s) - 1)
        out.append((((s[ids] - c) / size) * 2).round(3).tolist())   # -1..1 (y down)
    return out or None

def get_sketch(label):
    with LOCK:
        row = DB.execute("SELECT data FROM sketches WHERE name=?", (label,)).fetchone()
    return json.loads(row[0]) if row else None

def save_sketch(label, st):
    with LOCK:
        row = DB.execute("SELECT n FROM sketches WHERE name=?", (label,)).fetchone()
        n = (row[0] if row else 0) + 1
        DB.execute("INSERT OR REPLACE INTO sketches(name,data,n) VALUES(?,?,?)", (label, json.dumps(st), n))
        DB.commit()
    return n

def get_skill(name):
    with LOCK:
        row = DB.execute("SELECT v,eps,w FROM skills WHERE name=?", (name,)).fetchone()
    return row if row else (0.0, 0, None)

def set_skill(name, v, eps, w):
    with LOCK:
        DB.execute("INSERT OR REPLACE INTO skills(name,v,eps,w) VALUES(?,?,?,?)",
                   (name, float(v), int(eps), json.dumps(w) if w is not None else None))
        DB.commit()

def draw_cmd(label, st=None, world=True):
    st = st or get_sketch(label) or []
    c = {"draw": {"strokes": st, "skill": round(float(get_skill("draw")[0]), 3), "dur": 7}}
    if world:
        c["world"] = "draw"
    return c

def board_html(label=None, skill=0.0):
    if not label:
        return '<div class="board"><div class="cap">Teach me on the canvas, then ask me to draw it in the 3D studio.</div></div>'
    return (f'<div class="board"><div class="cap">Drawing: {_html.escape(label)} · skill {skill:.0%}</div>'
            f'<div class="bar"><i style="width:{int(skill*100)}%"></i></div></div>')

def paint_save(name, strokes_json):
    n = (name or "").strip().lower()
    if not n:
        return "Type a name for this drawing first.", F("confused", "What is it called?"), V("oops", "What is it called?")
    try:
        strokes = json.loads(strokes_json or "[]")
    except Exception:
        strokes = []
    st = sketch_norm(strokes)
    if st is None:
        return "Draw something on the canvas first.", F("confused", "The canvas is empty"), V("oops", "Draw something first")
    c = save_sketch(n, st)
    live["hold_until"] = time.time() + 9
    return (f"✏️ Learned to draw **{n}** (lesson {c}) — watch me draw it in the 3D studio.",
            F("learning", f"I can draw {n} now!", draw_cmd(n, st)), V("learn", f"Now I can draw {n}"))

def draw_ui(name):
    n = (name or "").strip().lower()
    if not n:
        return skip(), "Type what I should draw.", F("confused", "What should I draw?"), V("oops", "What should I draw?")
    if get_sketch(n) is None:
        return (board_html(), f"I can't draw **{n}** yet. Draw it on the canvas, type a name and press 💾 Teach.",
                F("confused", f"Teach me to draw {n}!"), V("say", f"Teach me to draw {n}"))
    skill = get_skill("draw")[0]
    live["kind"] = "draw"; live["draw_label"] = n; live["hold_until"] = time.time() + 9.5
    return (board_html(n, skill), f"✏️ Drawing **{n}** · skill {skill:.0%} — rate me with 👍 / 👎",
            F("drawing", f"Drawing {n}…", draw_cmd(n)), V("say", f"I will draw {n}"))

def draw_rate(kind):
    reward = 1.0 if kind == "good" else -1.0
    feedback[kind] = int(feedback.get(kind, 0)) + 1
    v, eps, w = get_skill("draw")
    v = max(0.0, min(1.0, v + 0.08 * reward))
    set_skill("draw", v, eps, json.loads(w) if w else None)
    save_json(FEEDBACK_PATH, feedback)
    msg = f"⭐ drawing skill {v:.0%} · 👍 {feedback['good']}  👎 {feedback['bad']}"
    if reward > 0:
        return msg, F("happy", "Yay! I'm getting better!"), V("ok", "Yay! Thank you")
    return msg, F("sad", "I'll practice more…"), V("oops", "Oops, I will practice")

# ===================== SIMULATION WORLD: drawing practice (vectorised) =====================
L1, L2 = 0.55, 0.45          # upper-arm / forearm length of the practice arm

def _feats(p):
    x, y = p[..., 0], p[..., 1]
    return np.stack([x, y, x * x, y * y, x * y, np.ones_like(x)], -1)

def sim_targets(label):
    st = get_sketch(label) if label else None
    if st:
        a = np.array([p for s in st for p in s], float)
        d = np.r_[0, np.cumsum(np.hypot(*np.diff(a, axis=0).T))]
        if d[-1] > 1e-6:
            t = np.linspace(0, d[-1], K)
            pts = np.stack([np.interp(t, d, a[:, 0]), -np.interp(t, d, a[:, 1])], -1) * 0.3
            return pts + np.array([0.0, 0.65])
    ang = -np.pi + (np.arange(K) + 0.5) * 2 * np.pi / K
    return np.stack([np.cos(ang), -np.sin(ang)], -1) * 0.3 + np.array([0.0, 0.65])

def _hand(Wp, T):
    """Wp: (P,12) policy weights -> hand paths (P,K,2) for target points T (K,2). One batched call for the whole population."""
    th = np.einsum("kf,pfj->pkj", _feats(T), Wp.reshape(-1, 6, 2))
    a = th[..., 0]; b = a + th[..., 1]
    return np.stack([L1 * np.cos(a) + L2 * np.cos(b), L1 * np.sin(a) + L2 * np.sin(b)], -1)

def _to_uv(pts):
    return np.stack([pts[..., 0] / 0.3, (pts[..., 1] - 0.65) / 0.3], -1).clip(-1.2, 1.2).round(3).tolist()

def sim_draw_train(label, gens, progress=gr.Progress()):
    label = (label or "").strip().lower() or None
    gens = int(gens)
    T = sim_targets(label)
    v0, eps0, wj = get_skill("draw")
    mu = np.array(json.loads(wj), np.float64) if wj else np.random.randn(12) * 0.2
    sd = np.full(12, 0.5); hist = []
    live["busy"] = True
    try:
        e0 = float(np.linalg.norm(_hand(mu[None], T)[0] - T, axis=1).mean())
        tu = _to_uv(T)
        yield (f"🧪 Practising **{label or 'a circle'}** in the 3D studio · {gens} generations…",
               sim_card([], [f"start error {e0:.3f} · skill {v0:.0%}"]),
               F("learning", "Practicing in simulation…", {"world": "draw", "target": tu, "trace": {"h": _to_uv(_hand(mu[None], T)[0]), "dur": 2.4}}),
               V("learn", "Let me practice"))
        step = max(1, gens // 10)
        for g in range(gens):
            pop = mu + sd * np.random.randn(64, 12)
            errs = np.linalg.norm(_hand(pop, T) - T[None], axis=2).mean(1)      # all 64 candidates at once
            el = np.argsort(errs)[:10]
            mu = pop[el].mean(0); sd = pop[el].std(0) + 0.01
            e = float(np.linalg.norm(_hand(mu[None], T)[0] - T, axis=1).mean()); hist.append(e)
            sk = max(0.0, 1.0 - e / 0.2)
            progress((g + 1) / gens, desc=f"Generation {g + 1}/{gens}")
            if g % step == 0 or g == gens - 1:
                yield (f"🧪 generation {g + 1}/{gens} · error {e:.3f} · skill {sk:.0%}",
                       sim_card(hist, [f"error {e:.3f} · skill {sk:.0%}"]),
                       F("learning", "Practicing in simulation…", {"trace": {"h": _to_uv(_hand(mu[None], T)[0]), "dur": 2.0}}), skip())
                time.sleep(0.6)
        sk = max(v0, max(0.0, 1.0 - hist[-1] / 0.2))
        set_skill("draw", sk, eps0 + gens, mu.tolist())
        st = get_sketch(label) if label else None
        cmd = {"world": "draw", "draw": {"strokes": st, "skill": round(sk, 3), "dur": 7}} if st else {"world": "draw"}
        yield (f"✅ Trained {gens} generations · drawing skill {sk:.0%}", sim_card(hist, [f"final error {hist[-1]:.3f} · skill {sk:.0%}"]),
               F("happy", "I got better at drawing!", cmd), V("done", "I practiced and got better"))
    except Exception as e:
        yield f"Simulation error: {e}", skip(), F("sad", "Something went wrong."), V("oops")
    finally:
        live["busy"] = False
        live["mood_at"] = time.time()

def sim_train(kind, lesson, draw_name, gens, progress=gr.Progress()):
    if kind and kind.startswith("🖌"):
        yield from sim_draw_train(draw_name, gens, progress)
    else:
        yield from sim_body_train(lesson, gens, progress)

# ---------- feedback ----------
def camera_rate(kind):
    reward = 1.0 if kind == "good" else -1.0
    feedback[kind] = int(feedback.get(kind, 0)) + 1
    for sid, w in last_vis["used"]:
        idx.reward(sid, reward, w)
    idx.flush(); save_json(FEEDBACK_PATH, feedback)
    msg = f"👍 {feedback['good']}  👎 {feedback['bad']}"
    return (msg, F("happy", "Yay!"), V("ok", "Yay!")) if reward > 0 else (msg, F("sad", "Oops!"), V("oops", "Oops, sorry"))

def camera_correct(right_label):
    right = (right_label or "").strip().lower()
    if last_vis["feat"] is None or not right:
        return "Show me something first.", F("confused", "What is it?"), V("oops")
    for sid, w in last_vis["used"]:
        idx.reward(sid, -1.0, w)
    teach_feat(last_vis["feat"], right)
    idx.flush()
    return f"✏️ Fixed: **{right}**", F("learning", f"Now I know it's {right}!"), V("learn", f"Now I know it's {right}")

def forget_visual():
    idx.clear(); last_vis.update(label=None, used=[], feat=None)
    with LOCK:
        DB.execute("DELETE FROM sketches"); DB.commit()
    return "Camera memories and drawings cleared.", F("sad", "I forgot what I saw…"), memory_list()

# ---------- REAL-TIME LIVE CAMERA (seeing objects) ----------
def learn_caption(n):
    return f"Learning {n}…"

def live_step(frame, teach_name, learning, mode):
    if frame is None:
        return skip(), skip(), skip()
    try:
        t = time.time()
        live["last_live"] = t
        a = to_arr(frame)
        live["arr"] = a
        if mode != MODE_SEE:
            return skip(), skip(), skip()
        feat = extract_features(a)
        live["feat"] = feat; last_vis["feat"] = feat
        col, light, detail = observe(a)
        g = a.mean(axis=2).reshape(16, 4, 16, 4).mean(axis=(1, 3))
        motion = 0.0 if live["prev_g"] is None else float(np.abs(g - live["prev_g"]).mean())
        live["prev_g"] = g
        mv = "still" if motion < 0.02 else "slight movement" if motion < 0.08 else "moving"
        seen = f"{col}, {light}, {mv}"
        if t - live["last_flush"] > 20:
            idx.flush(); live["last_flush"] = t
        name = (teach_name or "").strip().lower()

        # --- real-time object learning ---
        if learning and name:
            voice = skip()
            if t - live["last_save"] >= AUTO_EVERY:
                _, st = teach_feat(feat, name, obs=(col, light, detail))
                live["last_save"] = t
                if live["learn_name"] != name:
                    live["learn_name"] = name; live["learn_n"] = 0
                    voice = V("learn", f"Learning {name}")
                if st != "duplicate":
                    live["learn_n"] += 1
                    if live["learn_n"] % 12 == 0:
                        voice = V("tick", f"Learning {name}")
                    elif live["learn_n"] % 4 == 0:
                        voice = V("tick")
            n = idx.count(name)
            return (f"🎓 **{name}** · {n} views · {seen}",
                    face_update("learning", learn_caption(name)), voice)

        # --- recognition ---
        matches = idx.search(feat)
        live["hist"].append(matches[0][3] if matches else None)
        top, cnt = Counter(live["hist"]).most_common(1)[0]
        if len(live["hist"]) < 3 or cnt < 3:
            return f"🤔 looking… · {seen}", face_update("thinking", "Hmm…"), skip()
        if top is None:
            last_vis.update(label=None, used=[])
            live["prev_label"] = None
            voice = skip()
            if live["announced"] != "?" and t - live["last_say"] > 2.5:
                live["announced"] = "?"; live["last_say"] = t; live["kind"] = "vis"
                voice = V("say", "What is this? Teach me!")
            return (f"❓ unknown · {seen}", face_update("confused", "What is this? Teach me!"), voice)
        best = next((m for m in matches if m[3] == top), None)
        if best is None:
            return f"🤔 looking… · {seen}", face_update("thinking", "Hmm…"), skip()
        idx.touch(best[2])
        last_vis.update(label=top, used=[(int(idx.ids[best[2]]), 1.0)])
        if live["prev_label"] not in (None, top):
            live["surprise_until"] = t + 1.5
        live["prev_label"] = top
        voice = skip()
        if live["announced"] != top and t - live["last_say"] > 2.5:
            live["announced"] = top; live["last_say"] = t; live["kind"] = "vis"
            voice = V("say", f"I see {top}")
        saved = ""
        if best[1] < SELF_SIM and t - live["last_auto"] > 2.0:
            _, st = teach_feat(feat, top, reward=False)
            live["last_auto"] = t
            if st != "duplicate":
                saved = " 💾"
        text = f"👁️ **{top}** · {best[1]:.2f}{know_text(top)} · {seen}{saved}"
        if t < live["surprise_until"]:
            return text, face_update("surprised", f"Oh! {top}!"), voice
        return text, face_update("happy", f"I see {top}!"), voice
    except Exception as e:
        return f"⚠️ {e}", skip(), skip()

def start_learning(name, mode):
    n = (name or "").strip().lower()
    if mode != MODE_SEE:
        return False, "Choose 👁 Seeing in the Learning list first.", F("confused", "Pick what to learn!"), V("oops")
    if not n:
        return False, "Type a name first.", F("confused", "What is it called?"), V("oops", "What is it called?")
    live["learn_name"] = None; live["learn_n"] = 0
    if live["feat"] is not None:
        teach_feat(live["feat"], n)
    return (True, f"🎓 Learning **{n}** — move it around.", F("learning", learn_caption(n)),
            V("learn", f"Okay! Show me the {n}"))

def stop_learning(name, mode):
    n = (name or "").strip().lower()
    idx.flush()
    live.update(prev_g=None, feat=None, arr=None, announced=None, prev_label=None, learn_name=None, last_live=0.0)
    live["hist"].clear()
    cnt = idx.count(n) if n else 0
    if cnt:
        return False, None, f"✅ Stopped · **{n}**: {cnt} views", F("happy", f"I learned {n}!"), V("done", f"I learned {n}!")
    return False, None, "✅ Camera stopped.", F("idle", IDLE_CAP), V("tick")

def cam_upload(img):
    if img is None:
        return skip(), skip(), skip()
    a = to_arr(img); feat = extract_features(a)
    live["feat"] = feat; live["kind"] = "vis"
    r = recognize_feat(feat, explore=True)
    if r is None:
        return ("❓ unknown — type a name and press ▶ Learn", F("confused", "What is this? Teach me!"),
                V("say", "What is this? Teach me!"))
    return (f"👁️ **{r['label']}** · {r['sim']:.2f}{know_text(r['label'])}",
            F("happy", f"I see {r['label']}!"), V("say", f"I see {r['label']}"))

def photo_file(file):
    if file is None:
        return skip(), skip(), skip()
    try:
        img = Image.open(getattr(file, "name", file)).convert("RGB")
        return cam_upload(np.asarray(img))
    except Exception as e:
        return f"⚠️ {e}", skip(), skip()

# ===================== 3D BODY MOVES: capture -> store -> perform, re-teach, lessons =====================
# A move = frames x 49 unit direction vectors (body bones + both hands with 5 fingers each), see js_core.
NB = 49
LEN = dict(hw=.095, torso=.40, sw=.175, drop=.05, neck=.075, head=.10, ua=.24, fa=.23, th=.30, sh=.30, ft=.20, sole=.065)
PART_BONES = {"Torso + head": [0, 1, 2, 3, 4], "Arms": [5, 6, 7, 8], "Hands + fingers": list(range(15, 49)),
              "Legs": [9, 10, 11, 12, 13, 14]}
PART_NAMES = list(PART_BONES)
TEACH_REFINE = "🔁 Refine (blend with what I know)"
TEACH_REPLACE = "♻️ Replace (forget the old version)"

def _u(v):
    v = np.asarray(v, float)
    return v / np.maximum(np.linalg.norm(v, axis=-1, keepdims=True), 1e-9)

# ---- hand presets (same numbers as the browser) ----
_SPL = [0, .12, .03, -.06, -.15]
_FING = {"open": ([(1, 0)] * 3, [(.6, .7, 0), (.7, .7, 0), (.8, .6, 0)]),
         "relaxed": ([(.95, .3), (.75, .65), (.55, .8)], [(.6, .5, .2), (.8, .3, .3), (.8, .2, .4)]),
         "fist": ([(.2, 1), (-.6, .8), (-.9, -.4)], [(.4, .5, .4), (.2, .1, .9), (-.2, -.3, .9)]),
         "grip": ([(.5, .85), (.2, .95), (-.2, .9)], [(.5, .5, .5), (.4, .2, .8), (.3, 0, .9)])}

def hand_set(P, base, fwd, side, left, preset):
    fwd = _u(fwd); side = np.asarray(side, float)
    side = side - fwd * np.dot(side, fwd)
    if np.linalg.norm(side) < 0.5:
        side = np.array([0., 0., 1.]) - fwd * fwd[2]
    side = _u(side)
    n = np.cross(side, fwd); n = -n if left else n
    P[base] = fwd; P[base + 1] = side
    fc, tc = _FING[preset]
    for s in range(3):
        cf, cn = fc[s]
        for f in range(1, 5):
            P[base + 2 + f * 3 + s] = _u(cf * fwd + cn * n + _SPL[f] * side)
        a, b, c = tc[s]
        P[base + 2 + s] = _u(a * fwd + b * side + c * n)

def rest_pose():
    P = np.zeros((NB, 3))
    for i, v in {0: [0, 1, 0], 1: [1, 0, 0], 2: [1, 0, 0], 3: [0, 1, 0], 4: [0, 0, 1], 5: [.10, -1, 0], 6: [.06, -1, .08],
                 7: [-.10, -1, 0], 8: [-.06, -1, .08], 9: [0, -1, 0], 10: [0, -1, 0], 11: [0, 0, 1],
                 12: [0, -1, 0], 13: [0, -1, 0], 14: [0, 0, 1]}.items():
        P[i] = _u(v)
    hand_set(P, 15, P[6], [0, 0, 1], True, "relaxed")
    hand_set(P, 32, P[8], [0, 0, 1], False, "relaxed")
    return P
REST = rest_pose()

# ---- forward kinematics + balance (numpy, same dimensions as the browser robot) ----
def fk_np(P):
    L = LEN
    up, hips, shd = P[:, 0], P[:, 1], P[:, 2]
    J = {}
    J["pel"] = np.zeros_like(up)
    J["hipL"] = hips * L["hw"]; J["hipR"] = -hips * L["hw"]
    J["neckB"] = up * L["torso"]
    sb = J["neckB"] - up * L["drop"]
    J["shL"] = sb + shd * L["sw"]; J["shR"] = sb - shd * L["sw"]
    J["elL"] = J["shL"] + P[:, 5] * L["ua"]; J["wrL"] = J["elL"] + P[:, 6] * L["fa"]
    J["elR"] = J["shR"] + P[:, 7] * L["ua"]; J["wrR"] = J["elR"] + P[:, 8] * L["fa"]
    J["head"] = J["neckB"] + P[:, 3] * (L["neck"] + L["head"])
    J["knL"] = J["hipL"] + P[:, 9] * L["th"]; J["anL"] = J["knL"] + P[:, 10] * L["sh"]; J["toL"] = J["anL"] + P[:, 11] * L["ft"]
    J["knR"] = J["hipR"] + P[:, 12] * L["th"]; J["anR"] = J["knR"] + P[:, 13] * L["sh"]; J["toR"] = J["anR"] + P[:, 14] * L["ft"]
    g = np.minimum.reduce([J["anL"][:, 1], J["anR"][:, 1], J["toL"][:, 1], J["toR"][:, 1]]) - L["sole"]
    for k in J:
        J[k] = J[k] - np.stack([np.zeros_like(g), g, np.zeros_like(g)], -1)
    return J

def com_support(P):
    J = fk_np(P); m = lambda a, b: (J[a] + J[b]) / 2
    com = (0.40 * m("pel", "neckB") + 0.08 * J["head"] + 0.03 * (m("shL", "elL") + m("shR", "elR"))
           + 0.03 * (m("elL", "wrL") + m("elR", "wrR")) + 0.10 * (m("hipL", "knL") + m("hipR", "knR"))
           + 0.045 * (m("knL", "anL") + m("knR", "anR")) + 0.02 * (m("anL", "toL") + m("anR", "toR"))) / 0.74
    pts = np.stack([J["anL"], J["toL"], J["anR"], J["toR"]], 1)          # (T,4,3)
    ymin = pts[:, :, 1].min(1, keepdims=True)
    gr = (pts[:, :, 1] - ymin) < 0.05
    big = 9.0
    xmin = np.where(gr, pts[:, :, 0], big).min(1) - 0.045; xmax = np.where(gr, pts[:, :, 0], -big).max(1) + 0.045
    zmin = np.where(gr, pts[:, :, 2], big).min(1) - 0.04;  zmax = np.where(gr, pts[:, :, 2], -big).max(1) + 0.04
    seated = (P[:, 9, 1] > -0.5) & (P[:, 12, 1] > -0.5)                    # thighs near horizontal: the chair supports
    xmin = np.where(seated, np.minimum(xmin, -0.14), xmin); xmax = np.where(seated, np.maximum(xmax, 0.14), xmax)
    zmin = np.where(seated, np.minimum(zmin, -0.14), zmin); zmax = np.where(seated, np.maximum(zmax, 0.14), zmax)
    return com, (xmin, xmax, zmin, zmax)

def balance_viol(P):
    com, (x0, x1, z0, z1) = com_support(P)
    dx = np.maximum(0, np.maximum(x0 - com[:, 0], com[:, 0] - x1)); dz = np.maximum(0, np.maximum(z0 - com[:, 2], com[:, 2] - z1))
    return float(((dx + dz) / 0.1).mean())

def _ang(a, b):
    return np.arccos(np.clip((a * b).sum(-1), -1, 1))

def limit_viol(P):
    kn = np.maximum(_ang(P[:, 9], P[:, 10]) - 2.36, 0) + np.maximum(_ang(P[:, 12], P[:, 13]) - 2.36, 0)
    el = np.maximum(_ang(P[:, 5], P[:, 6]) - 2.62, 0) + np.maximum(_ang(P[:, 7], P[:, 8]) - 2.62, 0)
    dn = np.array([0., -1., 0.])
    hp = np.maximum(_ang(P[:, 9], dn) - 2.1, 0) + np.maximum(_ang(P[:, 12], dn) - 2.1, 0)
    return float((kn + el + hp).mean())

def _rotx(v, a):
    c, s = math.cos(a), math.sin(a)
    out = v.copy(); out[..., 1] = v[..., 1] * c - v[..., 2] * s; out[..., 2] = v[..., 1] * s + v[..., 2] * c
    return out

def _nl(rest, v, k):
    return _u(rest + k * (v - rest))

def transform(Pref, th):
    """The 4 knobs the simulator trains: leg amplitude, arm amplitude, forward lean, crouch."""
    leg, arm, lean, crouch = th
    P = Pref.copy()
    for b in (5, 6, 7, 8):
        P[:, b] = _nl(REST[b], P[:, b], arm)
    for b in (9, 10, 12, 13):
        P[:, b] = _nl(REST[b], P[:, b], leg)
    for b, sg in ((9, -1), (12, -1), (10, 1), (13, 1)):
        P[:, b] = _u(_rotx(P[:, b], sg * crouch))
    for b in (0, 3):
        P[:, b] = _u(_rotx(P[:, b], lean))
    return P

def body_cost(P, Pref):
    bal = balance_viol(P); lim = limit_viol(P)
    trk = float(_ang(P[:, :15], Pref[:, :15]).mean())
    jerk = float(np.linalg.norm(np.diff(P[:, :15], 2, axis=0), axis=-1).mean()) if len(P) > 3 else 0.0
    return 4 * bal + 3 * lim + 1.5 * trk + 0.5 * jerk, bal, lim, math.degrees(trk)

# ---- built-in lessons (the robot's "novice" attempt that the simulation world then improves) ----
def _blank(T):
    return np.repeat(REST[None], T, 0)

def _sag(a):                  # sagittal-plane direction for a leg/arm angle (forward positive)
    return [0, -math.cos(a), math.sin(a)]

def lesson_walk(T=40):
    P = _blank(T)
    for k in range(T):
        ph = 2 * math.pi * 2 * k / T
        aL = 0.55 * math.sin(ph); aR = -aL
        kbL = 0.7 * max(0, math.cos(ph)); kbR = 0.7 * max(0, -math.cos(ph))
        P[k, 9] = _u(_sag(aL)); P[k, 10] = _u(_sag(aL - kbL)); P[k, 12] = _u(_sag(aR)); P[k, 13] = _u(_sag(aR - kbR))
        bL = -0.65 * math.sin(ph); bR = -bL
        P[k, 5] = _u([.10, -math.cos(bL), math.sin(bL)]); P[k, 6] = _u([.06, -math.cos(bL + .45), math.sin(bL + .45)])
        P[k, 7] = _u([-.10, -math.cos(bR), math.sin(bR)]); P[k, 8] = _u([-.06, -math.cos(bR + .45), math.sin(bR + .45)])
        P[k, 0] = _u([0, math.cos(0.28), math.sin(0.28)]); P[k, 3] = _u([0, math.cos(0.1), math.sin(0.1)])
        hand_set(P[k], 15, P[k, 6], [0, 0, 1], True, "relaxed"); hand_set(P[k], 32, P[k, 8], [0, 0, 1], False, "relaxed")
    return P, 3.2

def lesson_sit(T=40):
    P = _blank(T)
    for k in range(T):
        s = min(1.0, k / (T * 0.65)); s = s * s * (3 - 2 * s)
        a = 1.45 * s
        for b, ang in ((9, a), (12, a), (10, 0.0), (13, 0.0)):
            P[k, b] = _u(_sag(ang)) if b in (9, 12) else _u([0, -1, 0])
        P[k, 0] = _u([0, 1, 0.22 * math.sin(math.pi * s)])
        for sd, (ub, fb, ux) in ((1, (5, 6, .05)), (-1, (7, 8, -.05))):
            P[k, ub] = _nl(REST[ub], _u([ux, -.9, .3]), s); P[k, fb] = _nl(REST[fb], _u([0, -.25, .97]), s)
        hand_set(P[k], 15, P[k, 6], [0, 0, 1], True, "relaxed"); hand_set(P[k], 32, P[k, 8], [0, 0, 1], False, "relaxed")
    return P, 3.0

def lesson_talk(T=48):
    P = _blank(T)
    for k in range(T):
        ph = 2 * math.pi * 2 * k / T
        P[k, 7] = _u([-.25, -.55, .55]); P[k, 8] = _u([.15 + .2 * math.sin(ph * 1.5), .2 + .45 * math.sin(ph + 1), .85])
        P[k, 5] = _u([.2, -.8, .35]); P[k, 6] = _u([0, -.2 + .12 * math.sin(ph), .98])
        P[k, 3] = _u([0, 1, .05 * math.sin(2 * ph)]); P[k, 4] = _u([.06 * math.sin(ph), .08 * math.sin(2 * ph), 1])
        P[k, 0] = _u([.03 * math.sin(ph / 2), 1, .01])
        hand_set(P[k], 15, P[k, 6], [0, 1, 0], True, "open"); hand_set(P[k], 32, P[k, 8], [0, 1, 0], False, "open")
    return P, 4.0

def _horse(P, s):
    """Wing Chun horse stance (yi ji kim yeung ma): knees bent and out, toes slightly in."""
    for sg, (th, sh, ft) in ((1, (9, 10, 11)), (-1, (12, 13, 14))):
        P[th] = _nl(REST[th], _u([sg * .5, -.8, .33]), s); P[sh] = _nl(REST[sh], _u([-sg * .03, -1, -.02]), s)
        P[ft] = _nl(REST[ft], _u([-sg * .15, 0, 1]), s)

def lesson_horse(T=36):
    P = _blank(T)
    for k in range(T):
        s = min(1.0, k / (T * 0.5)); s = s * s * (3 - 2 * s)
        _horse(P[k], s)
        P[k, 5] = _nl(REST[5], _u([0, -.8, .5]), s); P[k, 6] = _nl(REST[6], _u([-.35, .35, .87]), s)
        P[k, 7] = _nl(REST[7], _u([0, -.8, .5]), s); P[k, 8] = _nl(REST[8], _u([.35, .35, .87]), s)
        hand_set(P[k], 15, P[k, 6], [0, 1, 0], True, "open"); hand_set(P[k], 32, P[k, 8], [0, 1, 0], False, "open")
        P[k, 0] = _u([0, 1, -0.12 * s])
    return P, 2.5

def lesson_chain(T=48):
    P = _blank(T)
    gL = (_u([0, -.8, .5]), _u([-.35, .35, .87])); gR = (_u([0, -.8, .5]), _u([.35, .35, .87]))
    eL = (_u([-.12, -.04, .99]), _u([-.25, 0, .97])); eR = (_u([.12, -.04, .99]), _u([.25, 0, .97]))
    for k in range(T):
        _horse(P[k], 1.0)
        ph = 2 * math.pi * 4 * k / T
        pR = max(0, math.sin(ph)) ** 2; pL = max(0, -math.sin(ph)) ** 2
        P[k, 7] = _nl(gR[0], eR[0], pR); P[k, 8] = _nl(gR[1], eR[1], pR)
        P[k, 5] = _nl(gL[0], eL[0], pL); P[k, 6] = _nl(gL[1], eL[1], pL)
        hand_set(P[k], 15, P[k, 6], [0, 1, 0], True, "fist"); hand_set(P[k], 32, P[k, 8], [0, 1, 0], False, "fist")
        P[k, 0] = _u([0, 1, -0.18])
    return P, 3.0

LESSON_GEN = {"walk": lesson_walk, "sit down": lesson_sit, "talk": lesson_talk,
              "horse stance": lesson_horse, "chain punch": lesson_chain}
LESSONS = list(LESSON_GEN)

# ---- database ----
def get_move(name):
    with LOCK:
        row = DB.execute("SELECT data,dur,miss,n,prev,score,src FROM moves3d WHERE name=?", (name,)).fetchone()
    if not row:
        return None
    f = np.array(json.loads(row[0]), float).reshape(-1, NB, 3)
    return dict(f=f, dur=float(row[1]), miss=np.array(json.loads(row[2]), bool), n=row[3], has_prev=bool(row[4]),
                score=row[5] or 0.0, src=row[6] or "teach")

def save_move3(name, mv, prev=None, src="teach", score=0.0):
    data = json.dumps(np.round(mv["f"].reshape(len(mv["f"]), -1), 2).tolist())
    pj = None
    if prev is not None:
        pj = json.dumps({"f": np.round(prev["f"].reshape(len(prev["f"]), -1), 2).tolist(), "dur": prev["dur"],
                         "miss": prev["miss"].tolist()})
    with LOCK:
        old = DB.execute("SELECT n FROM moves3d WHERE name=?", (name,)).fetchone()
        DB.execute("INSERT OR REPLACE INTO moves3d(name,data,n,dur,prev,miss,score,src) VALUES(?,?,?,?,?,?,?,?)",
                   (name, data, (old[0] if old else 0) + 1, float(mv["dur"]), pj, json.dumps(np.asarray(mv["miss"]).tolist()),
                    float(score), src))
        DB.commit()

def moves_list():
    with LOCK:
        return [r[0] for r in DB.execute("SELECT name FROM moves3d ORDER BY name")]

def lesson_choices():
    return LESSONS + [m for m in moves_list() if m not in LESSONS]

def payload(name, f, dur):
    f = np.asarray(f)
    return {"name": name, "dur": round(float(dur), 2), "f": np.round(f.reshape(len(f), -1), 2).tolist()}

# ---- turning a raw camera take into a clean move ----
def parse_take(frames):
    ts, vs, ms = [], [], []
    for f in frames:
        if len(f.get("v", [])) == NB * 3 and len(f.get("m", "")) == NB:
            ts.append(f["t"]); vs.append(f["v"]); ms.append([c == "1" for c in f["m"]])
    if not ts:
        return None
    return (np.array(ts, float) / 1000.0, np.array(vs, float).reshape(len(ts), NB, 3) / 1000.0, np.array(ms, bool))

def resample_frames(f, n):
    N = len(f)
    if N == n:
        return f.copy()
    x = np.linspace(0, 1, N); xn = np.linspace(0, 1, n)
    out = np.zeros((n, NB, 3))
    for b in range(NB):
        for c in range(3):
            out[:, b, c] = np.interp(xn, x, f[:, b, c])
    return _u(out) * (np.linalg.norm(out, axis=-1, keepdims=True) > 1e-6)

def process_take(t, V, M):
    T = len(t)
    if T < 8:
        return None, "I got too few frames."
    nrm = np.linalg.norm(V, axis=2)
    ok = M & (nrm > 0.5)
    V = V / np.maximum(nrm[..., None], 1e-9) * ok[..., None]
    miss = np.zeros(NB, bool)
    for b in range(NB):
        idv = np.nonzero(ok[:, b])[0]
        if len(idv) < max(3, int(0.15 * T)):
            miss[b] = True; V[:, b] = 0; continue
        for c in range(3):
            V[:, b, c] = np.interp(t, t[idv], V[idv, b, c])
    V = _u(V) * (~miss)[None, :, None]
    body_q = float(ok[:, :15].mean()); hand_q = float(ok[:, 15:].mean())
    use = [b for b in range(NB) if not miss[b]]
    if not use:
        return None, "I could not see your body."
    def dev(ref):
        return np.degrees(np.arccos(np.clip((V[:, use] * V[ref][None, use]).sum(2), -1, 1))).max(1)
    i0 = np.nonzero(dev(0) > 14)[0]; i1 = np.nonzero(dev(T - 1) > 14)[0]
    if len(i0) == 0 or len(i1) == 0:
        return None, "I did not see a clear movement."
    a = max(0, int(i0[0]) - 3); b = min(T - 1, int(i1[-1]) + 3)
    if b - a < 6:
        return None, "The move was too short."
    t, V = t[a:b + 1], V[a:b + 1]
    dur = float(t[-1] - t[0])
    if dur < 0.4:
        return None, "The move was too short."
    n = int(np.clip(round(dur * 20), 14, 200))
    tg = np.linspace(t[0], t[-1], n)
    R = np.zeros((n, NB, 3))
    for bb in range(NB):
        if miss[bb]:
            continue
        for c in range(3):
            R[:, bb, c] = np.interp(tg, t, V[:, bb, c])
    k = np.array([.2, .6, .2])            # one light pass: keeps fast punches fast
    Rp = np.concatenate([R[:1], R, R[-1:]], 0)
    R = k[0] * Rp[:-2] + k[1] * Rp[1:-1] + k[2] * Rp[2:]
    R = _u(R) * (~miss)[None, :, None]
    return dict(f=R, dur=min(dur, 20.0), miss=miss, body_q=body_q, hand_q=hand_q), ""

def move_similarity(old, new):
    A = old["f"]; B = resample_frames(new["f"], len(A))
    use = [b for b in range(NB) if not old["miss"][b] and not new["miss"][b]]
    if not use:
        return 0.0
    err = np.degrees(_ang(A[:, use], B[:, use])).mean()
    return float(max(0.0, 1 - err / 45.0))

def blend_moves(old, new, alpha, parts):
    n = max(len(old["f"]), len(new["f"]))
    A = resample_frames(old["f"], n); B = resample_frames(new["f"], n)
    sel = set(b for p in parts for b in PART_BONES[p])
    out = A.copy(); miss = old["miss"].copy()
    for b in sel:
        if new["miss"][b]:
            continue
        if old["miss"][b]:
            out[:, b] = B[:, b]
        else:
            out[:, b] = _u((1 - alpha) * A[:, b] + alpha * B[:, b])
        miss[b] = False
    dur = (1 - alpha) * old["dur"] + alpha * new["dur"] if len(sel) >= 49 - 15 else old["dur"]
    return dict(f=out, dur=float(dur), miss=miss)

# ---- performing ----
def perform(name, speed=1.0):
    mv = get_move(name)
    if mv is None:
        return None
    t = time.time()
    live.update(kind="text", mood="move", mood_at=t, hold_until=t + mv["dur"] / max(float(speed or 1), 0.2) + 1.5)
    live["last_key"] = ("move", name)
    return face_html("move", "Doing: " + name, {"play": payload(name, mv["f"], mv["dur"]), "speed": float(speed or 1)})

def match_move(message):
    m = re.match(r"^\s*(?:please\s+)?(?:do|perform|show|play|fight|repeat|act)\s+(?:the\s+|me\s+)?(.+?)"
                 r"(?:\s+(?:move|movement|again|please))*\s*[.!?]?\s*$", message or "", re.I)
    if not m:
        return None
    n = m.group(1).strip().lower()
    return n if get_move(n) else None

def move_arm(name, mode, parts):
    n = (name or "").strip().lower()
    if not n:
        return "Type a move name first.", F("confused", "What is the move called?"), V("oops", "What is the move called?")
    known = get_move(n) is not None
    how = ("I will **refine** what I know" if mode == TEACH_REFINE else "I will **replace** my old version") if known else "This is a **new** move"
    return (f"🥊 {how}. Step back so I see your whole body (hands in view for fingers), then do **{n}**. "
            f"I start when you move and stop when you hold still.",
            F("thinking", "Show me the move!"), V("say", f"Show me {n}"))

def reteach_prepare(name, strong):
    n = (name or "").strip().lower()
    if not n or get_move(n) is None:
        return skip(), skip(), "Pick a learned move first, then press Re-teach.", F("confused", "Which move?"), V("oops", "Which move?")
    live["strong"] = bool(strong)
    msg = ("👎 Sorry! Do it again slowly and clearly — I will trust the new version more." if strong
           else "🔁 Do the move again. I will blend it with what I already know.")
    return gr.update(value=n), gr.update(value=TEACH_REFINE), msg, F("thinking", "Teach me again!"), V("say", "Show me again")

def move_save(name, data, mode, parts, speed, prefix=""):
    try:
        d = json.loads(data)
    except Exception:
        return skip(), skip(), skip(), skip()
    base = (name or "").strip().lower()
    if not base:
        return "Type the move name, then record again.", F("confused", "What is it called?"), V("oops"), skip()
    n = prefix + base
    parsed = parse_take(d.get("frames", []))
    if parsed is None:
        return "I got no usable frames — is your whole body in view?", F("confused", "I missed it…"), V("oops"), skip()
    take, why = process_take(*parsed)
    if take is None:
        return (f"{why} Step back, light the room and try again.", F("confused", "I missed it…"), V("oops", "I missed it"), skip())
    old = get_move(n)
    strong = bool(live.pop("strong", False))
    parts = [p for p in (parts or []) if p in PART_BONES] or PART_NAMES
    note = ""
    if old is None:
        mv = take; src = "teach"
    else:
        sim = move_similarity(old, take)
        if mode == TEACH_REPLACE:
            alpha = 1.0
        elif sim < 0.35:
            alpha = 1.0; note = f" (very different from before — {sim:.0%} match — so I trusted the new teaching fully)"
        else:
            alpha = 0.85 if strong else 0.6; note = f" (match with before {sim:.0%})"
        mv = blend_moves(old, take, alpha, parts)
        src = "reteach"
        note = f" · {alpha:.0%} new" + (f" · parts: {', '.join(parts)}" if len(parts) < 4 else "") + note
    save_move3(n, mv, prev=old, src=src, score=take["body_q"])
    hq = take["hand_q"]
    hint = "" if hq > 0.5 else (" ⚠️ I rarely saw your fingers — keep your hands in view and not too far." if hq < 0.15 else " (fingers partly seen)")
    saved = get_move(n)
    return (f"🥊 {'Re-learned' if old else 'Learned'} **{n}** v{saved['n']} · {mv['dur']:.1f}s · {len(mv['f'])} poses · "
            f"body seen {take['body_q']:.0%}, fingers {hq:.0%}{note}{hint} — not exact? press 👎 Not exact / 🔁 Re-teach.",
            perform(n, speed), V("learn", f"I learned {n}. Watch!"), gr.update(choices=moves_list(), value=n))

def undo_move(name):
    n = (name or "").strip().lower()
    with LOCK:
        row = DB.execute("SELECT prev FROM moves3d WHERE name=?", (n,)).fetchone()
    if not row or not row[0]:
        return "Nothing to undo for that move.", F("confused", "No older version."), V("oops"), skip()
    p = json.loads(row[0]); cur = get_move(n)
    prev = dict(f=np.array(p["f"], float).reshape(-1, NB, 3), dur=p["dur"], miss=np.array(p["miss"], bool))
    save_move3(n, prev, prev=cur, src="undo")      # swap: pressing undo again brings back the newer one
    return f"↩ Went back one version of **{n}**.", perform(n), V("say", "Okay, the older one"), gr.update(choices=moves_list(), value=n)

def perform_ui(name, speed):
    f = perform((name or "").strip().lower(), speed)
    if f is None:
        return "Pick a learned move first.", F("confused", "Which move?"), V("oops", "Which move?")
    return f"🥊 Performing **{name}**", f, V("say", f"Watch me do {name}")

# ---- simulation world: body skills (walk / sit / talk / learned moves) ----
def sim_card(hist, lines):
    pts = ""
    if len(hist) > 1:
        top = max(hist) or 1.0
        pts = " ".join(f"{10 + i * (280 / (len(hist) - 1)):.1f},{86 - (e / top) * 70:.1f}" for i, e in enumerate(hist))
    body = "".join(f"<div>{_html.escape(l)}</div>" for l in lines)
    return (f'<div class="board"><svg viewBox="0 0 300 100"><rect x="2" y="2" width="296" height="96" rx="10" fill="#06101f" '
            f'stroke="#fbbf24" stroke-opacity=".6"/><polyline points="{pts}" fill="none" stroke="#34d399" stroke-width="2.5"/>'
            f'<text x="10" y="14" fill="#8fd0ff" font-size="9">cost (lower is better)</text></svg>'
            f'<div class="cap" style="font-size:.8rem;line-height:1.5">{body}</div></div>')

def sim_body_train(lesson, gens, progress=gr.Progress()):
    name = (lesson or "").strip().lower()
    if not name:
        yield "Pick a lesson first.", skip(), F("confused", "Which lesson?"), V("oops"); return
    mv0 = get_move(name)
    if mv0 is not None:
        ref, dur, srcname = resample_frames(mv0["f"], 36), mv0["dur"], "your taught version"
        for b in range(NB):
            if mv0["miss"][b]:
                ref[:, b] = REST[b]
    elif name in LESSON_GEN:
        ref, dur = LESSON_GEN[name](); srcname = "my first (novice) attempt"
    else:
        yield f"I don't know **{name}** yet — teach it in 🥊 Fighting first.", skip(), F("confused", "Teach me first!"), V("oops"); return
    ref = np.asarray(ref, float)
    gens = int(gens)
    live["busy"] = True
    try:
        th0 = np.array([1.0, 1.0, 0.0, 0.0]); lo = np.array([0.3, 0.3, -0.5, 0.0]); hi = np.array([1.4, 1.4, 0.5, 0.45])
        c0, b0, l0, t0 = body_cost(transform(ref, th0), ref)
        yield (f"🧪 Training **{name}** from {srcname} · balance problems {b0:.2f}, joint-limit problems {l0:.2f}",
               sim_card([], [f"start cost {c0:.2f}"]),
               F("learning", f"Practicing {name}…", {"world": "life", "play": payload(name, ref, dur)}), V("learn", f"Let me practice {name}"))
        mu = th0.copy(); sd = np.array([.25, .25, .15, .12]); hist = []; best = (c0, th0)
        step = max(1, gens // 12)
        for g in range(gens):
            pop = np.clip(mu + sd * np.random.randn(40, 4), lo, hi); pop[0] = mu
            costs = np.array([body_cost(transform(ref, p), ref)[0] for p in pop])
            el = np.argsort(costs)[:8]
            mu = pop[el].mean(0); sd = pop[el].std(0) + 0.01
            if costs[el[0]] < best[0]:
                best = (float(costs[el[0]]), pop[el[0]].copy())
            hist.append(float(costs[el[0]]))
            progress((g + 1) / gens, desc=f"Generation {g + 1}/{gens}")
            if g % step == 0 or g == gens - 1:
                yield (f"🧪 generation {g + 1}/{gens} · best cost {best[0]:.2f}",
                       sim_card(hist, [f"best cost {best[0]:.2f} (start {c0:.2f})"]),
                       face_update("learning", f"Practicing {name}…"), skip())
                time.sleep(0.12)
        th = best[1]; Pn = transform(ref, th)
        c1, b1, l1, t1 = body_cost(Pn, ref)
        miss = np.zeros(NB, bool) if mv0 is None else mv0["miss"].copy()
        new = dict(f=Pn, dur=dur, miss=miss)
        save_move3(name, new, prev=mv0, src="sim", score=1 - min(1, b1))
        lines = [f"balance problems {b0:.2f} → {b1:.2f}", f"joint-limit problems {l0:.2f} → {l1:.2f}",
                 f"stays {max(0, 100 - t1 * 100 / 45):.0f}% like the reference (off by {t1:.1f}°)",
                 f"learned: legs ×{th[0]:.2f}, arms ×{th[1]:.2f}, lean {math.degrees(th[2]):+.0f}°, crouch {math.degrees(th[3]):.0f}°"]
        yield (f"✅ **{name}** trained · cost {c0:.2f} → {c1:.2f}. Say `do {name}` anywhere.", sim_card(hist, lines),
               F("happy", f"I can do {name} better!", {"world": "life", "play": payload(name, Pn, dur)}),
               V("done", f"I practiced {name} and got better"))
    except Exception as e:
        yield f"Simulation error: {e}", skip(), F("sad", "Something went wrong."), V("oops")
    finally:
        live["busy"] = False
        live["mood_at"] = time.time()


# ===================== PARTNER ROBOT · LEARN FROM VIDEO · FIGHT ARENA =====================
# Moves of the partner (red) robot are stored with the prefix "b:" (for example "b:jab").
CLIPS = DATA_DIR / "clips"; CLIPS.mkdir(parents=True, exist_ok=True)
WHO_ME = "🧑 Me (cyan robot)"; WHO_PARTNER = "🤖 Partner (red robot)"
PICK_BIG, PICK_LEFT, PICK_RIGHT = "Biggest person", "Person on the left", "Person on the right"

def payload_q(name, f, dur, maxn=90):
    f = np.asarray(f, float)
    if len(f) > maxn:
        f = resample_frames(f, maxn)
    return {"name": name, "dur": round(float(dur), 2), "s": 100, "f": np.round(f.reshape(len(f), -1) * 100).astype(int).tolist()}

def arena_payload():
    mine, partner = [], []
    for n in moves_list():
        mv = get_move(n)
        if mv is None:
            continue
        if n.startswith("b:"):
            if len(partner) < 6:
                partner.append(payload_q(n[2:], mv["f"], mv["dur"]))
        elif len(mine) < 6:
            mine.append(payload_q(n, mv["f"], mv["dur"]))
    defaults = not partner
    if defaults:
        P, d = LESSON_GEN["chain punch"]()
        partner.append(payload_q("chain punch (starter)", P, d))
    return {"mine": mine, "partner": partner, "defaults": defaults}

def open_arena_ui():
    live["hold_until"] = time.time() + 6
    return ("🥊 The fight arena opened in a new 3D window. Press 📷 Camera control there to fight with your own body — "
            "close it with ✖ or Esc.", F("happy", "Let's fight!", {"arena": arena_payload()}), V("say", "Let's fight"))

def yt_download(url, start, length):
    try:
        import yt_dlp
    except ImportError:
        raise RuntimeError("yt-dlp is not installed — run `%pip install yt-dlp` in a Colab cell and try again")
    start = max(0.0, float(start or 0)); length = max(2.0, min(20.0, float(length or 10))); end = start + length
    tag = hashlib.md5(f"{url}|{start}|{end}".encode()).hexdigest()[:10]
    got = [p for p in CLIPS.glob(f"yt_{tag}.*") if p.suffix.lower() in (".mp4", ".webm", ".mov", ".mkv")]
    if got:
        return got[0]
    opts = {"format": "bv*[height<=480][vcodec^=avc1]/b[height<=480][vcodec^=avc1]/bv*[height<=480]/b[height<=480]/worst",
            "outtmpl": str(CLIPS / f"yt_{tag}.%(ext)s"), "noplaylist": True, "quiet": True, "no_warnings": True,
            "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]), "force_keyframes_at_cuts": True,
            "merge_output_format": "mp4", "socket_timeout": 30, "retries": 2}
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])
    got = sorted((p for p in CLIPS.glob(f"yt_{tag}.*") if p.suffix.lower() in (".mp4", ".webm", ".mov", ".mkv")),
                 key=lambda p: p.stat().st_mtime)
    if not got:
        raise RuntimeError("the download finished but no video file was written")
    return got[-1]

def yt_learn_ui(url, file, start, length, who, pick, name, rate):
    n = (name or "").strip().lower()
    if not n:
        yield "Type the move name first (for example `jab`).", F("confused", "What is it called?"), V("oops", "What is it called?")
        return
    url = (url or "").strip(); start = float(start or 0); length = max(2.0, min(20.0, float(length or 10))); uploaded = False
    try:
        if file:
            src = Path(getattr(file, "name", file)); uploaded = True
            path = CLIPS / f"up_{hashlib.md5(str(src).encode()).hexdigest()[:8]}{src.suffix.lower()}"
            if not path.exists():
                shutil.copy(src, path)
        elif url:
            yield (f"⬇ Downloading {start:.0f}–{start + length:.0f} s of the video… (can take a minute)",
                   F("thinking", "Downloading the video…"), V("say", "Downloading the video"))
            path = yt_download(url, start, length)
        else:
            yield "Paste a YouTube link or upload a video file first.", F("confused", "Which video?"), V("oops", "Which video?")
            return
    except Exception as e:
        yield (f"I could not get that video: {str(e)[:180]}. YouTube often blocks Colab downloads — download the video on your own "
               f"computer and upload it in the 🎞 box instead.", F("confused", "I could not get the video."), V("oops", "I could not get the video"))
        return
    q = urllib.parse.quote(str(path), safe="/")
    cmd = {"yt": {"src": "/gradio_api/file=" + q, "src2": "/file=" + q, "pick": {PICK_LEFT: "left", PICK_RIGHT: "right"}.get(pick, "big"),
                  "start": start if uploaded else 0, "len": length, "rate": float(rate or 0.4)}}
    live["hold_until"] = time.time() + length / max(float(rate or 0.4), 0.1) + 25
    who_t = "the partner robot" if who == WHO_PARTNER else "my robot"
    yield (f"👀 Watching the video to teach **{n}** to {who_t}… (the preview below shows what I see)",
           F("thinking", f"Watching {n}…", cmd), V("say", f"Let me watch {n}"))

def yt_save(data, name, who, mode, parts, speed):
    return move_save(name, data, mode, parts, speed, prefix=("b:" if who == WHO_PARTNER else ""))

# ===================== MATH (inside chat: calc: 2+3*4) =====================
def calc_eval(message):
    m = re.match(r"^\s*(?:calc|calculate)\s*:\s*(.+)$", message or "", re.I)
    if not m:
        return None
    expr = m.group(1)
    ops = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
    def walk(n):
        if isinstance(n, ast.Expression): return walk(n.body)
        if isinstance(n, ast.Constant) and type(n.value) in (int, float): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in ops:
            a, b = walk(n.left), walk(n.right)
            if isinstance(n.op, ast.Pow) and abs(b) > 8: raise ValueError("exponent too large")
            return ops[type(n.op)](a, b)
        if isinstance(n, ast.UnaryOp) and type(n.op) in ops: return ops[type(n.op)](walk(n.operand))
        raise ValueError("use numbers and + - * / ** only")
    try:
        return True, expr, walk(ast.parse(expr, mode="eval"))
    except Exception as e:
        return False, expr, str(e)


# ===================== SMALL TALK · WHAT I LEARNED · FEATURE HELP =====================
EXPLAIN_LISTEN = ("👂 **Learn by listening**: when it is switched on I keep listening through the microphone — to you, or to a lecture "
                  "playing near it — and I remember every sentence I hear as knowledge. I stay quiet so I do not talk over you. "
                  "Later ask me “what did you learn?” and I will tell you. Switch it off to stop.")
EXPLAIN_TALK = ("🗣 **Talk mode**: a voice conversation. I greet you (“How are you?”), listen to you, answer out loud, then listen again. "
                "Switch it off to stop. If I do not know something I will say so — teach me with “remember this: …”.")

def _things(n):
    return f"{n} thing" + ("" if n == 1 else "s")

def learned_summary():
    with LOCK:
        rows = DB.execute("SELECT source, COUNT(*) FROM facts GROUP BY source").fetchall()
        heard = [r[0] for r in DB.execute("SELECT text FROM facts WHERE source='heard' ORDER BY id DESC LIMIT 4").fetchall()]
        taught = [r[0] for r in DB.execute("SELECT text FROM facts WHERE source IN ('chat-taught','user-taught','user-correction') ORDER BY id DESC LIMIT 3").fetchall()]
        sk = [r[0] for r in DB.execute("SELECT name FROM sketches ORDER BY name LIMIT 8").fetchall()]
    n_heard = sum(c for s_, c in rows if s_ == "heard")
    n_doc = sum(c for s_, c in rows if (s_ or "").startswith("document:"))
    n_typed = sum(c for s_, c in rows if s_ in ("chat-taught", "user-taught", "user-correction"))
    objs = list(idx.by_label)[:8]; mv = [m for m in moves_list()][:8]
    if not (n_heard or n_doc or n_typed or objs or sk or mv):
        t = "I haven't learned anything yet. Teach me by typing, reading a file, showing me things, or switch on 👂 Learn by listening."
        return t, t
    lines = ["Here is what I have learned so far:"]
    if n_heard: lines.append(f"- 👂 **by listening**: {n_heard} sentence(s). Latest: " + " · ".join(f"“{h[:80]}”" for h in heard))
    if n_typed: lines.append(f"- 📝 **taught by typing**: {n_typed}. Latest: " + " · ".join(f"“{h[:80]}”" for h in taught))
    if n_doc: lines.append(f"- 📚 **from documents**: {n_doc} pieces")
    if objs: lines.append("- 👁 **things I can recognise**: " + ", ".join(objs))
    if sk: lines.append("- ✏️ **drawings**: " + ", ".join(sk))
    if mv: lines.append("- 🥊 **moves**: " + ", ".join(mv))
    spoken = "I learned "
    parts = []
    if n_heard: parts.append(f"{_things(n_heard)} by listening")
    if n_typed: parts.append(f"{_things(n_typed)} you taught me")
    if n_doc: parts.append(f"{n_doc} pieces from documents")
    if objs: parts.append(f"{len(objs)} objects")
    if sk: parts.append(f"{len(sk)} drawings")
    if mv: parts.append(f"{len(mv)} moves")
    spoken += ", ".join(parts[:-1]) + (" and " if len(parts) > 1 else "") + parts[-1] + "."
    if heard:
        spoken += " The latest thing I heard was: " + heard[0][:90]
    return "\n".join(lines), spoken

def smalltalk(message, talk=False):
    """Friendly default replies. Returns (reply, mood, caption, say) or None."""
    t = re.sub(r"[^a-z0-9' ]", " ", (message or "").lower()); t = re.sub(r"\s+", " ", t).strip()
    if re.match(r"^talk[- ]?start\b", t):
        convo["asked"] = "how"
        return "Hi! Talk mode is on. How are you today?", "happy", "How are you today?", "Hi! How are you today?"
    if re.search(r"(what|explain|tell).{0,25}(learn(ing)? by listening|listen(ing)? mode|learn from listening)", t):
        return EXPLAIN_LISTEN, "happy", "That is my listening mode!", "When learn by listening is on, I listen and remember everything I hear. Ask me what did you learn, and I will tell you."
    if re.search(r"(what|explain|tell).{0,25}talk mode", t):
        return EXPLAIN_TALK, "happy", "That is my talk mode!", "Talk mode is a voice conversation. I greet you, listen, answer, and listen again."
    if re.search(r"what (have |did |do |all )?(you|u) (learn|learned|learnt|know|heard|hear)|tell me what you (learned|learnt|know|heard)|what.{0,18}(learned|learnt) (so far|today|from)|what have you been learning", t):
        text, say = learned_summary()
        return text, "learning", "Here is what I learned!", say
    if re.match(r"^(how are you|how are u|how r u|how do you do|how is it going|how's it going|hows it going|what's up|whats up|how have you been)\b", t):
        return "I'm doing great, thank you for asking! 😊 How are you?", "happy", "I'm great! How are you?", "I'm doing great, thank you for asking! How are you?"
    if re.match(r"^(hi|hello|hey|hii+|namaste|good (morning|afternoon|evening)|hello there|hey there)\b", t) and len(t.split()) <= 4:
        return "Hello! 👋 How are you today?", "happy", "Hello! How are you?", "Hello! How are you today?"
    if re.match(r"^(i am|i'm|im|i feel) (fine|good|great|well|okay|ok|happy|excellent|awesome|doing well|doing good)\b|^(fine|good|great|not bad|i am fine)\b", t):
        return "That is wonderful to hear! 😄 What would you like to teach me today?", "happy", "Wonderful!", "That is wonderful to hear! What would you like to teach me today?"
    if re.match(r"^(i am|i'm|im|i feel) (sad|tired|bad|not good|not well|upset|angry|sick|lonely|stressed|unhappy)\b|^not (good|well|great)\b", t):
        return ("I'm sorry to hear that. 💙 I'm here with you. Do you want to tell me about it, or shall we do something fun?",
                "sad", "I'm here for you.", "I'm sorry to hear that. I'm here with you. Do you want to tell me about it, or shall we do something fun?")
    m = re.match(r"^(?:my name is|call me|i am called|this is) ([a-z][a-z ]{0,20})$", t)
    if m:
        nm = m.group(1).strip().title(); add_fact(f"The user's name is {nm}.", "chat-taught")
        return f"Nice to meet you, {nm}! 😊 I will remember your name.", "happy", f"Nice to meet you, {nm}!", f"Nice to meet you, {nm}! I will remember your name."
    if re.match(r"^(what is my name|what's my name|do you know my name|who am i)\b", t):
        with LOCK:
            r = DB.execute("SELECT text FROM facts WHERE text LIKE 'The user''s name is %' ORDER BY id DESC LIMIT 1").fetchone()
        if r:
            nm = r[0].replace("The user's name is ", "").rstrip(".")
            return f"Your name is {nm}!", "happy", f"You are {nm}!", f"Your name is {nm}!"
        return "I don't know your name yet. Tell me: “my name is …”.", "confused", "What is your name?", "I don't know your name yet. What is your name?"
    if re.match(r"^(thanks|thank you|thank u|thx)\b", t):
        return "You're welcome! 😊", "happy", "You're welcome!", "You're welcome!"
    if re.match(r"^(bye|goodbye|good bye|see you|see ya|good night)\b", t):
        return "Goodbye! Come back soon. 👋", "happy", "Goodbye!", "Goodbye! Come back soon."
    if re.match(r"^(who are you|what are you|what is your name|what's your name|your name)\b", t):
        return ("I'm your learning robot. I learn from text, files, pictures, drawings, videos and the moves you show me.",
                "happy", "I'm your learning robot!", "I'm your learning robot. I learn from text, pictures, drawings, and moves you show me.")
    if re.match(r"^(what can you do|help|what do you do|how do you work)\b", t):
        return ("I can: 📝 learn what you type or read from files · 👂 learn by listening · 👁 recognise things on camera · ✏️ draw what you teach me · "
                "🥊 learn body moves and fingers from you or from a video · 🧪 practise in the simulation room.",
                "happy", "Here is what I can do!", "I can learn from text, listening, pictures, drawings, and fighting moves. Try teaching me something!")
    return None


# ===================== CONVERSATION =====================
convo = {"asked": None, "gen": 0, "joke": 0}
FACT_Q = re.compile(r"^(what|who|whom|whose|when|where|why|which|how (?:many|much|old|far|long|tall|big|does|did|to)|is|are|was|were|explain|define|tell me about|capital|meaning)\b")
SOCIAL_Q = re.compile(r"^((can|could|do|are|will|would|did|have) you\b|how are you)")
CONV_START = re.compile(r"^(i |i'm|im |my |you |you're|youre|we |let's|lets |ok\b|okay|yes|no\b|yeah|nope|maybe|please|wow|cool|nice|great|really|oh\b|hmm|haha|lol|thanks|good|bad)")
POS_W = set("good great fine happy awesome well excellent nice wonderful amazing okay ok cool better fantastic super lovely glad".split())
NEG_W = set("sad bad tired sick angry upset terrible awful lonely stressed worried depressed unhappy hurt".split())
SPECIAL_RE = re.compile(r"\b(joke|what time is it|what('s| is) the time|time now|today'?s date|what('s| is) the date|what day is it|what is today|i love you|i like you)\b")
JOKES = ["Why did the robot go to school? To improve its byte-ing skills!", "What do you call a robot who loves to dance? A bot with rhythm!",
         "Why was the computer cold? It left its Windows open!", "I told my battery a joke, but it was not charged up to laugh!",
         "Why do robots never panic? They have nerves of steel!"]
_EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\uFE0F\u200d]")

def strip_emoji(x):
    return _EMOJI.sub("", x or "")

def _norm(m):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9' ]", " ", (m or "").lower())).strip()

def R(reply, mood="happy", cap=None):
    return reply, mood, cap or strip_emoji(reply)[:34], reply

def note_asked(st):
    if st and re.search(r"how are you", st[0], re.I):
        convo["asked"] = "how"
    return st

def convo_reply(t, message):
    """Answers to the question I just asked (“How are you?”)."""
    if convo.get("asked") != "how":
        return None
    if "?" in (message or "") or FACT_Q.match(t):
        convo["asked"] = None; return None
    words = set(t.split())
    neg = bool(words & NEG_W) or bool(re.search(r"not (so |very )?(good|well|great|okay|ok)|so so|not feeling", t))
    pos = bool(words & POS_W) and not neg
    if not (neg or pos) and smalltalk(message):
        return None
    convo["asked"] = None
    if neg:
        return R("I'm sorry you feel that way. 💙 I'm here and I'm listening. Do you want to tell me what happened?", "sad", "I'm here for you.")
    if pos:
        return R("That's wonderful to hear! 😄 What made your day so good?", "happy", "That's wonderful!")
    return R("Thank you for telling me! 🙂 What would you like to do today — teach me something, draw, or practise a move?", "happy", "Thanks for telling me!")

def _special(t):
    if re.search(r"\b(joke|funny)\b", t):
        convo["joke"] += 1; return R(JOKES[convo["joke"] % len(JOKES)] + " 😄", "happy", "Here is a joke!")
    if re.search(r"\bwhat('s| is) the time\b|\bwhat time is it\b|\btime now\b", t):
        return R("It is " + datetime.now().strftime("%I:%M %p").lstrip("0") + " right now. ⏰", "happy", "It's " + datetime.now().strftime("%I:%M %p"))
    if re.search(r"\b(today'?s date|what('s| is) the date|what day is it|what is today)\b", t):
        return R("Today is " + datetime.now().strftime("%A, %d %B %Y") + ". 📅", "happy", "Today's date!")
    if re.search(r"\bi love you\b|\bi like you\b", t):
        return R("Aww, thank you! 💙 I like talking with you too.", "happy", "Aww, thank you!")
    if re.search(r"\byou('re| are) (so )?(smart|good|great|awesome|nice|funny|cute|clever|amazing)\b", t):
        return R("Thank you, that makes me happy! 😊 I learn from you.", "happy", "Thank you!")
    if re.search(r"\byou('re| are) (so )?(stupid|bad|dumb|useless|slow)\b", t):
        return R("I'm sorry I'm not perfect yet. 🙏 Teach me and I will get better.", "sad", "I'll try harder.")
    return None

def reflect(message, t):
    """Friendly answer to anything else, so a spoken conversation never stalls."""
    sp = _special(t)
    if sp:
        return sp
    m = re.match(r"^i (?:really )?(?:like|love|enjoy|prefer) (.+)$", t)
    if m:
        return R(f"That sounds lovely! 😊 What do you like most about {m.group(1)}?", "happy")
    m = re.match(r"^i (?:hate|dislike|don't like|do not like) (.+)$", t)
    if m:
        return R(f"I understand. What don't you like about {m.group(1)}?", "thinking")
    m = re.match(r"^i (?:want to|wanna|would like to|'d like to|need to|am going to|will) (.+)$", t)
    if m:
        return R(f"That sounds interesting! 🤔 Why do you want to {m.group(1)}?", "thinking")
    m = re.match(r"^(?:can|could|will|would) you (.+)$", t)
    if m:
        return R(f"I'm still learning to {m.group(1)}. 🙂 If you show me how, I can learn it!", "thinking")
    m = re.match(r"^do you (?:like|love|enjoy) (.+)$", t)
    if m:
        return R(f"I'm a robot, but I enjoy learning about {m.group(1)}! Do you like it?", "happy")
    if re.match(r"^(yes|yeah|yep|sure|okay|ok|of course)\b", t):
        return R("Great! 😊 What would you like to do next?", "happy")
    if re.match(r"^(no|nope|not really|nah)\b", t):
        return R("Okay, no problem. 🙂 Is there something else you would like to talk about?", "thinking")
    if re.match(r"^(wow|cool|nice|great|awesome|amazing|really|oh)\b", t):
        return R("Isn't it? 😄 Tell me more!", "happy")
    convo["gen"] += 1
    n_w = len(t.split())
    opts = ["I see. 🙂 Tell me more about that!", "That's interesting! 🤔 What else can you tell me?",
            "Hmm, I'm still learning about that. Would you like to teach me? Say “remember this” and then the fact."]
    if n_w >= 4 and convo["gen"] % 3 == 0:
        return R("Would you like me to remember that? 📝 Say “remember this” followed by what you want me to learn.", "thinking")
    return R(opts[convo["gen"] % 2], "thinking")

# ===================== CHAT AGENT =====================
def answer(message, talk=False, history=None):
    """Returns (reply, mood, caption, speech)."""
    live["kind"] = "text"
    m = re.match(r"^\s*listen[- ]?learn\s*:\s*(.+)$", message, re.I | re.S)
    if m:                                   # heard while "Learn by listening" is on: learn quietly, no talking over the teacher
        heard = m.group(1).strip()
        return f"👂 {add_fact(heard, 'heard')} — “{heard[:90]}”", "learning", "I'm listening and learning…", ""
    m = re.match(r"^\s*(?:remember this|teach yourself|learn this)\s*:\s*(.+)$", message, re.I | re.S)
    if m:
        return add_fact(m.group(1), "chat-taught"), "learning", "I learned something new!", "I learned something new!"
    c = calc_eval(message)
    if c:
        ok, expr, val = c
        if ok:
            return f"`{expr}` = **{val}**", "happy", "Math is fun!", f"The answer is {val}"
        return f"Can't calculate: {val}", "confused", "Hmm…", "I can't calculate that"
    if re.search(r"\bwhat (do|can) you see\b|\bwhat is this\b", message, re.I):
        if last_vis["label"]:
            return (f"I last saw **{last_vis['label']}**{know_text(last_vis['label'])}",
                    "happy", f"I saw {last_vis['label']}", f"I saw {last_vis['label']}")
        return "I haven't seen anything yet.", "confused", "I haven't seen anything.", "I haven't seen anything yet"
    t = _norm(message)
    ctx = convo_reply(t, message)                    # a reply to what I just asked
    if ctx:
        last.update(query=message, used=[]); return ctx
    st0 = smalltalk(message, talk)
    if st0 and re.search(r"learn|talk[- ]?start|listening|talk mode|how are|hello|^\s*hi\b|hey|my name|thank|bye", message, re.I):
        last.update(query=message, used=[])
        return note_asked(st0)
    ranked = rank(message)
    if ranked and CONV_START.match(t) and not FACT_Q.match(t) and ranked[0][1] < 0.7:
        ranked = []                                   # chatty sentence, not a question about my memory
    if not ranked:
        last.update(query=message, used=[])
        st = smalltalk(message, talk)
        if st:
            return note_asked(st)
        if SPECIAL_RE.search(t):
            return reflect(message, t)
        if FACT_Q.match(t) and not SOCIAL_Q.match(t):
            return ("I don't know that yet. 🤔 Teach me by saying: “remember this” and then the fact.", "confused", "I don't know that yet…",
                    "I don't know that yet. You can teach me by saying, remember this, and then the fact.")
        r = chat_llm(message, history)
        if r:
            return r, "happy", strip_emoji(r)[:34], r
        return reflect(message, t)
    cands = ranked[:TOP_K]
    explored = len(cands) > 1 and random.random() < epsilon()
    chosen = random.choice(cands[1:]) if explored else cands[0]
    others = [r for r in cands if r is not chosen][:2]
    last.update(query=message, used=[(chosen[2]["id"], 1.0)] + [(o[2]["id"], 0.3) for o in others])
    facts = [chosen[2]["text"]] + [o[2]["text"] for o in others]
    text = phrase(message, facts)
    return text, ("thinking" if explored else "happy"), ("Let me try another…" if explored else "I know this!"), text

def chat_send(message, history, talk=False):
    if not (message or "").strip():
        return history or [], "", skip(), skip()
    history = list(history or [])
    mv = match_move(message)
    if mv:
        history += [{"role": "user", "content": message.strip()},
                    {"role": "assistant", "content": f"🥊 Performing **{mv}** exactly as you taught me."}]
        return history, "", perform(mv), V("say", f"Watch me do {mv}")
    if re.match(r"^\s*(?:let'?s\s+|please\s+|i want to\s+|start\s+|open\s+(?:the\s+)?)?(?:fight|fighting|spar|sparring|duel|(?:fight )?arena)\s*[.!?]?\s*$", message, re.I):
        history += [{"role": "user", "content": message.strip()},
                    {"role": "assistant", "content": "🥊 Opening the fight arena in a new 3D window — close it with ✖ or Esc."}]
        live["hold_until"] = time.time() + 6
        return history, "", F("happy", "Let's fight!", {"arena": arena_payload()}), V("say", "Let's fight")
    reply, mood, cap, say = answer(message.strip(), bool(talk), history)
    shown = "👋 Talk mode on" if message.strip().lower().startswith("talk-start") else message.strip()
    history += [{"role": "user", "content": shown}, {"role": "assistant", "content": reply}]
    return history, "", F(mood, cap), (V("say", say_clean(strip_emoji(say))) if say else V("tick"))

def rate(kind):
    if not last["used"]:
        return "Ask something first.", F("confused", "Ask me first!"), V("oops", "Ask me first")
    reward = 1.0 if kind == "good" else -1.0
    feedback[kind] = int(feedback.get(kind, 0)) + 1
    rl_update(last["query"], last["used"], reward)
    save_json(FEEDBACK_PATH, feedback)
    msg = f"👍 {feedback['good']}  👎 {feedback['bad']}"
    return (msg, F("happy", "Thanks!"), V("ok", "Thank you!")) if reward > 0 else \
           (msg, F("sad", "Oops!"), V("oops", "Oops, I will do better"))

def rate_any(kind):
    if live["kind"] == "draw":
        return draw_rate(kind)
    if live["kind"] == "vis" and last_vis["used"]:
        return camera_rate(kind)
    return rate(kind)

def teach_ui(text):
    if not (text or "").strip():
        return "Type something to teach.", F("confused", "What should I learn?"), V("oops"), skip()
    return add_fact(text, "user-taught"), F("learning", "Yum! I learned!"), V("learn", "Yum! I learned it"), ""

def fix_any(text):
    text = (text or "").strip()
    if not text:
        return "Type the correct answer in the box.", F("confused", "What is right?"), V("oops"), skip()
    if live["kind"] == "draw":
        return ("For drawings: press 👎, then draw it again on the canvas and press 💾 Teach, or train me in 🧪 Simulation.",
                F("thinking", "Let me practice…"), V("say", "Let me practice"), skip())
    if live["kind"] == "vis" and last_vis["feat"] is not None:
        msg, face, voice = camera_correct(text)
        return msg, face, voice, ""
    q = last["query"]
    if q and last["used"]:
        rl_update(q, [(last["used"][0][0], 1.0)], -1.0)
    _, ids = add_fact(text, "user-correction", return_ids=True)
    if q:
        rl_update(q, [(i, 1.0) for i in ids], +1.0)
    return "✏️ Fixed.", F("learning", "Thanks! Fixed it."), V("learn", "Thank you, I fixed it"), ""

# ===================== DOCUMENT IMPORT =====================
DOC_EXT = (".pdf", ".txt", ".md")

def _import_paths(paths, progress):
    """Generator of (info, face, memory_box, voice). Reads PDF / TXT / MD files into the text memory."""
    live["busy"] = True
    try:
        paths = [Path(p) for p in paths]
        if not paths:
            yield "No file to read.", F("confused", "Give me a file!"), skip(), V("oops")
            return
        pieces_total = new_total = 0; done = []; bad = []
        for fi, path in enumerate(paths):
            yield (f"📖 Reading {path.name} ({fi + 1}/{len(paths)})…", F("thinking", f"Reading {path.name}…"), skip(), skip())
            src = f"document:{path.name}"
            try:
                if path.suffix.lower() == ".pdf":
                    pages = PdfReader(str(path)).pages
                    n, buf = len(pages), []
                    for i, p in enumerate(pages):
                        try:
                            buf.append(p.extract_text() or "")
                        except Exception:
                            buf.append("")
                        if len(buf) >= 10 or i == n - 1:
                            pieces = chunk_text("\n".join(buf), 320)
                            _, new = _add_pieces(pieces, src)
                            pieces_total += len(pieces); new_total += new; buf = []
                        progress((i + 1) / max(n, 1), desc=f"{path.name}: page {i + 1}/{n}")
                elif path.suffix.lower() in (".txt", ".md"):
                    pieces = chunk_text(path.read_text(encoding="utf-8", errors="ignore"), 320)
                    _, new = _add_pieces(pieces, src)
                    pieces_total += len(pieces); new_total += new
                else:
                    bad.append(path.name); continue
                done.append(path.name)
            except Exception as e:
                bad.append(f"{path.name} ({e})")
        if pieces_total == 0:
            why = ("; ".join(bad) if bad else "no text found (scanned PDFs need OCR)")
            yield f"I could not read anything: {why}", F("confused", "No text in it…"), skip(), V("oops")
            return
        extra = f" · skipped: {', '.join(bad)}" if bad else ""
        yield (f"📚 {len(done)} file(s) · {pieces_total:,} chunks ({new_total:,} new) · {count_facts():,} memories{extra}",
               F("learning", "I read the whole thing!"), memory_list(), V("learn", "Yum! I read the whole thing"))
    except Exception as e:
        yield f"Import error: {e}", F("sad", "Something went wrong."), skip(), V("oops")
    finally:
        live["busy"] = False
        live["mood_at"] = time.time()

def import_doc(files, progress=gr.Progress()):
    if not files:
        yield skip(), F("confused", "Give me a file!"), skip(), skip()
        return
    items = files if isinstance(files, (list, tuple)) else [files]
    yield from _import_paths([getattr(f, "name", f) for f in items], progress)

def import_path_ui(text, progress=gr.Progress()):
    t = (text or "").strip().strip("\"'")
    if not t:
        yield "Type the path of a file or folder first (for example /content/book.pdf).", F("confused", "Which file?"), skip(), V("oops")
        return
    p = Path(t).expanduser()
    if p.is_dir():
        paths = sorted(x for x in p.iterdir() if x.suffix.lower() in DOC_EXT)
    elif p.exists():
        paths = [p]
    else:
        yield (f"I can't find `{p}`. In Colab open the 📁 Files panel, upload the file there, then paste its path "
               f"(for example /content/book.pdf).", F("confused", "I can't find it."), skip(), V("oops"))
        return
    yield from _import_paths(paths, progress)

# ===================== BACKUP =====================
def make_backup():
    with LOCK:
        idx.flush()
        out = DATA_DIR / "brain_backup.db"
        if out.exists():
            out.unlink()
        bk = sqlite3.connect(str(out))
        DB.backup(bk)
        bk.close()
    return str(out)

def restore_backup(file):
    if file is None:
        return "Choose a .db file.", skip(), skip()
    try:
        with LOCK:
            src = sqlite3.connect(str(getattr(file, "name", file)))
            src.execute("SELECT COUNT(*) FROM facts")
            src.backup(DB)
            src.close()
            ensure_schema()
            idx.load()
        _kc.clear()
        return f"Restored: {count_facts():,} memories, {idx.n:,} views.", memory_list(), F("learning", "My memory is back!")
    except Exception as e:
        return f"Restore failed: {e}", skip(), F("sad", "Couldn't restore.")

# ===================== LEARNING DROP-DOWN =====================
SIM_BODY = "🚶 Body skills (walk, sit, talk, moves)"
SIM_DRAW = "🖌 Drawing practice"
WORLD_OF = {MODE_TEXT: "home", MODE_SEE: "home", MODE_DRAW: "draw", MODE_SIM: "life", MODE_FIGHT: "fight"}
MODE_HELP = {
    MODE_FIGHT: ("🥊 **Fighting ring** — 📷 Camera on, stand back so your whole body and hands are seen, type a move name, ● Record. "
                 "🪞 Live mirror makes the robot copy you in real time. 🥊 Open fight arena = a separate 3D window where you fight a partner robot "
                 "with your own body. Open “Learn from a YouTube link” to teach you or the partner robot from a video (left/right fighter). "
                 "Not exact? 🔁 Re-teach / 👎 Not exact."),
    MODE_DRAW: "✏️ **Drawing studio** — draw on the paint canvas, name it, press 💾 Teach. The robot draws it on the glass canvas in 3D.",
    MODE_SEE: "👁 **Seeing** — turn the camera on, type a name, press ▶ Learn.",
    MODE_SIM: "🧪 **Simulation room** — pick a lesson (walk, sit down, talk, horse stance, chain punch or a move you taught) and press ▶ Train.",
    MODE_TEXT: "💬 **Text** — write in the big box and press ＋ Teach.",
}

def sim_world(kind):
    w = "draw" if (kind or "").startswith("🖌") else "life"
    return F("learning", "Simulation world", {"world": w})

def mode_change(m, cur_move, cur_lesson):
    """The panels are shown/hidden instantly by the browser (applyMode). This only updates the help text, the 3D world and the lists."""
    soon = m in SOON
    note = f"**{m}** — only the name is here for now. This skill is not available yet." if soon else ""
    live["kind"] = "text"
    names = moves_list(); ch = lesson_choices()
    mv_up = gr.update(choices=names, value=(cur_move if cur_move in names else (names[0] if names else None))) if m == MODE_FIGHT else skip()
    ls_up = gr.update(choices=ch, value=(cur_lesson if cur_lesson else ch[0])) if m == MODE_SIM else skip()
    return (gr.update(value=note), F("drawing" if m == MODE_DRAW else "learning", m.split(" (")[0], {"world": WORLD_OF.get(m, "home")}),
            MODE_HELP.get(m, ""), mv_up, ls_up)

# ===================== BROWSER JS (voice, mic, sounds, canvas, 3D stage, pose + finger capture) =====================
JS_LIB = r"""
(function(){
if(window._lib) return; window._lib = 1;

window.sfx = function(kind){
  try{
    var C = window._ac || (window._ac = new (window.AudioContext||window.webkitAudioContext)());
    if(C.state==='suspended') C.resume();
    var seq = {learn:[[523,0],[659,.09],[784,.18],[1047,.27]], tick:[[1250,0]], done:[[784,0],[988,.12],[1319,.24]],
               oops:[[330,0],[247,.14]], ok:[[880,0],[1175,.1]], blip:[[700+Math.random()*800,0]], say:[]}[kind] || [];
    var soft = (kind==='tick' || kind==='blip');
    seq.forEach(function(p){
      var o=C.createOscillator(), g=C.createGain(), s=C.currentTime+p[1];
      o.type = kind==='tick' ? 'sine' : 'triangle'; o.frequency.value = p[0];
      g.gain.setValueAtTime(0.0001,s);
      g.gain.exponentialRampToValueAtTime(soft?0.04:0.14,s+0.02);
      g.gain.exponentialRampToValueAtTime(0.0001,s+(soft?0.09:0.22));
      o.connect(g); g.connect(C.destination); o.start(s); o.stop(s+0.25);
    });
  }catch(e){}
};

window._ls = {wanted:false};
window.cbOn = function(sel){ var c = document.querySelector(sel + ' input[type=checkbox]'); return !!(c && c.checked); };
window.listenSay = function(t){ var s = document.getElementById('listen_status'); if(s) s.textContent = t || ''; };

window.babySpeak = function(text){
  if(!window.speechSynthesis || !text) return;
  try{ speechSynthesis.cancel(); speechSynthesis.resume(); }catch(e){}
  try{ if(window._rec){ window._rec.abort(); } }catch(e){}       // do not listen to myself
  var u = new SpeechSynthesisUtterance(text);
  var vs = speechSynthesis.getVoices();
  var v = vs.find(function(x){return /female|zira|samantha|aria|jenny|google uk english female|google us english/i.test(x.name);})
          || vs.find(function(x){return /^en/i.test(x.lang);});
  if(v) u.voice = v;
  u.pitch = 1.8; u.rate = 1.05; u.volume = 1;
  var f = document.getElementById('face_box');
  u.onstart = function(){ window._talking = true; f && f.classList.add('talking'); };
  u.onboundary = function(){ window.sfx('blip'); };            // a little chirp on every word
  u.onerror = function(){ window._talking = false; f && f.classList.remove('talking'); };
  u.onend = function(){
    window._talking = false; f && f.classList.remove('talking');
    if(window._u !== u) return;                                // ignore cancelled utterances
    if(window.cbOn('#talk_mode') || window.cbOn('#listen_learn')) setTimeout(function(){ window.syncListen && window.syncListen(); }, 500);
  };
  window._u = u;
  setTimeout(function(){ speechSynthesis.speak(u); }, 80);
};

window.onVoice = function(v,on){
  if(!v || v===window._lastVoice) return;
  window._lastVoice = v;
  if(!on) return;
  var p = v.split('||');
  window.sfx(p[0]);
  if(p[1]) window.babySpeak(p[1]);
};

// send what was heard to the chat box (as if typed)
window.sendHeard = function(t, talk, learn){
  var msg = t;
  if(learn && !talk) msg = 'listen-learn: ' + t;              // quiet learning, no talking over the teacher
  else if(learn && talk) msg = 'remember this: ' + t;
  window.listenSay('heard: ' + t);
  var ta = document.querySelector('#msg_box textarea, #msg_box input');
  if(!ta) return;
  var proto = ta.tagName==='TEXTAREA' ? HTMLTextAreaElement.prototype : HTMLInputElement.prototype;
  Object.getOwnPropertyDescriptor(proto,'value').set.call(ta, msg);
  ta.dispatchEvent(new Event('input',{bubbles:true}));
  setTimeout(function(){ var b = document.querySelector('#ask_btn'); b && b.click(); }, 150);
};

// auto=true: started by the Talk mode / Learn-by-listening switches (keeps listening again and again)
window.startListen = function(auto){
  var SR = window.SpeechRecognition || window.webkitSpeechRecognition;
  var f = document.getElementById('face_box');
  if(!SR){ window.listenSay('⚠️ Listening needs Chrome or Edge.'); if(!auto) alert('Speech recognition works in Chrome or Edge.'); return; }
  try{ if(window._rec){ window._rec.onend = null; window._rec.abort(); } }catch(e){}
  var talk = window.cbOn('#talk_mode'), learn = window.cbOn('#listen_learn');
  var r = new SR(); r.lang = 'en-US'; r.interimResults = false; r.maxAlternatives = 1; r.continuous = (learn && !talk);
  window._rec = r;
  r.onstart = function(){
    window._listening = true; f && f.classList.add('listening'); if(!auto) window.sfx('tick');
    window.listenSay(talk ? '🎙 Listening… talk to me' : (learn ? '👂 Learning by listening… say things for me to learn' : '🎙 Listening…'));
  };
  r.onend = function(){
    window._listening = false; f && f.classList.remove('listening');
    if(window._rec === r) window._rec = null;
    if(window._ls.wanted) setTimeout(function(){ if(!window._rec && window._ls.wanted && !window._talking) window.startListen(true); }, 450);
    else window.listenSay('');
  };
  r.onerror = function(e){
    if(e.error === 'not-allowed' || e.error === 'service-not-allowed'){
      window._ls.wanted = false; window.listenSay('🚫 The microphone is blocked — click the 🔒 icon in the address bar and allow it.');
    } else if(e.error === 'network'){ window.listenSay('⚠️ Listening needs internet (the browser sends speech to its recogniser).'); }
  };
  r.onresult = function(e){
    for(var i = e.resultIndex; i < e.results.length; i++){
      if(!e.results[i].isFinal) continue;
      var t = (e.results[i][0].transcript || '').trim(); if(!t) continue;
      window.sendHeard(t, talk, learn);
    }
  };
  try{ r.start(); }catch(err){}
};
// follow the two switches: on = listen (and keep listening), off = stop
window.syncListen = function(){
  var want = window.cbOn('#talk_mode') || window.cbOn('#listen_learn');
  window._ls.wanted = want;
  if(!want){ try{ if(window._rec){ window._rec.onend = null; window._rec.abort(); } }catch(e){} window._rec = null; window._listening = false; window.listenSay(''); return; }
  if(!window._rec && !window._talking) window.startListen(true);
};
document.addEventListener('change', function(e){
  var t = e.target;
  if(t && t.type === 'checkbox' && t.closest && (t.closest('#talk_mode') || t.closest('#listen_learn'))){
    window.syncListen();
    if(t.checked && t.closest('#talk_mode')) setTimeout(window.talkGreet, 300);          // talk mode: greet first ("How are you?"), then listen
  }
});

window.sendText = function(msg){
  var ta = document.querySelector('#msg_box textarea, #msg_box input'); if(!ta) return;
  var proto = ta.tagName==='TEXTAREA' ? HTMLTextAreaElement.prototype : HTMLInputElement.prototype;
  Object.getOwnPropertyDescriptor(proto,'value').set.call(ta, msg); ta.dispatchEvent(new Event('input',{bubbles:true}));
  setTimeout(function(){ var b = document.querySelector('#ask_btn'); b && b.click(); }, 150);
};
window.toggleVoiceChat = function(){
  var c = document.querySelector('#talk_mode input[type=checkbox]'); if(!c){ window.startListen && window.startListen(false); return; }
  c.click();          // same as Talk mode: greet, listen, answer, listen again
  var b = document.getElementById('mic_btn'); if(b) b.classList.toggle('on', c.checked);
};
window.talkGreet = function(){ if(window.cbOn('#talk_mode')) window.sendText('talk-start:'); };
// show only the panel of the chosen learning mode, instantly, in the browser (no server round-trip)
window.applyMode = function(){
  var inp = document.querySelector('#mode_dd input'), m = inp ? inp.value : '', k = 'text';
  if(m.indexOf('Seeing') >= 0) k = 'see'; else if(m.indexOf('Drawing') >= 0) k = 'draw'; else if(m.indexOf('Simulation') >= 0) k = 'sim';
  else if(m.indexOf('Fighting') >= 0) k = 'fight'; else if(m && m.indexOf('Text') < 0) k = 'soon';
  var map = {text:'grp_text', see:'grp_cam', draw:'grp_draw', sim:'grp_sim', fight:'grp_fight'};
  Object.keys(map).forEach(function(key){ var el = document.getElementById(map[key]); if(el) el.style.display = (key === k) ? 'flex' : 'none'; });
  if(k === window._modeKey) return;
  var prev = window._modeKey; window._modeKey = k;
  if(window.RB) window.RB.run({world: {draw:'draw', sim:'life', fight:'fight'}[k] || 'home'});
  if(prev === 'fight' && window.fightCamStop) window.fightCamStop();
};

window.stopCam = function(){
  document.querySelectorAll('video').forEach(function(v){
    var s = v.srcObject;
    if(s && s.getTracks){ s.getTracks().forEach(function(t){ t.stop(); }); v.srcObject = null; }
  });
  if(window._camStream){ try{ window._camStream.getTracks().forEach(function(t){ t.stop(); }); }catch(e){} window._camStream = null; }
};

// ---------- paint canvas ----------
window._strokes = [];
window.initPaint = function(){
  var c = document.getElementById('paint'); if(!c || c._ok) return; c._ok = 1;
  var x = c.getContext('2d'); x.lineWidth = 5; x.lineCap = 'round'; x.lineJoin = 'round'; x.strokeStyle = '#22d3ee';
  var cur = null;
  function pos(e){ var r = c.getBoundingClientRect(); return [(e.clientX-r.left)*c.width/r.width, (e.clientY-r.top)*c.height/r.height]; }
  c.onpointerdown = function(e){ c.setPointerCapture(e.pointerId); cur = [pos(e)]; x.beginPath(); x.moveTo(cur[0][0], cur[0][1]); };
  c.onpointermove = function(e){ if(!cur) return; var p = pos(e); cur.push(p); x.lineTo(p[0], p[1]); x.stroke(); };
  c.onpointerup = c.onpointercancel = function(){
    if(cur){ if(cur.length === 1) cur.push(cur[0]); window._strokes.push(cur); } cur = null; };
};
window.getStrokes = function(){ return JSON.stringify(window._strokes || []); };
window.clearPaint = function(){
  var c = document.getElementById('paint'); if(c) c.getContext('2d').clearRect(0, 0, c.width, c.height);
  window._strokes = [];
};


// ===================== 3D HUMANOID CORE (pure maths + mesh building) =====================
window.RBC = (function(){
var NB = 49;                       // bones: 0-14 body, 15-31 left hand, 32-48 right hand
var D = {hw:0.095, torso:0.40, sw:0.175, shDrop:0.05, neck:0.075, head:0.10,
         ua:0.24, fa:0.23, palm:0.085, th:0.30, sh:0.30, ft:0.20, sole:0.065};
var FL = [[0.030,0.024,0.020],[0.036,0.022,0.018],[0.040,0.025,0.019],[0.036,0.023,0.018],[0.029,0.018,0.015]];
var FR = [0.0105,0.0095,0.0098,0.0092,0.0085];
var MCPX = [0, 0.028, 0.010, -0.010, -0.028];
var SPLAY = [0, 0.12, 0.03, -0.06, -0.15];
var HB = {L:15, R:32};

// ---- vector helpers ----
function add(a,b){return [a[0]+b[0],a[1]+b[1],a[2]+b[2]];}
function sub(a,b){return [a[0]-b[0],a[1]-b[1],a[2]-b[2]];}
function mul(a,k){return [a[0]*k,a[1]*k,a[2]*k];}
function dot(a,b){return a[0]*b[0]+a[1]*b[1]+a[2]*b[2];}
function cross(a,b){return [a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]];}
function len(a){return Math.sqrt(a[0]*a[0]+a[1]*a[1]+a[2]*a[2]);}
function nrm(a){var n=len(a); return n>1e-9?[a[0]/n,a[1]/n,a[2]/n]:[0,0,0];}
function getb(P,i){return [P[i*3],P[i*3+1],P[i*3+2]];}
function setb(P,i,v){var n=nrm(v); P[i*3]=n[0]; P[i*3+1]=n[1]; P[i*3+2]=n[2];}
function orth(v,ref){return nrm(sub(v, mul(ref, dot(v,ref))));}

// ---- hand presets (finger directions as mixes of fwd / palm-normal / side) ----
var FING = {
  open:   {f:[[1,0],[1,0],[1,0]],            t:[[.6,.7,0],[.7,.7,0],[.8,.6,0]]},
  relaxed:{f:[[.95,.3],[.75,.65],[.55,.8]],  t:[[.6,.5,.2],[.8,.3,.3],[.8,.2,.4]]},
  fist:   {f:[[.2,1],[-.6,.8],[-.9,-.4]],    t:[[.4,.5,.4],[.2,.1,.9],[-.2,-.3,.9]]},
  grip:   {f:[[.5,.85],[.2,.95],[-.2,.9]],   t:[[.5,.5,.5],[.4,.2,.8],[.3,0,.9]]}
};
function handN(fwd, side, left){ var n = cross(side, fwd); return left ? mul(n,-1) : n; }
function handSet(P, base, fwd, side, left, preset){
  fwd = nrm(fwd);
  side = orth(side, fwd);
  if(len(side) < 0.5){ side = orth([0,0,1], fwd); if(len(side)<0.5) side = orth([1,0,0], fwd); }
  var n = handN(fwd, side, left), pr = FING[preset] || FING.relaxed;
  setb(P, base, fwd); setb(P, base+1, side);
  for(var s=0; s<3; s++){
    for(var f=1; f<5; f++){
      var c = pr.f[s];
      setb(P, base+2+f*3+s, add(add(mul(fwd,c[0]), mul(n,c[1])), mul(side,SPLAY[f])));
    }
    var t = pr.t[s];
    setb(P, base+2+s, add(add(mul(fwd,t[0]), mul(side,t[1])), mul(n,t[2])));
  }
}

var REST = (function(){
  var P = new Float32Array(NB*3);
  setb(P,0,[0,1,0]); setb(P,1,[1,0,0]); setb(P,2,[1,0,0]); setb(P,3,[0,1,0]); setb(P,4,[0,0,1]);
  setb(P,5,[.10,-1,0]); setb(P,6,[.06,-1,.08]); setb(P,7,[-.10,-1,0]); setb(P,8,[-.06,-1,.08]);
  setb(P,9,[0,-1,0]); setb(P,10,[0,-1,0]); setb(P,11,[0,0,1]);
  setb(P,12,[0,-1,0]); setb(P,13,[0,-1,0]); setb(P,14,[0,0,1]);
  handSet(P, HB.L, getb(P,6), [0,0,1], true, 'relaxed');
  handSet(P, HB.R, getb(P,8), [0,0,1], false, 'relaxed');
  return P;
})();
function restPose(){ return new Float32Array(REST); }

// ---- pose maths ----
function nlerpPose(A, B, t, out){
  for(var i=0;i<NB;i++){
    var x=A[i*3]*(1-t)+B[i*3]*t, y=A[i*3+1]*(1-t)+B[i*3+1]*t, z=A[i*3+2]*(1-t)+B[i*3+2]*t;
    var n=Math.sqrt(x*x+y*y+z*z);
    if(n<1e-6){ x=B[i*3]; y=B[i*3+1]; z=B[i*3+2]; n=Math.sqrt(x*x+y*y+z*z)||1; }
    out[i*3]=x/n; out[i*3+1]=y/n; out[i*3+2]=z/n;
  }
  return out;
}
// frames: Array of Float32Array(147); u in [0,1]
function sampleMove(frames, u, out){
  var n = frames.length; if(n===1){ out.set(frames[0]); return out; }
  var f = Math.max(0, Math.min(1, u)) * (n-1), i = Math.floor(f); if(i>=n-1) i=n-2;
  return nlerpPose(frames[i], frames[i+1], f-i, out);
}
// turn the JSON move (array of arrays) into Float32Arrays; missing bones fall back to REST
function decodeMove(mv){
  var fr = [];
  for(var k=0;k<mv.f.length;k++){
    var a = new Float32Array(NB*3), src = mv.f[k];
    var sc = mv.s ? 1/mv.s : 1;
    for(var b=0;b<NB;b++){
      var x=src[b*3]*sc, y=src[b*3+1]*sc, z=src[b*3+2]*sc;
      if(x*x+y*y+z*z < 0.25){ x=REST[b*3]; y=REST[b*3+1]; z=REST[b*3+2]; }
      a[b*3]=x; a[b*3+1]=y; a[b*3+2]=z;
    }
    fr.push(a);
  }
  return {frames:fr, dur:mv.dur || 2, name:mv.name || ''};
}

// ---- forward kinematics (pelvis at the origin) ----
function fk(P){
  var J = {}, up=getb(P,0), hips=getb(P,1), shd=getb(P,2);
  J.pel = [0,0,0];
  J.hipL = mul(hips, D.hw); J.hipR = mul(hips, -D.hw);
  J.neckB = mul(up, D.torso);
  var sb = sub(J.neckB, mul(up, D.shDrop));
  J.shL = add(sb, mul(shd, D.sw)); J.shR = add(sb, mul(shd, -D.sw));
  J.elL = add(J.shL, mul(getb(P,5), D.ua)); J.wrL = add(J.elL, mul(getb(P,6), D.fa));
  J.elR = add(J.shR, mul(getb(P,7), D.ua)); J.wrR = add(J.elR, mul(getb(P,8), D.fa));
  J.head = add(J.neckB, mul(getb(P,3), D.neck + D.head));
  J.knL = add(J.hipL, mul(getb(P,9), D.th));  J.anL = add(J.knL, mul(getb(P,10), D.sh)); J.toL = add(J.anL, mul(getb(P,11), D.ft));
  J.knR = add(J.hipR, mul(getb(P,12), D.th)); J.anR = add(J.knR, mul(getb(P,13), D.sh)); J.toR = add(J.anR, mul(getb(P,14), D.ft));
  return J;
}
function groundY(J){ return Math.min(J.anL[1], J.anR[1], J.toL[1], J.toR[1]) - D.sole; }

// two-bone IK: returns [upperDir, foreDir] so that the wrist reaches target T (pelvis frame)
function ikArm(S, T, l1, l2, hint){
  var d = sub(T, S), dl = len(d);
  var mx = (l1 + l2) * 0.995, mn = Math.abs(l1 - l2) + 0.01;
  var u = nrm(d); if(dl < 1e-6) u = [0,0,1];
  dl = Math.max(mn, Math.min(mx, dl));
  var a = (l1*l1 - l2*l2 + dl*dl) / (2*dl), h = Math.sqrt(Math.max(l1*l1 - a*a, 0));
  var pole = orth(hint, u); if(len(pole) < 0.3) pole = orth([0,-1,0], u);
  var elbow = add(add(S, mul(u, a)), mul(pole, h));
  var wrist = add(S, mul(u, dl));
  return {ua: nrm(sub(elbow, S)), fa: nrm(sub(wrist, elbow)), wrist: wrist, elbow: elbow};
}

// ---- procedural mood / idle poses (written into P, which starts as REST) ----
function S(P,i,x,y,z){ setb(P,i,[x,y,z]); }
function moodPose(mood, t, talking, listening){
  var P = restPose(), s = Math.sin, c = Math.cos;
  var sway = 0.015*s(t*1.3), breath = 0.02*s(t*2.0);
  S(P,0, sway, 1, 0.01+breath); S(P,3, -sway, 1, 0.02);
  var arms = null;
  if(mood==='happy'){
    var w = s(t*7);
    S(P,7,-0.75,0.55,0.15); S(P,8,-0.25+0.55*w,0.85,0.25);
    handSet(P,HB.R,getb(P,8),[0,0,1],false,'open');
    var a = 0.14*(1+s(t*4.5));
    S(P,9,0,-c(a),s(a)); S(P,10,0,-c(a),-s(a)); S(P,12,0,-c(a),s(a)); S(P,13,0,-c(a),-s(a));
  } else if(mood==='learning'){
    var w2 = s(t*6);
    S(P,5,.65,.7,.1); S(P,6,.25+.3*w2,.9,.2); S(P,7,-.65,.7,.1); S(P,8,-.25-.3*w2,.9,.2);
    handSet(P,HB.L,getb(P,6),[0,0,1],true,'open'); handSet(P,HB.R,getb(P,8),[0,0,1],false,'open');
    var a2 = 0.16*(1+s(t*5));
    S(P,9,0,-c(a2),s(a2)); S(P,10,0,-c(a2),-s(a2)); S(P,12,0,-c(a2),s(a2)); S(P,13,0,-c(a2),-s(a2));
  } else if(mood==='thinking'){
    S(P,7,-.15,-.75,.45); S(P,8,.2,.5,.7);
    handSet(P,HB.R,getb(P,8),[0,1,0],false,'relaxed');
    S(P,3,-.12,1,0); S(P,4,.35,.1,1);
  } else if(mood==='confused'){
    S(P,5,.5,-.7,.3); S(P,6,.15,.3,.9); S(P,7,-.5,-.7,.3); S(P,8,-.15,.3,.9);
    handSet(P,HB.L,getb(P,6),[0,1,0],true,'open'); handSet(P,HB.R,getb(P,8),[0,1,0],false,'open');
    S(P,3,.18*s(t*2),1,0);
  } else if(mood==='surprised'){
    S(P,5,.8,.5,.1); S(P,6,.6,.8,.2); S(P,7,-.8,.5,.1); S(P,8,-.6,.8,.2);
    handSet(P,HB.L,getb(P,6),[0,0,1],true,'open'); handSet(P,HB.R,getb(P,8),[0,0,1],false,'open');
    S(P,0,0,1,-0.1);
  } else if(mood==='sad'){
    S(P,0,0,1,0.12); S(P,3,0,1,0.3); S(P,4,0,-.45,1);
  } else if(mood==='drawing'){
    S(P,7,-.1,-.5,.8); S(P,8,.05,0,1);
    handSet(P,HB.R,getb(P,8),[0,1,0],false,'grip');
  }
  if(talking){
    var g = s(t*5.5), g2 = s(t*3.1+1);
    S(P,7,-.25,-.55,.55); S(P,8,.15+.2*g,.2+.45*g2,.85);
    handSet(P,HB.R,getb(P,8),[0,1,0],false,'open');
    S(P,3,0,1,0.05*s(t*4)); S(P,4,0.06*s(t*2.2),0.08*s(t*4),1);
  }
  if(listening){ S(P,3,.25,1,0); S(P,4,-.3,0,1); }
  return P;
}

// ---- mesh builder (needs THREE) ----
function build(THREE, faceTex){
  var R = {}, root = new THREE.Group(); R.root = root;
  var UP = new THREE.Vector3(0,1,0), v1 = new THREE.Vector3(), m4 = new THREE.Matrix4(), vx = new THREE.Vector3(), vy = new THREE.Vector3(), vz = new THREE.Vector3();
  var mat = {
    shell:  new THREE.MeshStandardMaterial({color:0xeef2f8, metalness:0.25, roughness:0.38}),
    shell2: new THREE.MeshStandardMaterial({color:0xc7d1df, metalness:0.35, roughness:0.4}),
    dark:   new THREE.MeshStandardMaterial({color:0x1a2030, metalness:0.7, roughness:0.35}),
    joint:  new THREE.MeshStandardMaterial({color:0x2c3650, metalness:0.85, roughness:0.28}),
    skin:   new THREE.MeshStandardMaterial({color:0x2a3040, metalness:0.5, roughness:0.45}),
    led:    new THREE.MeshStandardMaterial({color:0x22d3ee, emissive:0x22d3ee, emissiveIntensity:1.4}),
    visor:  new THREE.MeshBasicMaterial({color:0xffffff, map:faceTex || null})
  };
  R.mat = mat;
  var gCyl = new THREE.CylinderGeometry(1,1,1,16), gTap = new THREE.CylinderGeometry(0.82,1,1,16),
      gSph = new THREE.SphereGeometry(1,16,12), gBox = new THREE.BoxGeometry(1,1,1), gTor = new THREE.TorusGeometry(1,0.12,8,24);
  function mk(g, m){ var o = new THREE.Mesh(g, m); o.castShadow = true; o.frustumCulled = false; root.add(o); return o; }
  function link(m, a, b, r){
    var dx=b[0]-a[0], dy=b[1]-a[1], dz=b[2]-a[2], L=Math.sqrt(dx*dx+dy*dy+dz*dz)||1e-6;
    m.position.set((a[0]+b[0])/2,(a[1]+b[1])/2,(a[2]+b[2])/2);
    v1.set(dx/L,dy/L,dz/L); m.quaternion.setFromUnitVectors(UP, v1); m.scale.set(r,L,r);
  }
  function hinge(m, c, axis, r, l){
    m.position.set(c[0],c[1],c[2]); v1.set(axis[0],axis[1],axis[2]); m.quaternion.setFromUnitVectors(UP, v1); m.scale.set(r,l,r);
  }
  function ball(m, c, r){ m.position.set(c[0],c[1],c[2]); m.scale.set(r,r,r); m.quaternion.identity(); }
  function basis(m, c, X, Y, Z, sx, sy, sz){
    vx.set(X[0],X[1],X[2]); vy.set(Y[0],Y[1],Y[2]); vz.set(Z[0],Z[1],Z[2]);
    m4.makeBasis(vx,vy,vz); m.quaternion.setFromRotationMatrix(m4); m.position.set(c[0],c[1],c[2]); m.scale.set(sx,sy,sz);
  }
  // torso
  var chest = mk(gBox, mat.shell), abd = mk(gBox, mat.shell2), pelv = mk(gBox, mat.shell), pack = mk(gBox, mat.dark),
      waist = mk(gCyl, mat.joint), ledBar = mk(gBox, mat.led), core = mk(gSph, mat.led), neckM = mk(gTap, mat.joint),
      collar = mk(gBox, mat.dark);
  // head
  var head = mk(gSph, mat.shell), visor = mk(gBox, mat.visor), earL = mk(gCyl, mat.joint), earR = mk(gCyl, mat.joint),
      lidar = mk(gCyl, mat.dark), ring = mk(gTor, mat.led), visorFrame = mk(gBox, mat.dark);
  R.head = head; R.visor = visor;
  // arms
  function makeArm(){ return {sh: mk(gSph, mat.joint), shCap: mk(gSph, mat.shell), ua: mk(gTap, mat.shell), el: mk(gCyl, mat.joint),
                              fa: mk(gTap, mat.shell2), wr: mk(gSph, mat.joint), band: mk(gCyl, mat.led)}; }
  function makeHand(){
    var h = {palm: mk(gBox, mat.dark), cuff: mk(gCyl, mat.shell), seg: [], jnt: []};
    for(var i=0;i<15;i++){ h.seg.push(mk(gTap, mat.skin)); h.jnt.push(mk(gSph, mat.joint)); }
    return h;
  }
  var armL = makeArm(), armR = makeArm(), handL = makeHand(), handR = makeHand();
  // legs
  function makeLeg(){ return {hip: mk(gCyl, mat.joint), hipCap: mk(gCyl, mat.dark), th: mk(gTap, mat.shell), kn: mk(gCyl, mat.joint),
                              knCap: mk(gCyl, mat.dark), sh: mk(gTap, mat.shell2), an: mk(gSph, mat.joint), foot: mk(gBox, mat.dark),
                              sole: mk(gBox, mat.shell2)}; }
  var legL = makeLeg(), legR = makeLeg();
  R.parts = {chest:chest, abd:abd, pelv:pelv, head:head, armL:armL, armR:armR, handL:handL, handR:handR, legL:legL, legR:legR};

  function applyArm(a, shp, elp, wrp, ua, fa, shdir, left){
    ball(a.sh, shp, 0.046); ball(a.shCap, add(shp, mul(shdir, left?0.03:-0.03)), 0.04);
    link(a.ua, shp, elp, 0.033);
    var ax = cross(ua, fa); if(len(ax) < 0.15) ax = shdir; ax = nrm(ax);
    hinge(a.el, elp, ax, 0.036, 0.075);
    link(a.fa, elp, wrp, 0.029);
    ball(a.wr, wrp, 0.024);
    hinge(a.band, add(elp, mul(fa, 0.12)), fa, 0.031, 0.012);
  }
  function applyHand(h, P, base, W, left){
    var fwd = nrm(getb(P, base)), sd = orth(getb(P, base+1), fwd);
    if(len(sd) < 0.5) sd = orth([0,0,1], fwd);
    var Z = cross(sd, fwd);
    basis(h.palm, add(W, mul(fwd, D.palm*0.5)), sd, fwd, Z, 0.072, D.palm, 0.026);
    hinge(h.cuff, add(W, mul(fwd, 0.006)), fwd, 0.026, 0.02);
    var K0 = add(W, mul(fwd, D.palm));
    for(var f=0; f<5; f++){
      var p = (f===0) ? add(add(W, mul(fwd, 0.03)), mul(sd, 0.034)) : add(K0, mul(sd, MCPX[f]));
      for(var s=0; s<3; s++){
        var d = getb(P, base+2+f*3+s), q = add(p, mul(d, FL[f][s])), idx = f*3+s;
        link(h.seg[idx], p, q, FR[f]*(1-0.08*s));
        ball(h.jnt[idx], p, FR[f]*1.18);
        p = q;
      }
    }
  }
  function applyLeg(l, hip, kn, an, to, th, sh, ft, hipsDir, left){
    hinge(l.hip, hip, hipsDir, 0.052, 0.085);
    hinge(l.hipCap, add(hip, mul(hipsDir, left?0.05:-0.05)), hipsDir, 0.045, 0.02);
    link(l.th, hip, kn, 0.047);
    var ax = cross(th, sh); if(len(ax) < 0.12) ax = hipsDir; ax = nrm(ax);
    hinge(l.kn, kn, ax, 0.043, 0.085);
    hinge(l.knCap, add(kn, mul(ax, 0.05)), ax, 0.036, 0.016);
    link(l.sh, kn, an, 0.037);
    ball(l.an, an, 0.032);
    var upv = orth([0,1,0], ft); if(len(upv) < 0.5) upv = orth([0,0,1], ft);
    var X = cross(upv, ft);
    basis(l.foot, add(add(an, mul(ft, 0.085)), mul(upv, -0.032)), X, upv, ft, 0.095, 0.046, 0.23);
    basis(l.sole, add(add(an, mul(ft, 0.085)), mul(upv, -0.058)), X, upv, ft, 0.1, 0.008, 0.235);
  }

  R._ref = null; R._root = [0,0];
  R.apply = function(P, opt){
    opt = opt || {};
    var J = fk(P), gy = groundY(J);
    var up = getb(P,0), hips = getb(P,1), shd = getb(P,2), neck = getb(P,3), face = getb(P,4);
    // foot lock: planted feet stay where they were, so walking / sitting move the pelvis
    var tx = 0, tz = 0;
    if(opt.lock){
      if(!R._ref) R._ref = {L:[J.anL[0],J.anL[2]], R:[J.anR[0],J.anR[2]]};
      var yl = Math.min(J.anL[1], J.anR[1]);
      var wl = Math.exp(-(J.anL[1]-yl)/0.03), wr = Math.exp(-(J.anR[1]-yl)/0.03), ws = wl+wr;
      tx = (wl*(R._ref.L[0]-J.anL[0]) + wr*(R._ref.R[0]-J.anR[0]))/ws;
      tz = (wl*(R._ref.L[1]-J.anL[2]) + wr*(R._ref.R[1]-J.anR[2]))/ws;
      tx = Math.max(-1.6, Math.min(1.6, tx)); tz = Math.max(-1.6, Math.min(1.6, tz));
      R._root[0] += (tx - R._root[0])*0.4; R._root[1] += (tz - R._root[1])*0.4;
    } else {
      R._ref = null;
      R._root[0] *= 0.9; R._root[1] *= 0.9;
    }
    root.position.set(R._root[0], -gy, R._root[1]);
    // torso blocks
    var X = orth(shd, up), Z = cross(X, up);
    basis(chest, mul(up, 0.30), X, up, Z, 0.30, 0.25, 0.17);
    basis(abd, mul(up, 0.145), X, up, Z, 0.21, 0.17, 0.135);
    var HX = orth(hips, up), HZ = cross(HX, up);
    basis(pelv, [0,0,0], HX, up, HZ, 0.25, 0.10, 0.155);
    basis(pack, add(mul(up, 0.30), mul(Z, -0.105)), X, up, Z, 0.19, 0.19, 0.05);
    hinge(waist, mul(up, 0.215), X, 0.052, 0.12);
    basis(ledBar, add(add(mul(up, 0.34), mul(Z, 0.088)), mul(X, 0)), X, up, Z, 0.12, 0.014, 0.006);
    ball(core, add(add(mul(up, 0.28), mul(Z, 0.09)), [0,0,0]), 0.024);
    basis(collar, mul(up, 0.405), X, up, Z, 0.17, 0.03, 0.10);
    link(neckM, mul(up, 0.40), add(J.neckB, mul(neck, D.neck)), 0.026);
    // head
    var HY = neck, HZ2 = orth(face, HY); if(len(HZ2) < 0.5) HZ2 = orth([0,0,1], HY);
    var HXh = cross(HY, HZ2), hc = add(J.neckB, mul(neck, D.neck + 0.092));
    basis(head, hc, HXh, HY, HZ2, 0.088, 0.10, 0.096);
    basis(visorFrame, add(hc, mul(HZ2, 0.078)), HXh, HY, HZ2, 0.156, 0.098, 0.050);
    basis(visor, add(hc, mul(HZ2, 0.1035)), HXh, HY, HZ2, 0.142, 0.084, 0.002);
    hinge(earL, add(hc, mul(HXh, 0.092)), HXh, 0.032, 0.03);
    hinge(earR, add(hc, mul(HXh, -0.092)), HXh, 0.032, 0.03);
    hinge(lidar, add(hc, mul(HY, 0.102)), HY, 0.03, 0.028);
    basis(ring, add(hc, mul(HY, 0.075)), HXh, HZ2, mul(HY,-1), 0.095, 0.095, 0.095);
    // arms
    applyArm(armL, J.shL, J.elL, J.wrL, getb(P,5), getb(P,6), shd, true);
    applyArm(armR, J.shR, J.elR, J.wrR, getb(P,7), getb(P,8), shd, false);
    applyHand(handL, P, HB.L, J.wrL, true);
    applyHand(handR, P, HB.R, J.wrR, false);
    // legs
    applyLeg(legL, J.hipL, J.knL, J.anL, J.toL, getb(P,9), getb(P,10), getb(P,11), hips, true);
    applyLeg(legR, J.hipR, J.knR, J.anR, J.toR, getb(P,12), getb(P,13), getb(P,14), hips, false);
    R.J = J;
    return J;
  };
  R.setColor = function(hex){ mat.led.color.setHex(hex); mat.led.emissive.setHex(hex); };
  return R;
}

return {NB:NB, D:D, REST:REST, HB:HB, FING:FING, restPose:restPose, nlerpPose:nlerpPose, sampleMove:sampleMove,
        decodeMove:decodeMove, fk:fk, groundY:groundY, ikArm:ikArm, moodPose:moodPose, handSet:handSet, build:build,
        v:{add:add,sub:sub,mul:mul,dot:dot,cross:cross,len:len,nrm:nrm,getb:getb,setb:setb,orth:orth}};
})();

// ===================== 3D STAGE (Three.js) =====================
window.RB = {ready:false, queue:[], mood:'idle', world:'home'};
var MOODCOL = {determined:'#fbbf24', idle:'#22d3ee', thinking:'#a78bfa', happy:'#34d399', learning:'#fbbf24', confused:'#fb923c',
               surprised:'#f472b6', sad:'#60a5fa', drawing:'#f59e0b', move:'#fbbf24'};
RB.setMood = function(m){ RB.mood = MOODCOL[m] ? (m==='move' ? 'idle' : m) : 'idle'; RB.col = MOODCOL[m] || MOODCOL.idle; };
RB.col = MOODCOL.idle;
RB.run = function(cmd){
  if(!cmd) return;
  if(!RB.ready){ RB.queue.push(cmd); if(RB.queue.length > 8) RB.queue.shift(); return; }
  RB._run(cmd);
};

window.initStage = async function(){
  var host = document.getElementById('stage'); if(!host) return;
  if(host._st === 'busy' || host._st === 'ok') return;
  if(host._st === 'fail' && Date.now() - host._failT < 8000) return;
  host._st = 'busy';
  try{
    host.textContent = 'loading the 3D world…';
    var THREE = await import('https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js'); window._THREE = THREE;
    host.textContent = '';
    window.setupStage(THREE, host);
    host._st = 'ok';
  }catch(e){ host._st = 'fail'; host._failT = Date.now(); host.textContent = '3D failed to start: ' + e.message; }
};

window.drawFace = function(ctx, W, H, mood, t, talking, col){
  var bg = ctx.createRadialGradient(W/2, H/2, 10, W/2, H/2, W*0.7); bg.addColorStop(0, '#0b1d33'); bg.addColorStop(1, '#030a14');
  ctx.shadowBlur = 0; ctx.fillStyle = bg; ctx.fillRect(0, 0, W, H);
  ctx.shadowColor = col; ctx.shadowBlur = 12; ctx.strokeStyle = col; ctx.fillStyle = col; ctx.lineCap = 'round'; ctx.lineJoin = 'round';
  var ex = [W*0.29, W*0.71], ey = H*0.42, rx = W*0.085, ry = H*0.25, my = H*0.80;
  var blink = ((t % 4.0) > 3.88) ? 0.1 : 1, lx = 0, ly = 0;
  if(mood === 'idle'){ lx = 5*Math.sin(t*0.7); ly = 2*Math.sin(t*0.5); }
  else if(mood === 'thinking'){ lx = 12; ly = -9; } else if(mood === 'confused'){ lx = -9; ly = -4; } else if(mood === 'sad'){ ly = 8; }
  function eye(x, k, sc){
    ctx.fillStyle = col; ctx.beginPath(); ctx.ellipse(x + lx, ey + ly, rx*sc, ry*k*blink*sc, 0, 0, Math.PI*2); ctx.fill();
    ctx.shadowBlur = 0; ctx.fillStyle = '#ffffffcc'; ctx.beginPath(); ctx.arc(x + lx - rx*0.28*sc, ey + ly - ry*0.3*k*sc, rx*0.22*sc, 0, Math.PI*2); ctx.fill();
    ctx.shadowBlur = 12; ctx.fillStyle = col;
  }
  function arcEye(x){ ctx.lineWidth = 11; ctx.beginPath(); ctx.arc(x, ey + 16, rx, Math.PI*1.12, Math.PI*1.88); ctx.stroke(); }
  function star(x, y, r){ ctx.beginPath(); for(var i = 0; i < 8; i++){ var a = i*Math.PI/4, q = (i % 2 === 0) ? r : r*0.35; ctx.lineTo(x + Math.cos(a)*q, y + Math.sin(a)*q); } ctx.closePath(); ctx.fill(); }
  function brow(x1, y1, x2, y2){ ctx.lineWidth = 7; ctx.beginPath(); ctx.moveTo(x1, y1); ctx.lineTo(x2, y2); ctx.stroke(); }
  function cheeks(){ ctx.shadowBlur = 0; ctx.fillStyle = '#ff7aa866';
    ctx.beginPath(); ctx.ellipse(W*0.12, H*0.68, W*0.06, H*0.07, 0, 0, Math.PI*2); ctx.fill();
    ctx.beginPath(); ctx.ellipse(W*0.88, H*0.68, W*0.06, H*0.07, 0, 0, Math.PI*2); ctx.fill(); ctx.shadowBlur = 12; ctx.fillStyle = col; }
  if(mood === 'happy'){ arcEye(ex[0]); arcEye(ex[1]); cheeks(); }
  else if(mood === 'learning'){
    var k = 1 + 0.12*Math.sin(t*8); star(ex[0], ey, rx*1.35*k); star(ex[1], ey, rx*1.35*(2 - k)); cheeks();
    for(var i = 0; i < 3; i++){ var a = t*2 + i*2.1; star(W*(0.5 + 0.44*Math.cos(a)), H*(0.5 + 0.4*Math.sin(a*1.3)), 5); }
  } else if(mood === 'surprised'){
    ctx.lineWidth = 7;
    [0, 1].forEach(function(i){ ctx.beginPath(); ctx.arc(ex[i], ey, rx*1.25, 0, Math.PI*2); ctx.stroke(); ctx.beginPath(); ctx.arc(ex[i], ey, rx*0.4, 0, Math.PI*2); ctx.fill(); });
    brow(ex[0]-24, ey-ry*1.4, ex[0]+22, ey-ry*1.55); brow(ex[1]-22, ey-ry*1.55, ex[1]+24, ey-ry*1.4);
    ctx.font = 'bold 40px Arial'; ctx.fillText('!', W*0.9, H*0.3);
  } else if(mood === 'confused'){
    eye(ex[0], 1, 1.05); eye(ex[1], 0.55, 0.9);
    brow(ex[0]-24, ey-ry*1.3, ex[0]+22, ey-ry*1.0); brow(ex[1]-22, ey-ry*1.0, ex[1]+24, ey-ry*1.45);
    ctx.font = 'bold 44px Arial'; ctx.fillText('?', W*0.86, H*0.32);
  } else if(mood === 'sad'){
    eye(ex[0], 0.75, 1); eye(ex[1], 0.75, 1);
    brow(ex[0]-26, ey-ry*1.1, ex[0]+22, ey-ry*1.5); brow(ex[1]+26, ey-ry*1.1, ex[1]-22, ey-ry*1.5);
    var ty = ey + ry*0.8 + ((t*40) % 40); ctx.shadowBlur = 0; ctx.fillStyle = '#6ec6ff'; ctx.beginPath(); ctx.ellipse(ex[0] + rx*0.9, ty, 5, 8, 0, 0, Math.PI*2); ctx.fill(); ctx.fillStyle = col; ctx.shadowBlur = 12;
  } else if(mood === 'thinking'){
    eye(ex[0], 0.95, 1); eye(ex[1], 0.95, 1);
    brow(ex[0]-24, ey-ry*1.2, ex[0]+22, ey-ry*1.2); brow(ex[1]-22, ey-ry*1.55, ex[1]+24, ey-ry*1.35);
    for(var j = 0; j < 3; j++){ ctx.globalAlpha = 0.3 + 0.7*((Math.sin(t*5 - j*0.9) + 1)/2); ctx.beginPath(); ctx.arc(W*0.8 + j*14, H*0.16, 4 + j, 0, Math.PI*2); ctx.fill(); }
    ctx.globalAlpha = 1;
  } else if(mood === 'drawing'){
    eye(ex[0], 0.55, 1); eye(ex[1], 0.55, 1); brow(ex[0]-24, ey-ry*0.95, ex[0]+22, ey-ry*0.8); brow(ex[1]-22, ey-ry*0.8, ex[1]+24, ey-ry*0.95);
  } else if(mood === 'determined'){
    eye(ex[0], 0.6, 1); eye(ex[1], 0.6, 1); brow(ex[0]-26, ey-ry*1.4, ex[0]+24, ey-ry*0.85); brow(ex[1]+26, ey-ry*1.4, ex[1]-24, ey-ry*0.85);
  } else { eye(ex[0], 1, 1); eye(ex[1], 1, 1); }
  ctx.lineWidth = 7; ctx.fillStyle = col;
  if(talking){
    ctx.beginPath(); ctx.ellipse(W/2, my, W*0.08, H*0.035 + H*0.1*Math.abs(Math.sin(t*14)), 0, 0, Math.PI*2); ctx.fill();
  } else if(mood === 'happy' || mood === 'learning'){
    ctx.beginPath(); ctx.arc(W/2, my - 20, W*0.13, Math.PI*0.12, Math.PI*0.88); ctx.stroke();
  } else if(mood === 'surprised'){
    ctx.beginPath(); ctx.ellipse(W/2, my, W*0.04, H*0.09, 0, 0, Math.PI*2); ctx.fill();
  } else if(mood === 'sad'){
    ctx.beginPath(); ctx.arc(W/2, my + 26, W*0.09, Math.PI*1.2, Math.PI*1.8); ctx.stroke();
  } else if(mood === 'confused'){
    ctx.beginPath(); ctx.moveTo(W*0.38, my); ctx.quadraticCurveTo(W*0.44, my-14, W*0.5, my); ctx.quadraticCurveTo(W*0.56, my+14, W*0.62, my); ctx.stroke();
  } else if(mood === 'determined'){
    ctx.beginPath(); ctx.moveTo(W*0.38, my - 4); ctx.lineTo(W*0.62, my - 4); ctx.stroke();
  } else if(mood === 'thinking' || mood === 'drawing'){
    ctx.beginPath(); ctx.moveTo(W*0.42, my); ctx.lineTo(W*0.58, my - 4); ctx.stroke();
  } else {
    ctx.beginPath(); ctx.arc(W/2, my - 14, W*0.08, Math.PI*0.2, Math.PI*0.8); ctx.stroke();
  }
  ctx.shadowBlur = 0;
};

window.setupStage = function(THREE, host){
  var C = window.RBC, V = C.v;
  var W0 = host.clientWidth || 480, H0 = host.clientHeight || 400;
  var renderer = new THREE.WebGLRenderer({antialias:true});
  renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 1.5));
  renderer.setSize(W0, H0);
  renderer.shadowMap.enabled = true; renderer.shadowMap.type = THREE.PCFSoftShadowMap;
  host.appendChild(renderer.domElement);
  var scene = new THREE.Scene(); scene.background = new THREE.Color(0x050a1c); scene.fog = new THREE.Fog(0x050a1c, 7, 18);
  var camera = new THREE.PerspectiveCamera(38, W0/H0, 0.05, 60);

  scene.add(new THREE.HemisphereLight(0xcfe8ff, 0x1a2040, 0.85));
  var key = new THREE.DirectionalLight(0xffffff, 1.5); key.position.set(2.5, 4.5, 3.2); key.castShadow = true;
  key.shadow.mapSize.set(512, 512); var sc = key.shadow.camera; sc.left = -3; sc.right = 3; sc.top = 3; sc.bottom = -3; sc.near = 0.5; sc.far = 14;
  scene.add(key);
  var rim = new THREE.DirectionalLight(0x6aa8ff, 0.7); rim.position.set(-3, 2.5, -3); scene.add(rim);

  // ---- visor (expression screen) ----
  var fc = document.createElement('canvas'); fc.width = 256; fc.height = 128;
  var fctx = fc.getContext('2d'), ftex = new THREE.CanvasTexture(fc);
  var R = C.build(THREE, ftex); scene.add(R.root);

  // ---- helpers for worlds ----
  function stdMat(c, m, r){ return new THREE.MeshStandardMaterial({color:c, metalness:m === undefined ? 0.2 : m, roughness:r === undefined ? 0.7 : r}); }
  function box(g, w, h, d, col, x, y, z, m, r){ var o = new THREE.Mesh(new THREE.BoxGeometry(w, h, d), stdMat(col, m, r)); o.position.set(x, y, z); o.castShadow = true; o.receiveShadow = true; g.add(o); return o; }
  function cyl(g, rt, rb, h, col, x, y, z, m, r){ var o = new THREE.Mesh(new THREE.CylinderGeometry(rt, rb, h, 20), stdMat(col, m, r)); o.position.set(x, y, z); o.castShadow = true; o.receiveShadow = true; g.add(o); return o; }
  function rod(g, a, b, rad, col){
    var o = new THREE.Mesh(new THREE.CylinderGeometry(rad, rad, 1, 8), stdMat(col, 0.1, 0.6));
    var d = new THREE.Vector3(b[0]-a[0], b[1]-a[1], b[2]-a[2]), L = d.length();
    o.position.set((a[0]+b[0])/2, (a[1]+b[1])/2, (a[2]+b[2])/2); o.quaternion.setFromUnitVectors(new THREE.Vector3(0,1,0), d.normalize()); o.scale.set(1, L, 1);
    o.castShadow = true; g.add(o); return o;
  }
  function floorDisc(g, rad, col){ var o = new THREE.Mesh(new THREE.CircleGeometry(rad, 64), stdMat(col, 0.3, 0.55)); o.rotation.x = -Math.PI/2; o.receiveShadow = true; g.add(o); return o; }
  function gridFloor(g, size, col1, col2){ var gh = new THREE.GridHelper(size, size*4, col1, col2); gh.position.y = 0.002; gh.material.transparent = true; gh.material.opacity = 0.5; g.add(gh); }
  function checker(){ var c = document.createElement('canvas'); c.width = c.height = 128; var x = c.getContext('2d');
    x.fillStyle = '#1b2a4a'; x.fillRect(0,0,128,128); x.fillStyle = '#223559'; x.fillRect(0,0,64,64); x.fillRect(64,64,64,64);
    var t = new THREE.CanvasTexture(c); t.wrapS = t.wrapT = THREE.RepeatWrapping; t.repeat.set(8, 8); return t; }

  var worlds = {}, builders = {};
  // HOME: holographic stage
  builders.home = function(){
    var g = new THREE.Group(); worlds.home = g; scene.add(g);
    floorDisc(g, 9, 0x070d22);
    var pl = cyl(g, 1.25, 1.3, 0.06, 0x0e1a38, 0, -0.03, 0, 0.4, 0.4);
    var ring = new THREE.Mesh(new THREE.TorusGeometry(1.2, 0.012, 8, 72), new THREE.MeshBasicMaterial({color:0x22d3ee})); ring.rotation.x = Math.PI/2; ring.position.y = 0.004; g.add(ring);
    var ring2 = new THREE.Mesh(new THREE.TorusGeometry(1.6, 0.008, 8, 72), new THREE.MeshBasicMaterial({color:0x1e6fa8})); ring2.rotation.x = Math.PI/2; ring2.position.y = 0.003; g.add(ring2);
    gridFloor(g, 12, 0x22d3ee, 0x123a55);
  };
  // DRAW STUDIO: glass canvas in front of the robot
  var CV = {cx:-0.08, cy:1.02, cz:0.32, hw:0.25, hh:0.22};
  builders.draw = function(){
    var g = new THREE.Group(); worlds.draw = g; scene.add(g);
    floorDisc(g, 9, 0x1a1410);
    var wood = new THREE.Mesh(new THREE.CircleGeometry(2.2, 48), stdMat(0x3a2a1e, 0.1, 0.8)); wood.rotation.x = -Math.PI/2; wood.position.y = 0.003; wood.receiveShadow = true; g.add(wood);
    var glass = new THREE.Mesh(new THREE.PlaneGeometry(CV.hw*2, CV.hh*2), new THREE.MeshBasicMaterial({color:0x22d3ee, transparent:true, opacity:0.10, side:THREE.DoubleSide, depthWrite:false}));
    glass.position.set(CV.cx, CV.cy, CV.cz); g.add(glass);
    var fcol = 0xf59e0b, t = 0.012;
    box(g, CV.hw*2+t*2, t, t, fcol, CV.cx, CV.cy+CV.hh, CV.cz, 0.5, 0.4); box(g, CV.hw*2+t*2, t, t, fcol, CV.cx, CV.cy-CV.hh, CV.cz, 0.5, 0.4);
    box(g, t, CV.hh*2, t, fcol, CV.cx-CV.hw, CV.cy, CV.cz, 0.5, 0.4); box(g, t, CV.hh*2, t, fcol, CV.cx+CV.hw, CV.cy, CV.cz, 0.5, 0.4);
    box(g, 0.02, CV.cy-CV.hh, 0.02, 0x8895b0, CV.cx-CV.hw, (CV.cy-CV.hh)/2, CV.cz, 0.6, 0.4);
    box(g, 0.02, CV.cy-CV.hh, 0.02, 0x8895b0, CV.cx+CV.hw, (CV.cy-CV.hh)/2, CV.cz, 0.6, 0.4);
    box(g, CV.hw*2+0.2, 0.02, 0.2, 0x8895b0, CV.cx, 0.01, CV.cz, 0.6, 0.4);
    // paint pots
    [0xef4444, 0x3b82f6, 0x22c55e, 0xfacc15].forEach(function(c, i){ cyl(g, 0.04, 0.035, 0.07, c, 0.55 + i*0.1, 0.035, 0.3, 0.1, 0.5); });
    box(g, 0.5, 0.03, 0.2, 0x5b4636, 0.7, 0.4, 0.3, 0.1, 0.7);
    gridFloor(g, 8, 0xf59e0b, 0x4a3512);
  };
  // FIGHT RING + wooden dummy
  builders.fight = function(){
    var g = new THREE.Group(); worlds.fight = g; scene.add(g);
    floorDisc(g, 10, 0x05060f); g.children[0].position.y = -0.27;
    var S = 1.55;
    box(g, S*2+0.4, 0.27, S*2+0.4, 0x15151f, 0, -0.135, 0, 0.2, 0.7);
    var mc = document.createElement('canvas'); mc.width = mc.height = 256; var x = mc.getContext('2d');
    x.fillStyle = '#12224a'; x.fillRect(0,0,256,256); x.strokeStyle = '#e2e8f0'; x.lineWidth = 6; x.strokeRect(14,14,228,228);
    x.strokeStyle = '#ef4444'; x.lineWidth = 3; x.beginPath(); x.arc(128,128,70,0,Math.PI*2); x.stroke();
    var mt = new THREE.CanvasTexture(mc); var mat = new THREE.Mesh(new THREE.PlaneGeometry(S*2, S*2), new THREE.MeshStandardMaterial({map:mt, roughness:0.8, metalness:0.1}));
    mat.rotation.x = -Math.PI/2; mat.position.y = 0.002; mat.receiveShadow = true; g.add(mat);
    var pc = [0xdc2626, 0xf8fafc, 0x2563eb, 0xf8fafc], pos = [[-S,-S],[S,-S],[S,S],[-S,S]];
    pos.forEach(function(p, i){ cyl(g, 0.055, 0.055, 1.35, pc[i], p[0], 0.675, p[1], 0.3, 0.5); box(g, 0.14, 0.12, 0.14, 0x111827, p[0], 0.45, p[1], 0.2, 0.6); });
    var rc = [0xdc2626, 0xf8fafc, 0x2563eb];
    [0.45, 0.82, 1.18].forEach(function(h, j){
      for(var i = 0; i < 4; i++){ var a = pos[i], b = pos[(i+1)%4]; rod(g, [a[0], h, a[1]], [b[0], h, b[1]], 0.014, rc[j]); }
    });
    // wing-chun wooden dummy (faces the robot)
    var dg = new THREE.Group(); dg.position.set(0, 0, 0.95); g.add(dg); var wd = 0x8a5a2b;
    cyl(dg, 0.095, 0.1, 1.45, wd, 0, 0.72, 0, 0.05, 0.85);
    box(dg, 0.34, 0.12, 0.34, 0x4b3320, 0, 0.06, 0, 0.05, 0.9);
    rod(dg, [-0.1, 1.1, -0.06], [-0.2, 1.1, -0.44], 0.03, wd); rod(dg, [0.1, 1.1, -0.06], [0.2, 1.1, -0.44], 0.03, wd);
    rod(dg, [0.0, 0.84, -0.07], [0.05, 0.84, -0.40], 0.03, wd);
    rod(dg, [0.06, 0.62, -0.06], [0.1, 0.5, -0.36], 0.032, wd); rod(dg, [0.1, 0.5, -0.36], [0.1, 0.22, -0.3], 0.03, wd);
    var sp = new THREE.SpotLight(0xfff2d0, 2.2, 12, 0.7, 0.5); sp.position.set(0, 5, 1); sp.target.position.set(0, 0.5, 0.3); g.add(sp); g.add(sp.target);
  };
  // LIFE ROOM: floor, wall, chair, table, window
  builders.life = function(){
    var g = new THREE.Group(); worlds.life = g; scene.add(g);
    floorDisc(g, 9, 0x0a1022);
    var fl = new THREE.Mesh(new THREE.PlaneGeometry(5, 5), new THREE.MeshStandardMaterial({map:checker(), roughness:0.6, metalness:0.2}));
    fl.rotation.x = -Math.PI/2; fl.position.y = 0.002; fl.receiveShadow = true; g.add(fl);
    box(g, 5, 2.6, 0.08, 0x16213f, 0, 1.3, -1.6, 0.1, 0.9);
    box(g, 1.3, 1.0, 0.04, 0x0b1630, 1.2, 1.5, -1.54, 0.3, 0.3); box(g, 1.34, 0.04, 0.05, 0x8895b0, 1.2, 1.0, -1.52, 0.5, 0.4); box(g, 1.34, 0.04, 0.05, 0x8895b0, 1.2, 2.0, -1.52, 0.5, 0.4);
    // chair (seat top y = 0.34, centre z = -0.30)
    box(g, 0.36, 0.04, 0.36, 0x3b5bdb, 0, 0.32, -0.30, 0.2, 0.6);
    [[-0.15,-0.15],[0.15,-0.15],[-0.15,0.15],[0.15,0.15]].forEach(function(p){ box(g, 0.03, 0.30, 0.03, 0x8895b0, p[0], 0.15, -0.30+p[1], 0.6, 0.4); });
    box(g, 0.36, 0.34, 0.03, 0x3b5bdb, 0, 0.52, -0.46, 0.2, 0.6);
    // table + cup
    box(g, 0.7, 0.04, 0.4, 0x7a5a3a, -1.1, 0.7, 0.1, 0.1, 0.7);
    [[-0.3,-0.16],[0.3,-0.16],[-0.3,0.16],[0.3,0.16]].forEach(function(p){ box(g, 0.04, 0.68, 0.04, 0x5a4028, -1.1+p[0], 0.34, 0.1+p[1], 0.1, 0.7); });
    cyl(g, 0.04, 0.035, 0.09, 0xf8fafc, -1.0, 0.765, 0.1, 0.1, 0.4);
    // lamp
    cyl(g, 0.02, 0.02, 1.5, 0x8895b0, 1.8, 0.75, -0.9, 0.6, 0.4); cyl(g, 0.16, 0.08, 0.2, 0xfbbf24, 1.8, 1.55, -0.9, 0.1, 0.5);
    var pl = new THREE.PointLight(0xffd9a0, 1.0, 5); pl.position.set(1.8, 1.5, -0.7); g.add(pl);
  };
  function ensureWorld(n){ if(!worlds[n] && builders[n]){ builders[n](); worlds[n].visible = false; } return worlds[n]; }
  ensureWorld('home'); worlds.home.visible = true;

  // ---- dot layers (pen strokes on the glass canvas) ----
  function dotLayer(col, rad, max){
    var m = new THREE.InstancedMesh(new THREE.SphereGeometry(rad, 8, 6), new THREE.MeshBasicMaterial({color:col}), max);
    m.count = 0; m.frustumCulled = false; worlds.draw.add(m);
    var d = new THREE.Object3D(), L = {mesh:m, n:0, max:max, last:null};
    L.add = function(x, y, z){
      if(L.n >= max) return; d.position.set(x, y, z); d.updateMatrix(); m.setMatrixAt(L.n++, d.matrix); m.count = L.n; m.instanceMatrix.needsUpdate = true; L.last = [x, y, z];
    };
    L.clear = function(){ L.n = 0; m.count = 0; L.last = null; };
    return L;
  }
  var inkLayer = null, tgtLayer = null, trcLayer = null;
  function ensureDraw(){ ensureWorld('draw'); if(!inkLayer){ inkLayer = dotLayer(0xf59e0b, 0.0042, 5000); tgtLayer = dotLayer(0x64748b, 0.0028, 600); trcLayer = dotLayer(0x34d399, 0.0045, 3000); } }
  var pen = new THREE.Mesh(new THREE.CylinderGeometry(0.008, 0.008, 1, 8), new THREE.MeshStandardMaterial({color:0xf59e0b, emissive:0xf59e0b, emissiveIntensity:0.6}));
  pen.visible = false; scene.add(pen);

  // ---- camera ----
  var CAMS = {home:{ty:0.95, tz:0, dist:2.7, yaw:0.0, pitch:0.1}, draw:{tx:-0.05, ty:1.0, tz:0.1, dist:1.9, yaw:0.0, pitch:0.04},
              fight:{ty:0.9, tz:0.45, dist:4.3, yaw:0.62, pitch:0.2}, life:{tx:0, ty:0.8, tz:-0.1, dist:3.5, yaw:0.42, pitch:0.14}};
  var cam = {tx:0, ty:0.85, tz:0, dist:3.2, yaw:0, pitch:0.12}, goal = Object.assign({tx:0, ty:0.85, tz:0}, CAMS.home);
  var hud = document.createElement('div'); hud.className = 'stagehud'; host.appendChild(hud);
  var fhud = document.createElement('canvas'); fhud.width = 256; fhud.height = 128; fhud.style.cssText = 'position:absolute;right:10px;top:8px;width:170px;height:85px;border-radius:12px;border:1px solid #22d3ee99;background:#030a14;pointer-events:none;box-shadow:0 0 16px #22d3ee55'; fhud.style.setProperty('width', '170px', 'important'); fhud.style.setProperty('height', '85px', 'important'); host.appendChild(fhud); var fhctx = fhud.getContext('2d');
  var NAMES = {home:'🏠 Home stage', draw:'✏️ Drawing studio', fight:'🥊 Fighting ring', life:'🧪 Simulation room'};
  function setWorld(name){
    if(!ensureWorld(name)) name = 'home';
    Object.keys(worlds).forEach(function(k){ worlds[k].visible = (k === name); });
    RB.world = name; var c = CAMS[name]; goal = Object.assign({tx:0, ty:0.85, tz:0}, c);
    hud.textContent = NAMES[name] + '  ·  drag to rotate · scroll to zoom';
    scene.fog.color.setHex(name === 'draw' ? 0x120d08 : 0x050a1c); scene.background.setHex(name === 'draw' ? 0x120d08 : 0x050a1c);
    if(name !== 'draw'){ if(inkLayer){ inkLayer.clear(); tgtLayer.clear(); trcLayer.clear(); } ST.draw = null; pen.visible = false; }
  }
  var drag = null;
  var dom = renderer.domElement;
  dom.style.touchAction = 'none';
  dom.addEventListener('pointerdown', function(e){ drag = [e.clientX, e.clientY]; try{ dom.setPointerCapture(e.pointerId); }catch(_){} });
  dom.addEventListener('pointermove', function(e){ if(!drag) return; goal.yaw -= (e.clientX - drag[0])*0.008; goal.pitch = Math.max(-0.1, Math.min(1.2, goal.pitch + (e.clientY - drag[1])*0.006)); drag = [e.clientX, e.clientY]; });
  dom.addEventListener('pointerup', function(){ drag = null; }); dom.addEventListener('pointercancel', function(){ drag = null; });
  dom.addEventListener('wheel', function(e){ e.preventDefault(); goal.dist = Math.max(1.0, Math.min(8, goal.dist*Math.exp(e.deltaY*0.001))); }, {passive:false});
  var ro = new ResizeObserver(function(){ var w = host.clientWidth, h = host.clientHeight; if(w < 10 || h < 10) return; renderer.setSize(w, h); camera.aspect = w/h; camera.updateProjectionMatrix(); });
  ro.observe(host);

  // ---- state ----
  var ST = {play:null, draw:null};
  var cur = C.restPose(), tgt = C.restPose();

  function startPlay(c){
    var mv = C.decodeMove(c.play);
    ST.play = {mv:mv, t:0, speed:Math.max(0.2, Math.min(3, c.speed || 1)), from:new Float32Array(cur)};
    ST.draw = null; pen.visible = false;
  }
  function uv2w(u, v, lift){ return [CV.cx + u*CV.hw*0.9, CV.cy + v*CV.hh*0.9, CV.cz + (lift || 0)]; }
  function startDraw(d, opts){
    opts = opts || {}; ensureDraw();
    inkLayer.clear(); trcLayer.clear(); if(!opts.keepTarget) tgtLayer.clear();
    var layer = opts.trace ? trcLayer : inkLayer;
    var strokes = (d.strokes || []).filter(function(s){ return s.length > 1; });
    if(!strokes.length) return;
    var tot = 0; strokes.forEach(function(s){ for(var i = 1; i < s.length; i++) tot += Math.hypot(s[i][0]-s[i-1][0], s[i][1]-s[i-1][1]); });
    var T = opts.dur || 7, speed = Math.max(0.2, tot / T), segs = [], t = 0, last = null;
    strokes.forEach(function(s){
      var p0 = [s[0][0], opts.flipY ? s[0][1] : -s[0][1]];
      var dt0 = last ? Math.max(0.2, Math.hypot(p0[0]-last[0], p0[1]-last[1])/(speed*3)) : 0.4;
      segs.push({a:last || p0, b:p0, t0:t, t1:t + dt0, down:false}); t += dt0; last = p0;
      for(var i = 1; i < s.length; i++){
        var p = [s[i][0], opts.flipY ? s[i][1] : -s[i][1]], dd = Math.max(0.01, Math.hypot(p[0]-last[0], p[1]-last[1])/speed);
        segs.push({a:last, b:p, t0:t, t1:t + dd, down:true}); t += dd; last = p;
      }
    });
    ST.draw = {segs:segs, T:t, t:0, k:0, skill:(d.skill === undefined ? 0.5 : d.skill), layer:layer, hold:1.6, lastDot:null};
    ST.play = null;
  }
  function setTarget(pts){
    ensureDraw(); tgtLayer.clear();
    for(var i = 0; i < pts.length; i++){ var w = uv2w(pts[i][0], pts[i][1], 0.002); tgtLayer.add(w[0], w[1], w[2]); }
  }
  function drawPose(P, dt){
    var D = ST.draw; D.t += dt;
    var tt = Math.min(D.t, D.T);
    while(D.k < D.segs.length-1 && tt > D.segs[D.k].t1) D.k++;
    var sg = D.segs[D.k], f = Math.max(0, Math.min(1, (tt - sg.t0)/Math.max(1e-6, sg.t1 - sg.t0)));
    var u = sg.a[0] + (sg.b[0]-sg.a[0])*f, v = sg.a[1] + (sg.b[1]-sg.a[1])*f;
    var amp = (1 - D.skill)*0.07, tm = D.t;
    u += amp*(Math.sin(tm*9.1 + 1) + 0.6*Math.sin(tm*17.3)); v += amp*(Math.sin(tm*8.3) + 0.6*Math.sin(tm*15.1 + 2));
    var down = sg.down && D.t <= D.T;
    var tip = uv2w(u, v, down ? 0 : 0.05);
    var rp = R.root.position;
    var J0 = C.fk(P), wt = [tip[0]-rp.x, tip[1]-rp.y, tip[2]-0.14-rp.z];
    var ik = C.ikArm(J0.shR, wt, C.D.ua, C.D.fa, [-0.3,-1,-0.1]);
    V.setb(P, 7, ik.ua); V.setb(P, 8, ik.fa);
    C.handSet(P, C.HB.R, [0,-0.1,1], [0,1,0], false, 'grip');
    V.setb(P, 0, [0, 1, 0.06]);
    var wp = [ik.wrist[0]+rp.x, ik.wrist[1]+rp.y, ik.wrist[2]+rp.z];
    var a = new THREE.Vector3(wp[0], wp[1], wp[2]), b = new THREE.Vector3(tip[0], tip[1], tip[2]), L = a.distanceTo(b)+0.02;
    pen.visible = true; pen.position.copy(a).add(b).multiplyScalar(0.5);
    pen.quaternion.setFromUnitVectors(new THREE.Vector3(0,1,0), b.clone().sub(a).normalize()); pen.scale.set(1, L, 1);
    if(down){
      var ex = tip[0], ey = tip[1], ez = CV.cz + 0.004;
      if(D.lastDot){
        var dx = ex - D.lastDot[0], dy = ey - D.lastDot[1], dl = Math.hypot(dx, dy), n = Math.floor(dl/0.003);
        for(var i = 1; i <= n; i++) D.layer.add(D.lastDot[0] + dx*i/n, D.lastDot[1] + dy*i/n, ez);
        if(n > 0) D.lastDot = [ex, ey];
      } else { D.layer.add(ex, ey, ez); D.lastDot = [ex, ey]; }
    } else { D.lastDot = null; }
    if(D.t > D.T + D.hold){ ST.draw = null; pen.visible = false; }
  }

  RB._run = function(c){
    if(c.world) setWorld(c.world);
    if(c.stop){ ST.play = null; ST.draw = null; pen.visible = false; }
    if(c.target) setTarget(c.target);
    if(c.draw) startDraw(c.draw, {dur:c.draw.dur || 7});
    if(c.trace) startDraw({strokes:[c.trace.h], skill:1}, {trace:true, flipY:true, dur:c.trace.dur || 2.6, keepTarget:true});
    if(c.play) startPlay(c);
  };

  // ---- main loop ----
  var clock = new THREE.Clock(), lastFace = 0, curMood = '', tAcc = 0, perf = {n:0, t:0, lvl:0};
  function loop(){
    if(!host.isConnected){ renderer.dispose(); RB.ready = false; return; }
    requestAnimationFrame(loop);
    if(document.hidden || window._arenaOn){ clock.getDelta(); return; }          // nothing to draw: save the GPU
    var raw = clock.getDelta(), dt = Math.min(0.05, raw); tAcc += dt; perf.n++; perf.t += raw;
    if(perf.t > 2.5){ var fps = perf.n/perf.t; perf.n = 0; perf.t = 0;
      if(fps < 24 && perf.lvl < 1){ perf.lvl = 1; key.castShadow = false; renderer.setPixelRatio(1); renderer.setSize(host.clientWidth, host.clientHeight); hud.textContent += '  ·  ⚡ light mode'; } }
    // camera easing
    var k = 1 - Math.exp(-dt*5);
    ['tx','ty','tz','dist','yaw','pitch'].forEach(function(n){ cam[n] += (goal[n] - cam[n])*k; });
    camera.position.set(cam.tx + cam.dist*Math.sin(cam.yaw)*Math.cos(cam.pitch), cam.ty + cam.dist*Math.sin(cam.pitch), cam.tz + cam.dist*Math.cos(cam.yaw)*Math.cos(cam.pitch));
    camera.lookAt(cam.tx, cam.ty, cam.tz);
    // pose
    var lock = false;
    var mirror = window._mirrorMain && window._puppet && (performance.now() - window._puppetT) < 700;
    if(mirror){ C.nlerpPose(cur, window._puppet, 1 - Math.exp(-dt*30), cur); ST.play = null; }
    else if(ST.play){
      var p = ST.play; p.t += dt*p.speed; var u = p.t/p.mv.dur;
      C.sampleMove(p.mv.frames, Math.min(u, 1), tgt);
      var w = Math.min(1, p.t/0.35); w = w*w*(3 - 2*w);
      C.nlerpPose(p.from, tgt, w, cur); lock = true;
      if(u >= 1) ST.play = null;
    } else {
      var mood = ST.draw ? 'drawing' : RB.mood;
      var mp = C.moodPose(mood, tAcc, !!window._talking, !!window._listening);
      if(ST.draw) drawPose(mp, dt);
      C.nlerpPose(cur, mp, 1 - Math.exp(-dt*7), cur);
    }
    R.apply(cur, {lock:lock});
    if(curMood !== RB.col){ curMood = RB.col; R.setColor(parseInt(RB.col.slice(1), 16)); }
    if(tAcc - lastFace > 0.066){ lastFace = tAcc; var fmood = ST.play ? 'determined' : (ST.draw ? 'drawing' : RB.mood); window.drawFace(fctx, 256, 128, fmood, tAcc, !!window._talking, RB.col); ftex.needsUpdate = true; window.drawFace(fhctx, 256, 128, fmood, tAcc, !!window._talking, RB.col); }
    renderer.render(scene, camera);
  }
  RB.ready = true; RB.setMood(RB.mood);
  setWorld(RB.world || 'home');
  RB.queue.splice(0).forEach(function(c){ RB._run(c); });
  loop();
};

// poll the caption element the python side re-renders: mood + commands
window.faceSync = function(){
  var f = document.querySelector('#face_box .facebox'); if(!f) return;
  var n = f.getAttribute('data-n'); if(n === window._faceN) return; window._faceN = n;
  window.RB.setMood(f.getAttribute('data-mood') || 'idle');
  var c = f.getAttribute('data-cmd');
  if(c){ try{
    var cmd = JSON.parse(c);
    if(cmd.arena && window.Arena) window.Arena.open(cmd.arena);
    if(cmd.yt && window.learnFromVideo) window.learnFromVideo(cmd.yt);
    window.RB.run(cmd);
  }catch(e){ console.log('cmd error', e); } }
};

// ---------- body + finger learning (MediaPipe pose + hands, runs in the browser) ----------
// each frame -> 49 unit direction vectors (see RBC): 15 body bones + 2 hands x (palm 2 + 15 finger bones)
var POSEB = [ [[23,24],[11,12]], [24,23], [12,11], [[11,12],[7,8]], [[7,8],0],
              [11,13], [13,15], [12,14], [14,16], [23,25], [25,27], [27,31], [24,26], [26,28], [28,32] ];
var HANDB = [ [0,9], [17,5], [1,2],[2,3],[3,4], [5,6],[6,7],[7,8], [9,10],[10,11],[11,12], [13,14],[14,15],[15,16], [17,18],[18,19],[19,20] ];
var HCON = [[0,1],[1,2],[2,3],[3,4],[0,5],[5,6],[6,7],[7,8],[5,9],[9,10],[10,11],[11,12],[9,13],[13,14],[14,15],[15,16],[13,17],[17,18],[18,19],[19,20],[0,17]];
var PLINES = [[11,12],[11,13],[13,15],[12,14],[14,16],[11,23],[12,24],[23,24],[23,25],[25,27],[24,26],[26,28],[27,31],[28,32]];

window._mv = {armed:false, on:false, buf:[], pre:[], last:null, lastTips:null, lastMove:0, xs:null, zs:null};
window.fightSay = function(t){ ['fight_status','aw_status'].forEach(function(id){ var s = document.getElementById(id); if(s) s.textContent = t; }); };
window._mpBase = 'https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.14';
window.loadMP = async function(){
  if(window._mpMod) return;
  var base = window._mpBase;
  var mod = await import(base + '/vision_bundle.mjs'); window._mpMod = mod;
  window._mpFs = await mod.FilesetResolver.forVisionTasks(base + '/wasm');
};
// kind: 'lite' = fast (live mirror / arena), 'full' = more exact (recording, videos)
window.loadPose = async function(kind){
  kind = kind || 'full'; var key = '_pose_' + kind;
  if(window[key]) return window[key];
  await window.loadMP();
  var mod = window._mpMod, fs = window._mpFs;
  var mk = function(d){ return mod.PoseLandmarker.createFromOptions(fs, {baseOptions:{
    modelAssetPath:'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_' + kind + '/float16/1/pose_landmarker_' + kind + '.task',
    delegate:d}, runningMode:'VIDEO', numPoses:1}); };
  try{ window[key] = await mk('GPU'); }catch(e){ window[key] = await mk('CPU'); }
  window._pose = window._pose || window[key];
  return window[key];
};
window.loadHands = async function(){
  if(window._hands) return window._hands;
  var mod = window._mpMod, fs = window._mpFs;
  var mk = function(d){ return mod.HandLandmarker.createFromOptions(fs, {baseOptions:{
    modelAssetPath:'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
    delegate:d}, runningMode:'VIDEO', numHands:2}); };
  try{ window._hands = await mk('GPU'); }catch(e){ try{ window._hands = await mk('CPU'); }catch(e2){ window._hands = null; } }
  return window._hands;
};

// build the 49 vectors for one frame. returns {v:[147 ints], m:'0101..'} or null
window.buildFrame = function(res, hres, M){
  var lw = res.worldLandmarks && res.worldLandmarks[0], li = res.landmarks && res.landmarks[0];
  if(!lw || !li) return null;
  if(M.xs === null){      // calibrate the axes once, assuming the teacher faces the camera
    if((lw[23].visibility||0) > 0.5 && (lw[24].visibility||0) > 0.5 && (lw[0].visibility||0) > 0.5){
      M.xs = (lw[23].x - lw[24].x) >= 0 ? 1 : -1;
      M.zs = (lw[0].z - (lw[7].z + lw[8].z)/2) < 0 ? -1 : 1;
    } else return null;
  }
  var xs = M.xs, zs = M.zs;
  function pt(spec, L){
    var ids = (typeof spec === 'number') ? [spec] : spec, x=0, y=0, z=0, vis = 1;
    ids.forEach(function(i){ x += L[i].x; y += L[i].y; z += L[i].z; vis = Math.min(vis, L[i].visibility === undefined ? 1 : L[i].visibility); });
    var n = ids.length; return {p:[xs*x/n, -y/n, zs*z/n], vis:vis};
  }
  var v = new Array(147).fill(0), m = new Array(49).fill('0');
  function put(b, a, c, minv){
    if(a.vis < minv || c.vis < minv) return;
    var dx = c.p[0]-a.p[0], dy = c.p[1]-a.p[1], dz = c.p[2]-a.p[2], n = Math.sqrt(dx*dx+dy*dy+dz*dz);
    if(n < 1e-6) return;
    v[b*3] = Math.round(dx/n*1000); v[b*3+1] = Math.round(dy/n*1000); v[b*3+2] = Math.round(dz/n*1000); m[b] = '1';
  }
  for(var b=0; b<15; b++){ var sp = POSEB[b]; put(b, pt(sp[0], lw), pt(sp[1], lw), 0.5); }
  var nh = 0;
  if(hres && hres.worldLandmarks && hres.worldLandmarks.length){
    var hs = hres.worldLandmarks, hi = hres.landmarks, wl = li[15], wr = li[16];
    var maxd = M.strict ? 0.12 : 0.25;
    var wv = function(w){ return (w.visibility||0) > 0.3; };
    var dd = function(h, w){ return wv(w) ? Math.hypot(hi[h][0].x - w.x, hi[h][0].y - w.y) : 9; };
    var asg = [];
    if(hs.length === 1){
      var dl = dd(0, wl), dr = dd(0, wr);
      if(Math.min(dl, dr) < maxd) asg.push([0, dl <= dr ? 'L' : 'R']);
      else if(!M.strict && dl > 8 && dr > 8) asg.push([0, hi[0][0].x > 0.5 ? 'L' : 'R']);
    } else {
      var best2 = null, bs = 1e9, i2, j2;
      for(i2 = 0; i2 < hs.length; i2++){ for(j2 = 0; j2 < hs.length; j2++){ if(i2 === j2) continue;
        var sc2 = Math.min(dd(i2, wl), 3) + Math.min(dd(j2, wr), 3); if(sc2 < bs){ bs = sc2; best2 = [i2, j2]; } } }
      if(best2){ if(dd(best2[0], wl) < maxd) asg.push([best2[0], 'L']); if(dd(best2[1], wr) < maxd) asg.push([best2[1], 'R']); }
    }
    asg.forEach(function(p){
      var base = p[1] === 'L' ? 15 : 32, HW = hs[p[0]];
      var P = HW.map(function(q){ return {p:[xs*q.x, -q.y, zs*q.z], vis:1}; });
      for(var k = 0; k < 17; k++){ put(base + k, P[HANDB[k][0]], P[HANDB[k][1]], 0.5); }
      nh++;
    });
  }
  return {v:v, m:m.join(''), nh:nh};
};

window._camIds = {v:'fight_cam', o:'fight_overlay'};
window.fightCamStart = async function(vid, ov){
  window._camIds = {v: vid || 'fight_cam', o: ov || 'fight_overlay'};
  var v0 = document.getElementById(window._camIds.v); if(!v0) return;
  try{
    window.fightSay('loading camera + body & hand models…');
    if(!window._camStream) window._camStream = await navigator.mediaDevices.getUserMedia({video:{width:640, height:480}});
    if(v0.srcObject !== window._camStream){ v0.srcObject = window._camStream; try{ await v0.play(); }catch(e){} }
    var pose = await window.loadPose(window._liveSink ? 'lite' : 'full'), hands = await window.loadHands();
    window._poseOn = true;
    window.fightSay('camera on' + (hands ? ' (body + fingers)' : ' (body only)') + ' — stand back so your whole body is visible');
    if(window._poseLoop) return; window._poseLoop = true;
    var lastT = -1;
    var loop = function(){
      if(!window._poseOn){ window._poseLoop = false; return; }
      var v = document.getElementById(window._camIds.v), cv = document.getElementById(window._camIds.o);
      if(!v || !cv){ requestAnimationFrame(loop); return; }
      if(window._camStream && v.srcObject !== window._camStream){ v.srcObject = window._camStream; v.play().catch(function(){}); }
      var cx = cv.getContext('2d');
      if(v.readyState >= 2 && v.currentTime !== lastT){
        lastT = v.currentTime; var now = performance.now();
        var pz = (window._liveSink ? window._pose_lite : window._pose_full) || pose;
        var res = pz.detectForVideo(v, now), hres;
        if(window._liveSink && !window._mv.armed && hands && ((window._hf = (window._hf || 0) + 1) % 2) === 0 && window._lastH !== undefined) hres = window._lastH;   // fingers every 2nd frame while mirroring = faster
        else { hres = hands ? hands.detectForVideo(v, now) : null; window._lastH = hres; }
        var lm = res.landmarks && res.landmarks[0];
        cv.width = v.videoWidth; cv.height = v.videoHeight; cx.clearRect(0, 0, cv.width, cv.height);
        if(lm){
          cx.strokeStyle = '#22d3ee'; cx.lineWidth = 4;
          PLINES.forEach(function(l){ cx.beginPath(); cx.moveTo(lm[l[0]].x*cv.width, lm[l[0]].y*cv.height); cx.lineTo(lm[l[1]].x*cv.width, lm[l[1]].y*cv.height); cx.stroke(); });
          var asp = v.videoWidth / v.videoHeight, M = window._mv, mot = 0, DIST = [13,14,15,16,27,28];
          var cur = DIST.map(function(i){ return [lm[i].x*asp, lm[i].y]; });
          if(M.last){ for(var i=0;i<cur.length;i++) mot += Math.abs(cur[i][0]-M.last[i][0]) + Math.abs(cur[i][1]-M.last[i][1]); mot /= cur.length; }
          M.last = cur;
          var tips = [];
          if(hres && hres.landmarks){
            cx.strokeStyle = '#fbbf24'; cx.lineWidth = 3;
            hres.landmarks.forEach(function(H){
              HCON.forEach(function(l){ cx.beginPath(); cx.moveTo(H[l[0]].x*cv.width, H[l[0]].y*cv.height); cx.lineTo(H[l[1]].x*cv.width, H[l[1]].y*cv.height); cx.stroke(); });
              [4,8,12,16,20].forEach(function(t){ tips.push([H[t].x*asp - H[0].x*asp, H[t].y - H[0].y, H[0].x*asp, H[0].y]); });
            });
          }
          var fm = 0;   // finger motion: tip position relative to the wrist
          if(M.lastTips && M.lastTips.length === tips.length && tips.length){
            for(var j=0;j<tips.length;j++) fm += Math.abs(tips[j][0]-M.lastTips[j][0]) + Math.abs(tips[j][1]-M.lastTips[j][1]);
            fm /= tips.length;
          }
          M.lastTips = tips; mot += fm;
          if(M.armed || window._liveSink){
            var fr = window.buildFrame(res, hres, M);
            if(fr){
              fr.t = now;
              if(window._liveSink) window._liveSink(fr);
              if(M.armed){
                if(!M.on){
                  M.pre.push(fr); if(M.pre.length > 8) M.pre.shift();
                  if(mot > 0.012){ M.on = true; M.buf = M.pre.slice(); M.lastMove = now; window.fightSay('● recording…'); window.sfx('tick'); }
                } else {
                  M.buf.push(fr); if(mot > 0.005) M.lastMove = now;
                  if(M.buf.length % 15 === 0) window.fightSay('● recording… ' + M.buf.length + ' frames' + (fr.nh ? ' · hands seen' : ' · no hands'));
                  if(now - M.lastMove > 1300 || M.buf.length > 900) window.endMove();
                }
              }
            }
          }
        }
      }
      requestAnimationFrame(loop);
    };
    loop();
  }catch(e){ window.fightSay('camera/pose error: ' + e.message); }
};
window.armMove = async function(){
  await window.fightCamStart('fight_cam', 'fight_overlay');
  var M = window._mv; M.armed = true; M.on = false; M.buf = []; M.pre = []; M.xs = null; M.zs = null; M.lastTips = null;
  window.fightSay('ready — do the move now (I start when you move, stop when you hold still)');
};
window.endMove = function(){
  var M = window._mv; M.armed = false;
  if(!M.on){ window.fightSay('nothing recorded'); return; }
  M.on = false;
  var buf = M.buf, step = Math.max(1, Math.ceil(buf.length / 240));
  var frames = []; for(var i = 0; i < buf.length; i += step){ frames.push({t:Math.round(buf[i].t), v:buf[i].v, m:buf[i].m}); }
  M.buf = []; window.fightSay('saved ' + frames.length + ' frames — watch the robot repeat it'); window.sfx('done');
  var ta = document.querySelector('#move_box textarea, #move_box input'); if(!ta) return;
  var proto = ta.tagName === 'TEXTAREA' ? HTMLTextAreaElement.prototype : HTMLInputElement.prototype;
  Object.getOwnPropertyDescriptor(proto, 'value').set.call(ta, JSON.stringify({k:Date.now(), frames:frames}));
  ta.dispatchEvent(new Event('input', {bubbles:true}));
};
window.fightCamStop = function(){ window._poseOn = false; window._mv.armed = false; window._mirrorMain = false; window._liveSink = null; window.stopCam && window.stopCam(); window.fightSay('camera off'); };

// ===================== NEW: live mirror, learn from a video, 3D fight arena =====================
window.frameToPose = function(fr){
  var P = window.RBC.restPose();
  for(var b = 0; b < 49; b++){
    if(fr.m.charAt(b) === '1'){
      var x = fr.v[b*3]/1000, y = fr.v[b*3+1]/1000, z = fr.v[b*3+2]/1000, n = Math.sqrt(x*x + y*y + z*z) || 1;
      P[b*3] = x/n; P[b*3+1] = y/n; P[b*3+2] = z/n;
    }
  }
  return P;
};

// ---- live mirror: the main robot copies the teacher's body and fingers, frame by frame ----
window._mirrorMain = false;
window.mirrorToggle = async function(){
  if(window._mirrorMain){
    window._mirrorMain = false; if(window._liveSink === window._mainSink) window._liveSink = null;
    window.fightSay('mirror off'); if(!window._arenaOn && window.fightCamStop) window.fightCamStop();
    return;
  }
  window._mirrorMain = true; window._mv.xs = null; window._mv.zs = null;
  window._mainSink = function(fr){ window._puppet = window.frameToPose(fr); window._puppetT = performance.now(); };
  window._liveSink = window._mainSink;
  await window.fightCamStart('fight_cam', 'fight_overlay');
  window.fightSay('🪞 mirror on — the robot copies your body and fingers (press the button again to stop)');
};

// ---- learn from a video (YouTube link downloaded by python, or an uploaded file) ----
window.loadPoseMulti = async function(){
  if(window._pose2) return window._pose2;
  await window.loadMP();
  var mod = window._mpMod, fs = window._mpFs;
  var mk = function(d){ return mod.PoseLandmarker.createFromOptions(fs, {baseOptions:{
    modelAssetPath:'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task',
    delegate:d}, runningMode:'VIDEO', numPoses:2}); };
  try{ window._pose2 = await mk('GPU'); }catch(e){ window._pose2 = await mk('CPU'); }
  return window._pose2;
};
window.loadHandsVid = async function(){
  if(window._hands2) return window._hands2;
  var mod = window._mpMod, fs = window._mpFs;
  var mk = function(d){ return mod.HandLandmarker.createFromOptions(fs, {baseOptions:{
    modelAssetPath:'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
    delegate:d}, runningMode:'VIDEO', numHands:4}); };
  try{ window._hands2 = await mk('GPU'); }catch(e){ try{ window._hands2 = await mk('CPU'); }catch(e2){ window._hands2 = null; } }
  return window._hands2;
};
window.ytSay = function(t){ var s = document.getElementById('yt_status'); if(s) s.textContent = t; };
window.learnFromVideo = async function(cfg){
  var v = document.getElementById('yt_vid'), cv = document.getElementById('yt_overlay');
  if(!v || !cv){ return; }
  if(window._ytBusy){ window.ytSay('I am still learning from the last video — please wait.'); return; }
  window._ytBusy = true;
  try{
    window.ytSay('loading the video…');
    await new Promise(function(res, rej){
      var tries = [cfg.src, cfg.src2], i = 0;
      function next(){
        if(i >= tries.length || !tries[i]){ rej(new Error('the browser cannot open this video file')); return; }
        v.onloadeddata = function(){ res(); }; v.onerror = function(){ i++; next(); };
        v.src = tries[i]; v.load();
      }
      next();
    });
    window.ytSay('loading body + hand models…');
    var pose = await window.loadPoseMulti(), hands = await window.loadHandsVid();
    v.muted = true; v.playsInline = true; v.playbackRate = Math.max(0.1, Math.min(1, cfg.rate || 0.4));
    var M = {xs:null, zs:null, strict:true}, frames = [], lastTs = 0, pickMode = cfg.pick || 'big', maxT = (cfg.start || 0) + (cfg.len || 20), cx = cv.getContext('2d');
    v.currentTime = cfg.start || 0; await v.play();
    await new Promise(function(done){
      var finished = false;
      function finish(){ if(finished) return; finished = true; try{ v.pause(); }catch(e){} done(); }
      v.onended = finish;
      function step(){
        if(finished) return;
        if(v.ended || v.currentTime >= maxT){ finish(); return; }
        var ts = Math.max(lastTs + 1, Math.round(v.currentTime*1000)); lastTs = ts;
        try{
          var res = pose.detectForVideo(v, ts), hres = hands ? hands.detectForVideo(v, ts) : null;
          var n = res.landmarks ? res.landmarks.length : 0, sel = null;
          cv.width = v.videoWidth || 320; cv.height = v.videoHeight || 240; cx.clearRect(0, 0, cv.width, cv.height);
          if(n){
            var best = 0;
            if(n > 1){
              var hx = res.landmarks.map(function(L){ return (L[23].x + L[24].x)/2; });
              var span = res.landmarks.map(function(L){ return Math.abs((L[11].y + L[12].y)/2 - (L[23].y + L[24].y)/2); });
              for(var q = 1; q < n; q++){
                if(pickMode === 'left' ? hx[q] < hx[best] : (pickMode === 'right' ? hx[q] > hx[best] : span[q] > span[best])) best = q;
              }
            }
            sel = {worldLandmarks:[res.worldLandmarks[best]], landmarks:[res.landmarks[best]]};
            var lm = res.landmarks[best]; cx.strokeStyle = '#22d3ee'; cx.lineWidth = 3;
            PLINES.forEach(function(l){ cx.beginPath(); cx.moveTo(lm[l[0]].x*cv.width, lm[l[0]].y*cv.height); cx.lineTo(lm[l[1]].x*cv.width, lm[l[1]].y*cv.height); cx.stroke(); });
            if(hres && hres.landmarks){ cx.strokeStyle = '#fbbf24'; cx.lineWidth = 2;
              hres.landmarks.forEach(function(H){ HCON.forEach(function(l){ cx.beginPath(); cx.moveTo(H[l[0]].x*cv.width, H[l[0]].y*cv.height); cx.lineTo(H[l[1]].x*cv.width, H[l[1]].y*cv.height); cx.stroke(); }); }); }
            var fr = window.buildFrame(sel, hres, M); if(fr){ fr.t = ts; frames.push(fr); }
          }
          if(frames.length % 10 === 0) window.ytSay('watching… ' + (v.currentTime).toFixed(1) + ' s · ' + frames.length + ' poses');
        }catch(e){ console.log('video frame error', e); }
        if(v.requestVideoFrameCallback) v.requestVideoFrameCallback(step); else requestAnimationFrame(step);
      }
      step();
    });
    if(frames.length < 8){ window.ytSay('I could not find a person in this part of the video — try another start time or "Left/Right person".'); window._ytBusy = false; return; }
    var stride = Math.max(1, Math.ceil(frames.length/360)), out = [];
    for(var i = 0; i < frames.length; i += stride) out.push({t:frames[i].t, v:frames[i].v, m:frames[i].m});
    window.ytSay('watched ' + frames.length + ' poses — turning them into a move…'); window.sfx && window.sfx('done');
    var ta = document.querySelector('#yt_box textarea, #yt_box input');
    if(ta){
      var proto = ta.tagName === 'TEXTAREA' ? HTMLTextAreaElement.prototype : HTMLInputElement.prototype;
      Object.getOwnPropertyDescriptor(proto, 'value').set.call(ta, JSON.stringify({k:Date.now(), frames:out}));
      ta.dispatchEvent(new Event('input', {bubbles:true}));
    }
  }catch(e){ window.ytSay('video error: ' + e.message); }
  window._ytBusy = false;
};

// ---- the fight arena: a separate 3D window with two robots; close it with the X button or Esc ----
window.Arena = (function(){
var A = {alive:false}, ST = null;
var CSSTXT = '#arena_win{position:fixed;left:2.5vw;top:2.5vh;width:95vw;height:95vh;z-index:100000;background:#030617;border:2px solid #22d3ee99;'
 + 'border-radius:18px;box-shadow:0 0 60px #000,0 0 40px #22d3ee55;display:flex;flex-direction:column;font:13px sans-serif;color:#d6f3ff}'
 + '#arena_win .aw_bar{display:flex;gap:8px;align-items:center;padding:8px 12px;border-bottom:1px solid #22d3ee44;flex-wrap:wrap}'
 + '#arena_win .aw_bar b{font-family:Orbitron,sans-serif;letter-spacing:1px;color:#7fe8ff}'
 + '#arena_win button{background:#14326499;color:#bfeaff;border:1px solid #22d3ee88;border-radius:8px;padding:5px 10px;cursor:pointer;font-size:12px}'
 + '#arena_win button:hover{box-shadow:0 0 12px #22d3ee99}#arena_win button.on{background:#0e7490}'
 + '#arena_win #aw_close{background:#7f1d1d;border-color:#f87171}'
 + '#arena_win .sp{flex:1}#arena_win #aw_score{font-size:18px;font-weight:700;color:#fbbf24;font-family:Orbitron,sans-serif}'
 + '#arena_win .aw_body{flex:1;display:flex;min-height:0}#arena_win #aw_stage{flex:1;position:relative;min-width:0}'
 + '#arena_win #aw_stage canvas{width:100%!important;height:100%!important;display:block;cursor:grab}'
 + '#arena_win .aw_side{width:300px;padding:10px;border-left:1px solid #22d3ee44;overflow:auto;display:flex;flex-direction:column;gap:8px}'
 + '#arena_win .mir{position:relative;transform:scaleX(-1);border-radius:12px;overflow:hidden;border:2px solid #22d3ee88;background:#000;min-height:100px}'
 + '#arena_win .mir video,#arena_win .mir canvas{width:100%;display:block}#arena_win .mir canvas{position:absolute;left:0;top:0;height:100%}'
 + '#arena_win .chip{display:inline-block;margin:2px;padding:3px 8px;border-radius:10px;border:1px solid #22d3ee77;background:#0b1a33;cursor:pointer}'
 + '#arena_win .chip.p{border-color:#f87171aa;cursor:default}#arena_win .hint{color:#7fb4d6;font-size:12px;line-height:1.5}'
 + '@media (max-width:800px){#arena_win .aw_body{flex-direction:column}#arena_win .aw_side{width:auto;border-left:0;border-top:1px solid #22d3ee44}}';

A.open = async function(data){
  if(A.alive){ A.setData(data || {}); return; }
  if(A._opening) return; A._opening = true;
  try{
    if(!document.getElementById('aw_css')){ var s = document.createElement('style'); s.id = 'aw_css'; s.textContent = CSSTXT; document.head.appendChild(s); }
    var win = document.createElement('div'); win.id = 'arena_win';
    win.innerHTML = '<div class="aw_bar"><b>🥊 FIGHT ARENA</b><span id="aw_score">YOU 0 – 0 PARTNER</span><span class="sp"></span>'
      + '<button id="aw_cam">📷 Camera control: OFF</button><button id="aw_spar">▶ Partner attacks: OFF</button>'
      + '<button id="aw_reset">⟲ Reset</button><label style="font-size:12px">⚡ Speed <input id="aw_speed" type="range" min="0.6" max="3" step="0.1" value="1.6" style="vertical-align:middle;width:90px"></label><button id="aw_close">✖ Close window</button></div>'
      + '<div class="aw_body"><div id="aw_stage">loading 3D arena…</div><div class="aw_side">'
      + '<div class="mir"><video id="arena_cam" autoplay muted playsinline></video><canvas id="arena_ov"></canvas></div>'
      + '<div id="aw_status" class="hint">Camera off. Press “Camera control” and stand back: the cyan robot copies your whole body and fingers.</div>'
      + '<div><b>Your moves</b> (keys 1-9)<div id="aw_mine"></div></div><div><b>Partner (red) knows</b><div id="aw_part"></div></div>'
      + '<div class="hint">Arrow keys walk. The red partner keeps its distance and spars with the moves it learned from video. Esc closes this window.</div>'
      + '</div></div>';
    document.body.appendChild(win); A.win = win;
    var THREE = window._THREE || await import('https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js'); window._THREE = THREE;
    if(!A.win || !document.getElementById('arena_win')) return;      // closed while loading
    A._pending = data || {}; A.build(THREE, win);
  }catch(e){
    var h = document.getElementById('aw_stage'); if(h) h.textContent = 'arena failed to start: ' + e.message;
  }
  A._opening = false;
};

A.setData = function(data){
  if(!ST) return;
  var C = window.RBC;
  ST.mine = (data.mine || []).map(function(m){ var d = C.decodeMove(m); d.name = m.name; return d; });
  ST.partner = (data.partner || []).map(function(m){ var d = C.decodeMove(m); d.name = m.name; return d; });
  var mine = document.getElementById('aw_mine'), part = document.getElementById('aw_part');
  if(mine){ mine.innerHTML = ''; ST.mine.forEach(function(m, i){ var c = document.createElement('span'); c.className = 'chip'; c.textContent = (i < 9 ? (i+1) + ': ' : '') + m.name; c.onclick = function(){ A.playMine(i); }; mine.appendChild(c); });
    if(!ST.mine.length) mine.textContent = 'none yet — teach moves in the Fighting panel'; }
  if(part){ part.innerHTML = ''; ST.partner.forEach(function(m){ var c = document.createElement('span'); c.className = 'chip p'; c.textContent = m.name; part.appendChild(c); });
    if(data.defaults) { var n = document.createElement('div'); n.className = 'hint'; n.textContent = 'These are starter moves. Teach the partner from a video to give it real ones.'; part.appendChild(n); } }
};

A.debug = function(){ return ST; };
A.speed = function(){ var e = document.getElementById('aw_speed'); return e ? (parseFloat(e.value) || 1.5) : 1.5; };
A.playMine = function(i){ if(ST && ST.mine[i]) startPlay(ST.a, ST.mine[i], A.speed()); };

function startPlay(bot, mv, speed){ bot.play = {mv:mv, t:0, speed:speed || 1, from:new Float32Array(bot.cur)}; }

A.close = function(){
  if(!A.alive && !A.win) return;
  A.alive = false;
  if(A.raf) cancelAnimationFrame(A.raf);
  document.removeEventListener('keydown', A.onKey); document.removeEventListener('keyup', A.onKeyUp);
  try{ if(ST && ST.renderer) ST.renderer.dispose(); }catch(e){}
  if(window._arenaOn){ window._arenaOn = false; window._liveSink = null; if(window.fightCamStop) window.fightCamStop(); }
  if(A.win && A.win.parentNode) A.win.parentNode.removeChild(A.win);
  A.win = null; ST = null; A._opening = false;
};

A.build = function(THREE, win){
  var C = window.RBC, V = C.v, host = win.querySelector('#aw_stage'); host.textContent = '';
  var renderer = new THREE.WebGLRenderer({antialias:true}); renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 1.5));
  renderer.setSize(host.clientWidth || 800, host.clientHeight || 500); renderer.shadowMap.enabled = true; renderer.shadowMap.type = THREE.PCFSoftShadowMap;
  host.appendChild(renderer.domElement);
  var scene = new THREE.Scene(); scene.background = new THREE.Color(0x040814); scene.fog = new THREE.Fog(0x040814, 8, 20);
  var camera = new THREE.PerspectiveCamera(40, (host.clientWidth || 800)/(host.clientHeight || 500), 0.05, 60);
  scene.add(new THREE.HemisphereLight(0xcfe8ff, 0x1a2040, 0.9));
  var key = new THREE.DirectionalLight(0xffffff, 1.4); key.position.set(2.5, 5, 3); key.castShadow = true; key.shadow.mapSize.set(512, 512);
  var sc = key.shadow.camera; sc.left = -3; sc.right = 3; sc.top = 3; sc.bottom = -3; sc.near = 0.5; sc.far = 14; scene.add(key);
  var sp = new THREE.SpotLight(0xfff2d0, 2.4, 14, 0.8, 0.5); sp.position.set(0, 5.5, 0); sp.target.position.set(0, 0, 0); scene.add(sp); scene.add(sp.target);
  function mat(c, m, r){ return new THREE.MeshStandardMaterial({color:c, metalness:m === undefined ? 0.2 : m, roughness:r === undefined ? 0.7 : r}); }
  function box(w, h, d, c, x, y, z){ var o = new THREE.Mesh(new THREE.BoxGeometry(w, h, d), mat(c)); o.position.set(x, y, z); o.castShadow = o.receiveShadow = true; scene.add(o); return o; }
  function cyl(rt, rb, h, c, x, y, z){ var o = new THREE.Mesh(new THREE.CylinderGeometry(rt, rb, h, 20), mat(c, 0.3, 0.5)); o.position.set(x, y, z); o.castShadow = true; scene.add(o); return o; }
  function rod(a, b, r, c){ var o = new THREE.Mesh(new THREE.CylinderGeometry(r, r, 1, 8), mat(c, 0.1, 0.6)); var d = new THREE.Vector3(b[0]-a[0], b[1]-a[1], b[2]-a[2]), L = d.length();
    o.position.set((a[0]+b[0])/2, (a[1]+b[1])/2, (a[2]+b[2])/2); o.quaternion.setFromUnitVectors(new THREE.Vector3(0,1,0), d.normalize()); o.scale.set(1, L, 1); scene.add(o); }
  var fl = new THREE.Mesh(new THREE.CircleGeometry(12, 48), mat(0x05060f, 0.3, 0.6)); fl.rotation.x = -Math.PI/2; fl.position.y = -0.27; scene.add(fl);
  var S = 1.55; box(S*2+0.4, 0.27, S*2+0.4, 0x15151f, 0, -0.135, 0);
  var mc = document.createElement('canvas'); mc.width = mc.height = 256; var x = mc.getContext('2d');
  x.fillStyle = '#12224a'; x.fillRect(0,0,256,256); x.strokeStyle = '#e2e8f0'; x.lineWidth = 6; x.strokeRect(14,14,228,228);
  x.strokeStyle = '#ef4444'; x.lineWidth = 3; x.beginPath(); x.arc(128,128,70,0,Math.PI*2); x.stroke();
  var mp = new THREE.Mesh(new THREE.PlaneGeometry(S*2, S*2), new THREE.MeshStandardMaterial({map:new THREE.CanvasTexture(mc), roughness:0.8, metalness:0.1}));
  mp.rotation.x = -Math.PI/2; mp.position.y = 0.002; mp.receiveShadow = true; scene.add(mp);
  var pc = [0xdc2626, 0xf8fafc, 0x2563eb, 0xf8fafc], ps = [[-S,-S],[S,-S],[S,S],[-S,S]];
  ps.forEach(function(p, i){ cyl(0.055, 0.055, 1.35, pc[i], p[0], 0.675, p[1]); box(0.14, 0.12, 0.14, 0x111827, p[0], 0.45, p[1]); });
  [0.45, 0.82, 1.18].forEach(function(h, j){ for(var i = 0; i < 4; i++){ var a = ps[i], b = ps[(i+1)%4]; rod([a[0], h, a[1]], [b[0], h, b[1]], 0.014, [0xdc2626, 0xf8fafc, 0x2563eb][j]); } });

  function makeBot(col, hex, pos){
    var cv = document.createElement('canvas'); cv.width = 256; cv.height = 128;
    var tex = new THREE.CanvasTexture(cv), R = C.build(THREE, tex), holder = new THREE.Group();
    holder.add(R.root); scene.add(holder); R.setColor(hex);
    return {R:R, holder:holder, ctx:cv.getContext('2d'), tex:tex, col:col, cur:C.restPose(), play:null, hitT:-9, scoreT:-9, score:0, pos:pos.slice(), prev:null, cool:0, yaw:0};
  }
  var a = makeBot('#22d3ee', 0x22d3ee, [0, -0.45]), b = makeBot('#f87171', 0xf87171, [0, 0.45]);
  ST = {renderer:renderer, scene:scene, camera:camera, a:a, b:b, mine:[], partner:[], sparring:false, aiT:1.5, keys:{}, puppet:null, puppetT:0, camOn:false};
  A.setData(A._pending || {});
  A.alive = true; window._arenaOn = true;

  var cam = {yaw:0.55, pitch:0.3, dist:3.4}, drag = null, dom = renderer.domElement;
  dom.addEventListener('pointerdown', function(e){ drag = [e.clientX, e.clientY]; try{ dom.setPointerCapture(e.pointerId); }catch(_){} });
  dom.addEventListener('pointermove', function(e){ if(!drag) return; cam.yaw -= (e.clientX-drag[0])*0.008; cam.pitch = Math.max(0.02, Math.min(1.3, cam.pitch + (e.clientY-drag[1])*0.006)); drag = [e.clientX, e.clientY]; });
  dom.addEventListener('pointerup', function(){ drag = null; });
  dom.addEventListener('wheel', function(e){ e.preventDefault(); cam.dist = Math.max(1.6, Math.min(8, cam.dist*Math.exp(e.deltaY*0.001))); }, {passive:false});
  new ResizeObserver(function(){ var w = host.clientWidth, h = host.clientHeight; if(w < 10 || h < 10) return; renderer.setSize(w, h); camera.aspect = w/h; camera.updateProjectionMatrix(); }).observe(host);

  var scoreEl = win.querySelector('#aw_score');
  function showScore(){ scoreEl.textContent = 'YOU ' + a.score + ' – ' + b.score + ' PARTNER'; }
  win.querySelector('#aw_close').onclick = A.close;
  win.querySelector('#aw_reset').onclick = function(){ a.score = b.score = 0; a.pos = [0, -0.45]; b.pos = [0, 0.45]; showScore(); };
  var sparBtn = win.querySelector('#aw_spar'), camBtn = win.querySelector('#aw_cam');
  sparBtn.onclick = function(){ ST.sparring = !ST.sparring; sparBtn.textContent = '▶ Partner attacks: ' + (ST.sparring ? 'ON' : 'OFF'); sparBtn.className = ST.sparring ? 'on' : ''; };
  camBtn.onclick = async function(){
    if(ST.camOn){ ST.camOn = false; window._liveSink = null; camBtn.textContent = '📷 Camera control: OFF'; camBtn.className = ''; if(window.fightCamStop) window.fightCamStop(); window._arenaOn = true; return; }
    ST.camOn = true; camBtn.textContent = '📷 Camera control: ON'; camBtn.className = 'on';
    window._mv.xs = null; window._mv.zs = null; window._mirrorMain = false;
    window._liveSink = function(fr){ if(ST){ ST.puppet = window.frameToPose(fr); ST.puppetT = performance.now(); } };
    await window.fightCamStart('arena_cam', 'arena_ov');
  };
  A.onKey = function(e){
    if(!A.alive) return;
    if(e.key === 'Escape'){ A.close(); return; }
    var t = e.target && e.target.tagName; if(t === 'INPUT' || t === 'TEXTAREA') return;
    if(/^Arrow/.test(e.key)){ ST.keys[e.key] = true; e.preventDefault(); }
    if(/^[1-9]$/.test(e.key)){ A.playMine(parseInt(e.key, 10) - 1); }
  };
  A.onKeyUp = function(e){ if(ST) delete ST.keys[e.key]; };
  document.addEventListener('keydown', A.onKey); document.addEventListener('keyup', A.onKeyUp);

  var tmp = C.restPose(), clock = new THREE.Clock(), tAcc = 0, lastFace = 0, V3 = THREE.Vector3;
  function stepPlay(bot, dt){
    var p = bot.play; p.t += dt*p.speed; var u = p.t/p.mv.dur;
    C.sampleMove(p.mv.frames, Math.min(u, 1), tmp);
    var w = Math.min(1, p.t/0.3); w = w*w*(3 - 2*w); C.nlerpPose(p.from, tmp, w, bot.cur);
    if(u >= 1) bot.play = null;
  }
  function wpts(bot){
    var J = bot.R.J, r = bot.R.root, up = V.getb(bot.cur, 0);
    var W = function(q){ return r.localToWorld(new V3(q[0], q[1], q[2])); };
    var tipL = V.add(J.wrL, V.mul(V.getb(bot.cur, 6), 0.15)), tipR = V.add(J.wrR, V.mul(V.getb(bot.cur, 8), 0.15));   // fist, not wrist
    return {hands:[W(tipL), W(tipR)], feet:[W(J.toL), W(J.toR)], chest:W(V.mul(up, 0.30)), head:W(J.head)};
  }
  function clampPos(p){ p[0] = Math.max(-1.2, Math.min(1.2, p[0])); p[1] = Math.max(-1.2, Math.min(1.2, p[1])); }
  function check(att, def, ap, dp, dt, now){
    var pts = ap.hands.concat(ap.feet), prev = att.prev;
    if(!att.pk) att.pk = [0, 0, 0, 0];
    for(var i = 0; i < pts.length; i++){
      if(prev){
        var sp = pts[i].distanceTo(prev[i])/Math.max(dt, 1e-3); if(sp > 14) sp = 0;
        att.pk[i] = Math.max(att.pk[i]*Math.exp(-dt*3), sp);          // fast recently = a real strike, even at full stretch
      }
      if(prev && now - att.cool > 0.55){
        att.dmin = Math.min(att.dmin === undefined ? 9 : att.dmin, pts[i].distanceTo(dp.chest)); att.pkmax = Math.max(att.pkmax || 0, att.pk[i]);
        if(att.pk[i] > 1.4){
          var reach = i < 2 ? 0.42 : 0.5;
          if(pts[i].distanceTo(dp.chest) < reach || pts[i].distanceTo(dp.head) < reach - 0.06){
            att.cool = now; att.score++; att.scoreT = now; def.hitT = now; if(window.sfx) window.sfx('oops'); showScore(); break;
          }
        }
      }
    }
    att.prev = pts.map(function(p){ return p.clone(); });
  }
  function loop(){
    if(!A.alive) return;
    A.raf = requestAnimationFrame(loop);
    var dt = Math.min(window._dtCap || 0.05, clock.getDelta()), now = performance.now()/1000; tAcc += dt;
    // walking (arrow keys), the partner keeps its distance and faces you
    var k = ST.keys, fx = b.pos[0]-a.pos[0], fz = b.pos[1]-a.pos[1], fl = Math.hypot(fx, fz) || 1; fx /= fl; fz /= fl;
    var mvf = (k.ArrowUp ? 1 : 0) - (k.ArrowDown ? 1 : 0), mvs = (k.ArrowLeft ? 1 : 0) - (k.ArrowRight ? 1 : 0);
    a.pos[0] += (fx*mvf + fz*mvs)*0.9*dt; a.pos[1] += (fz*mvf - fx*mvs)*0.9*dt; clampPos(a.pos);
    var want = 0.85, tx = a.pos[0] + (b.pos[0]-a.pos[0])/ (Math.hypot(b.pos[0]-a.pos[0], b.pos[1]-a.pos[1]) || 1)*want, tz = a.pos[1] + (b.pos[1]-a.pos[1])/(Math.hypot(b.pos[0]-a.pos[0], b.pos[1]-a.pos[1]) || 1)*want;
    b.pos[0] += (tx - b.pos[0])*Math.min(1, dt*1.3); b.pos[1] += (tz - b.pos[1])*Math.min(1, dt*1.3); clampPos(b.pos);
    [[a, b], [b, a]].forEach(function(pr){
      var me = pr[0], op = pr[1], yaw = Math.atan2(op.pos[0]-me.pos[0], op.pos[1]-me.pos[1]);
      var d = yaw - me.yaw; d = Math.atan2(Math.sin(d), Math.cos(d)); me.yaw += d*Math.min(1, dt*6);
      me.holder.position.set(me.pos[0], 0, me.pos[1]); me.holder.rotation.y = me.yaw;
    });
    // your robot: camera puppet > a key-triggered move > guard
    if(ST.camOn && ST.puppet && performance.now() - ST.puppetT < 600){ C.nlerpPose(a.cur, ST.puppet, 1 - Math.exp(-dt*30), a.cur); a.play = null; }
    else if(a.play){ stepPlay(a, dt); }
    else { C.nlerpPose(a.cur, C.moodPose('idle', tAcc, false, false), 1 - Math.exp(-dt*6), a.cur); }
    // the partner: sparring AI
    if(ST.sparring && !b.play && now - b.hitT > 0.7 && ST.partner.length){
      ST.aiT -= dt;
      if(ST.aiT <= 0){ startPlay(b, ST.partner[Math.floor(Math.random()*ST.partner.length)], A.speed()*(0.9 + Math.random()*0.3)); ST.aiT = (1.2 + Math.random()*1.4)/Math.max(1, A.speed()*0.7); }
    }
    if(b.play) stepPlay(b, dt); else C.nlerpPose(b.cur, C.moodPose('idle', tAcc + 3, false, false), 1 - Math.exp(-dt*6), b.cur);
    // getting hit: lean back and flinch
    [a, b].forEach(function(bot){
      var P = bot.cur, h = now - bot.hitT;
      if(h < 0.45){ P = new Float32Array(bot.cur); var kk = 1 - h/0.45;
        V.setb(P, 0, [0, 1, -0.45*kk]); V.setb(P, 3, [0, 1, -0.35*kk]); V.setb(P, 5, [0.6*kk + 0.1, -0.5, -0.3*kk]); V.setb(P, 7, [-0.6*kk - 0.1, -0.5, -0.3*kk]); }
      bot.R.apply(P, {lock:false});
    });
    scene.updateMatrixWorld(true);
    var pa = wpts(a), pb = wpts(b);
    check(a, b, pa, pb, dt, now); check(b, a, pb, pa, dt, now);
    camera.position.set(Math.sin(cam.yaw)*Math.cos(cam.pitch)*cam.dist, 0.85 + Math.sin(cam.pitch)*cam.dist, Math.cos(cam.yaw)*Math.cos(cam.pitch)*cam.dist);
    camera.lookAt(0, 0.8, 0);
    if(tAcc - lastFace > 0.066){ lastFace = tAcc;
      [a, b].forEach(function(bot){ var mood = (now - bot.hitT < 0.6) ? 'surprised' : (now - bot.scoreT < 0.8 ? 'happy' : (bot.play ? 'determined' : 'idle'));
        window.drawFace(bot.ctx, 256, 128, mood, tAcc, false, bot.col); bot.tex.needsUpdate = true; });
    }
    renderer.render(scene, camera);
  }
  loop();
};
return A;
})();

window.pollAll = function(){
  try{ window.applyMode && window.applyMode(); window.initPaint && window.initPaint(); window.initStage && window.initStage(); window.faceSync && window.faceSync(); }catch(e){}
};

})();
"""

POLL_JS = r"""
if(!window._pollSet){
  window._pollSet = true;
  setInterval(function(){
    try{
      window.pollAll && window.pollAll();
      var el = document.querySelector('#voice_box textarea, #voice_box input');
      if(!el) return;
      var cb = document.querySelector('#voice_on input[type=checkbox]');
      window.onVoice(el.value || '', cb ? cb.checked : true);
    }catch(e){}
  }, 200);
  document.addEventListener('change', function(e){
    var t = e.target;
    if(t && t.type === 'checkbox' && t.closest && t.closest('#voice_on') && t.checked){
      window.babySpeak('Hello! I can talk now.'); window.sfx('ok');
    }
  });
}
"""
HEAD_LINK = ("<link rel=\"preconnect\" href=\"https://fonts.googleapis.com\"><link rel=\"stylesheet\" "
             "href=\"https://fonts.googleapis.com/css2?family=Orbitron:wght@500;700&display=swap\" media=\"print\" onload=\"this.media='all'\">")
HEAD_JS = HEAD_LINK + "<script>" + JS_LIB + POLL_JS + "</script>"

CSS = r"""
html,body,gradio-app{background:#030617!important}
body::before{content:"";position:fixed;inset:0;z-index:-3;
  background:radial-gradient(circle at 12% 8%,#0e2a6a66 0,transparent 45%),
             radial-gradient(circle at 88% 92%,#5b1d8a55 0,transparent 45%),
             linear-gradient(180deg,#030617,#07102b)}
body::after{content:"";position:fixed;left:-50%;right:-50%;bottom:0;height:55%;z-index:-2;pointer-events:none;
  background-image:linear-gradient(#22d3ee26 1px,transparent 1px),linear-gradient(90deg,#22d3ee26 1px,transparent 1px);
  background-size:56px 56px;transform:perspective(420px) rotateX(64deg);transform-origin:50% 100%;
  animation:gridmove 5s linear infinite;-webkit-mask-image:linear-gradient(transparent,#000 70%);mask-image:linear-gradient(transparent,#000 70%)}
@keyframes gridmove{to{background-position:0 56px,0 0}}

.gradio-container{max-width:1280px!important;margin:0 auto!important;background:transparent!important;
  --body-text-color:#d6f3ff;--body-text-color-subdued:#7fb4d6;--block-background-fill:rgba(8,18,44,.55);
  --block-border-color:#1e6fa8aa;--block-label-text-color:#7fd8ff;--background-fill-primary:rgba(8,18,44,.55);
  --background-fill-secondary:rgba(8,18,44,.55);--input-background-fill:rgba(4,10,28,.75);
  --border-color-primary:#1e6fa8aa;--button-secondary-background-fill:rgba(20,50,100,.6);
  --button-secondary-text-color:#bfeaff;--button-secondary-border-color:#22d3ee88}
.gradio-container footer{display:none!important}
.gradio-container .block,.gradio-container .form{backdrop-filter:blur(8px);border-color:#1e6fa8aa!important}
.gradio-container button,.gradio-container textarea,.gradio-container input,.gradio-container label,
.gradio-container .prose,.gradio-container .message,.gradio-container .message *,.gradio-container .md{font-size:12px!important}
.gradio-container button{min-height:28px!important;padding:2px 8px!important;line-height:1.2!important;
  border:1px solid #22d3ee88!important;transition:box-shadow .2s,transform .1s}
.gradio-container button:hover{box-shadow:0 0 14px #22d3ee99;transform:translateY(-1px)}
.gradio-container textarea,.gradio-container input[type=text]{padding:5px 8px!important}
.gradio-container .block{padding:4px!important}
.gradio-container .gap,.gradio-container .form{gap:4px!important}
#voice_box{display:none!important}
#mode_dd label span,#mode_dd .label-wrap{font-family:'Orbitron',sans-serif;letter-spacing:1px;color:#7fe8ff}
#face_box{padding:0!important;border:none!important;background:transparent!important}
.facebox{text-align:center;padding:4px 10px;border-radius:12px;border:1px solid var(--acc);
  box-shadow:0 0 18px -6px var(--acc);background:#070b1c}
.emo{font-size:1.5rem;vertical-align:middle}
#mic_btn.on{background:#0e7490!important;box-shadow:0 0 14px #22d3ee}
.cap{font-family:'Orbitron',sans-serif;color:#bfeaff;font-size:.9rem;font-weight:500;min-height:1.3rem;letter-spacing:.5px}
.board{text-align:center}.board svg{width:100%;max-width:340px;display:block;margin:0 auto}
.bar{height:8px;border-radius:6px;background:#10203f;margin:4px auto;max-width:300px;overflow:hidden}
.bar i{display:block;height:100%;background:linear-gradient(90deg,#f59e0b,#34d399)}
#stage_box{padding:0!important;border:none!important;background:transparent!important}
#stage{position:relative;width:100%;height:min(58vh,540px);min-height:320px;border-radius:18px;overflow:hidden;
  border:1px solid #22d3ee66;background:#030617;color:#7fe8ff;font:12px sans-serif;display:flex;align-items:center;justify-content:center;
  box-shadow:0 0 28px -8px #22d3ee}
#stage canvas{display:block;width:100%!important;height:100%!important;cursor:grab}
.stagehud{position:absolute;left:10px;top:8px;color:#bfeaff;font:11px 'Orbitron',sans-serif;background:#030617aa;
  padding:3px 8px;border-radius:8px;pointer-events:none}
#teach_box textarea{min-height:230px!important;font-size:13px!important}
#grp_cam,#grp_draw,#grp_sim,#grp_fight{display:none}
.gradio-container code,.gradio-container .prose code{background:#0b1a33!important;color:#7fe8ff!important;padding:1px 5px;border-radius:5px}
#move_box,#yt_box{display:none!important}
.mir2{position:relative;max-width:340px;margin:0 auto;border-radius:14px;overflow:hidden;border:2px solid #fbbf2488;background:#000;min-height:90px}
.mir2 video,.mir2 canvas{width:100%;display:block}.mir2 canvas{position:absolute;left:0;top:0;height:100%}
.gradio-container ul.options,.gradio-container .options{z-index:9999!important}
.gradio-container .block{overflow:visible!important}
#paint{touch-action:none;display:block;margin:0 auto;width:100%;max-width:340px;background:#06101f;border:2px solid #f59e0b99;border-radius:14px;cursor:crosshair}
.fightbox .mir{position:relative;transform:scaleX(-1);max-width:340px;margin:0 auto;border-radius:14px;overflow:hidden;border:2px solid #22d3ee88;background:#000;min-height:120px}
.fightbox video,.fightbox canvas{width:100%;display:block}
.fightbox canvas{position:absolute;left:0;top:0;height:100%}
"""

# ===================== UI =====================
_LAUNCH_HAS_HEAD = "head" in inspect.signature(gr.Blocks.launch).parameters   # Gradio 6 moved head/css to launch()
_HEAD_OK = True
if _LAUNCH_HAS_HEAD:
    demo = gr.Blocks(title="OpenLearn Baby")
else:
    try:
        demo = gr.Blocks(title="OpenLearn Baby", head=HEAD_JS, css=CSS)
    except TypeError:
        demo = gr.Blocks(title="OpenLearn Baby"); _HEAD_OK = False

with demo:
    with gr.Row(equal_height=False):
        # ---------------- LEFT: 3D stage (robot + worlds), caption, small chat ----------------
        with gr.Column(scale=5, elem_id="left"):
            gr.HTML('<div id="stage">loading the 3D world…</div>', elem_id="stage_box")
            face = gr.HTML(F("idle", IDLE_CAP), elem_id="face_box")
            try:
                chatbot = gr.Chatbot(height=140, show_label=False, type="messages")
            except TypeError:
                chatbot = gr.Chatbot(height=140, show_label=False)
            with gr.Row():
                msg = gr.Textbox(show_label=False, placeholder="Ask or teach…  (e.g. 'do chain punch')", lines=1, max_lines=1,
                                 scale=8, container=False, elem_id="msg_box")
                ask = gr.Button("➤", size="sm", variant="primary", scale=1, min_width=40, elem_id="ask_btn")
                mic = gr.Button("🎙", size="sm", scale=1, min_width=40, elem_id="mic_btn")

        # ---------------- RIGHT: learning drop-down, then everything else ----------------
        with gr.Column(scale=4, elem_id="right"):
            mode_dd = gr.Dropdown(choices=MODES, value=MODE_TEXT, label="🎓 LEARNING", interactive=True, elem_id="mode_dd")
            info = gr.Markdown(elem_id="info")

            with gr.Column(visible=True, elem_id="grp_text") as grp_text:
                teach_box = gr.Textbox(label="📝 Teach me — type or paste any text here", lines=10, max_lines=30,
                                       placeholder="Write facts here. Many lines are fine. Then press ＋ Teach.",
                                       elem_id="teach_box")
                with gr.Row():
                    teach = gr.Button("＋ Teach", size="sm", variant="primary")
                    up = gr.UploadButton("📎 Read PDF/TXT (one or more)", file_types=[".pdf", ".txt", ".md"], file_count="multiple", size="sm")
                with gr.Row():
                    path_box = gr.Textbox(show_label=False, placeholder="…or a path on this machine, e.g. /content/book.pdf (file or folder)",
                                          lines=1, max_lines=1, scale=6, container=False)
                    path_btn = gr.Button("📂 Read path", size="sm", scale=2)

            with gr.Column(visible=True, elem_id="grp_cam") as grp_cam:
                # streaming works only with a webcam-only source
                cam = gr.Image(sources=["webcam"], streaming=True, type="numpy", show_label=False, height=190)
                with gr.Row():
                    name = gr.Textbox(show_label=False, placeholder="Name of what you show…", lines=1, max_lines=1,
                                      scale=6, container=False)
                    go = gr.Button("▶ Learn", size="sm", variant="primary", scale=2)
                    stop = gr.Button("■ Stop", size="sm", variant="stop", scale=2)

            with gr.Column(visible=True, elem_id="grp_draw") as grp_draw:
                gr.HTML('<canvas id="paint" width="300" height="300"></canvas>')
                with gr.Row():
                    paint_name = gr.Textbox(show_label=False, placeholder="Name of this drawing (cat, house…)", lines=1,
                                            max_lines=1, scale=5, container=False)
                    paint_btn = gr.Button("💾 Teach", size="sm", variant="primary", scale=2)
                    paint_clr = gr.Button("🧽 Clear", size="sm", scale=2)
                with gr.Row():
                    draw_name = gr.Textbox(show_label=False, placeholder="What should I draw?", lines=1, max_lines=1,
                                           scale=6, container=False)
                    draw_btn = gr.Button("🖌 Draw", size="sm", variant="primary", scale=2)
                board = gr.HTML(board_html())
                stroke_box = gr.Textbox(visible=False)

            with gr.Column(visible=True, elem_id="grp_sim") as grp_sim:
                sim_kind = gr.Radio([SIM_BODY, SIM_DRAW], value=SIM_BODY, label="What should I practise in the simulation world?")
                sim_lesson = gr.Dropdown(choices=lesson_choices(), value=LESSONS[0], allow_custom_value=True,
                                         label="📋 Lesson (walk · sit down · talk · horse stance · chain punch · or a move you taught)")
                with gr.Row():
                    sim_name = gr.Textbox(show_label=False, placeholder="Drawing to practise (empty = circle)", lines=1,
                                          max_lines=1, scale=6, container=False)
                    sim_btn = gr.Button("▶ Train in the 3D world", size="sm", variant="primary", scale=3)
                gens = gr.Slider(10, 120, value=40, step=10, label="Generations (practice rounds)")
                sim_view = gr.HTML(sim_card([], ["Pick a lesson and press Train."]))

            with gr.Column(visible=True, elem_id="grp_fight") as grp_fight:
                gr.HTML('<div class="fightbox"><div class="mir"><video id="fight_cam" autoplay muted playsinline></video>'
                        '<canvas id="fight_overlay"></canvas></div><div id="fight_status" class="cap">camera off</div></div>')
                with gr.Row():
                    cam_on = gr.Button("📷 Camera on", size="sm")
                    cam_off = gr.Button("Camera off", size="sm")
                with gr.Row():
                    move_name = gr.Textbox(show_label=False, placeholder="Name of the move (punch, kick, wave…)", lines=1,
                                           max_lines=1, scale=5, container=False)
                    rec = gr.Button("● Record", size="sm", variant="primary", scale=2)
                    endrec = gr.Button("■ Save", size="sm", variant="stop", scale=2)
                teach_mode = gr.Radio([TEACH_REFINE, TEACH_REPLACE], value=TEACH_REFINE, label="If I already know this move")
                parts = gr.CheckboxGroup(PART_NAMES, value=PART_NAMES, label="Which body parts should this teaching change?")
                with gr.Row():
                    move_dd = gr.Dropdown(choices=moves_list(), value=(moves_list() or [None])[0], label="📋 Learned moves (pick one)",
                                          info="Moves you recorded appear here", interactive=True, scale=5)
                    perform_btn = gr.Button("▶ Perform", size="sm", scale=2)
                speed = gr.Slider(0.3, 3.0, value=1.4, step=0.1, label="Speed (slow motion ↔ fast)")
                with gr.Row():
                    reteach = gr.Button("🔁 Re-teach", size="sm")
                    notexact = gr.Button("👎 Not exact — teach again", size="sm")
                    undo = gr.Button("↩ Undo", size="sm")
                with gr.Row():
                    arena_btn = gr.Button("🥊 Open fight arena (3D window)", size="sm", variant="primary")
                    mirror_btn = gr.Button("🪞 Live mirror (robot copies me)", size="sm")
                with gr.Accordion("🎞 Learn fighting from a YouTube link or a video file", open=False):
                    yt_url = gr.Textbox(show_label=False, placeholder="https://www.youtube.com/watch?v=…  (link to a fight / training video)",
                                        lines=1, max_lines=1)
                    yt_file = gr.File(label="…or upload a video file (mp4, webm, mov)", file_types=["video"], type="filepath")
                    with gr.Row():
                        yt_start = gr.Number(value=0, label="Start at (seconds)", precision=0, minimum=0)
                        yt_len = gr.Number(value=10, label="Length (2–20 s)", precision=0, minimum=2, maximum=20)
                        yt_rate = gr.Slider(0.2, 1.0, value=0.4, step=0.1, label="Watching speed")
                    yt_who = gr.Radio([WHO_ME, WHO_PARTNER], value=WHO_PARTNER, label="Who should learn it?")
                    yt_pick = gr.Radio([PICK_BIG, PICK_LEFT, PICK_RIGHT], value=PICK_BIG, label="Which fighter in the video?")
                    yt_name = gr.Textbox(show_label=False, placeholder="Name of this move (jab, bong sau, front kick…)", lines=1, max_lines=1)
                    yt_btn = gr.Button("▶ Watch the video and learn", size="sm", variant="primary")
                gr.HTML('<div class="fightbox"><div class="mir2"><video id="yt_vid" muted playsinline></video><canvas id="yt_overlay"></canvas></div>'
                        '<div id="yt_status" class="cap" style="font-size:.8rem">no video yet</div></div>')
                move_box = gr.Textbox(elem_id="move_box", show_label=False)
                yt_box = gr.Textbox(elem_id="yt_box", show_label=False)

            soon_md = gr.Markdown("")

            with gr.Row():
                like = gr.Button("👍", size="sm", min_width=36)
                dislike = gr.Button("👎", size="sm", min_width=36)
                fix = gr.Button("✏️ Fix", size="sm", min_width=36)
                pic = gr.UploadButton("🖼", file_types=["image"], size="sm", min_width=36)
            with gr.Row():
                voice_on = gr.Checkbox(value=True, label="🔊 Voice", container=False, min_width=70, elem_id="voice_on")
                talk_mode = gr.Checkbox(value=False, label="🗣 Talk mode", container=False, min_width=90, elem_id="talk_mode")
                listen_learn = gr.Checkbox(value=False, label="👂 Learn by listening", container=False, min_width=130,
                                           elem_id="listen_learn")
                qwen_on = gr.Checkbox(value=True, label="🤖 Qwen voice", container=False, min_width=100, elem_id="qwen_on")
            gr.HTML('<div id="listen_status" class="cap" style="font-size:.78rem"></div>')

            gr.Markdown("👂 **Learn by listening** = I keep listening and remember what I hear (ask me “what did you learn?”) · "
                        "🗣 **Talk mode** = a voice chat: I greet you, listen, answer, and listen again.")
            with gr.Accordion("🧠 Memory", open=False) as acc:
                with gr.Row():
                    mem_search = gr.Textbox(show_label=False, placeholder="Search memory…", lines=1, max_lines=1,
                                            scale=6, container=False)
                    mem_go = gr.Button("🔍", size="sm", scale=1, min_width=40)
                memory_box = gr.Textbox(value=memory_list(), lines=7, show_label=False, interactive=False)
                with gr.Row():
                    bk = gr.Button("⬇ Backup", size="sm")
                    rs = gr.UploadButton("⬆ Restore", file_types=[".db"], size="sm")
                    clr = gr.Button("🗑 Text", size="sm", variant="stop")
                    clrv = gr.Button("🗑 Views", size="sm", variant="stop")
                bkfile = gr.File(show_label=False)

    learning = gr.State(False)
    voice = gr.Textbox(elem_id="voice_box", interactive=False, show_label=False)

    # ---- learning drop-down (also switches the 3D world) ----
    mode_dd.change(None, [mode_dd], None, queue=False, js="(m)=>{window.applyMode&&window.applyMode();}")
    mode_dd.change(mode_change, [mode_dd, move_dd, sim_lesson], [soon_md, face, info, move_dd, sim_lesson])
    sim_kind.change(sim_world, [sim_kind], [face])
    qwen_on.change(qwen_toggle, [qwen_on], [info])

    # ---- chat + microphone ----
    ask.click(chat_send, [msg, chatbot, talk_mode], [chatbot, msg, face, voice])
    msg.submit(chat_send, [msg, chatbot, talk_mode], [chatbot, msg, face, voice])
    mic.click(None, None, None, js="()=>{window.toggleVoiceChat&&window.toggleVoiceChat();}")
    teach.click(teach_ui, [teach_box], [info, face, voice, teach_box])
    like.click(lambda: rate_any("good"), outputs=[info, face, voice])
    dislike.click(lambda: rate_any("bad"), outputs=[info, face, voice])
    fix.click(fix_any, [msg], [info, face, voice, msg])
    up.upload(import_doc, [up], [info, face, memory_box, voice])
    path_btn.click(import_path_ui, [path_box], [info, face, memory_box, voice])
    path_box.submit(import_path_ui, [path_box], [info, face, memory_box, voice])
    pic.upload(photo_file, [pic], [info, face, voice])

    # ---- camera (seeing lessons) ----
    stream_args = dict(fn=live_step, inputs=[cam, name, learning, mode_dd], outputs=[info, face, voice], show_progress="hidden")
    try:
        cam.stream(**stream_args, stream_every=0.3, time_limit=None, concurrency_limit=1)
    except TypeError:
        cam.stream(**stream_args)
    go.click(start_learning, [name, mode_dd], [learning, info, face, voice])
    stop.click(stop_learning, [name, mode_dd], [learning, cam, info, face, voice],
               js="(n,m)=>{window.stopCam&&window.stopCam();return [n,m];}")

    # ---- canvas drawing (the robot draws it on the glass canvas in 3D) ----
    paint_btn.click(paint_save, [paint_name, stroke_box], [info, face, voice], js="(n,s)=>[n,window.getStrokes()]")
    paint_clr.click(None, None, None, js="()=>{window.clearPaint&&window.clearPaint();}")
    draw_btn.click(draw_ui, [draw_name], [board, info, face, voice])
    draw_name.submit(draw_ui, [draw_name], [board, info, face, voice])

    # ---- simulation world ----
    sim_btn.click(sim_train, [sim_kind, sim_lesson, sim_name, gens], [info, sim_view, face, voice])

    # ---- body + finger moves (fighting ring) with re-teaching ----
    cam_on.click(None, None, None, js="()=>{window.fightCamStart&&window.fightCamStart();}")
    cam_off.click(None, None, None, js="()=>{window.fightCamStop&&window.fightCamStop();}")
    rec.click(move_arm, [move_name, teach_mode, parts], [info, face, voice],
              js="(n,m,p)=>{if(n&&n.trim()&&window.armMove)window.armMove();return [n,m,p];}")
    endrec.click(None, None, None, js="()=>{window.endMove&&window.endMove();}")
    move_box.change(move_save, [move_name, move_box, teach_mode, parts, speed], [info, face, voice, move_dd])
    arena_btn.click(open_arena_ui, None, [info, face, voice])
    mirror_btn.click(None, None, None, js="()=>{window.mirrorToggle&&window.mirrorToggle();}")
    yt_btn.click(yt_learn_ui, [yt_url, yt_file, yt_start, yt_len, yt_who, yt_pick, yt_name, yt_rate], [info, face, voice])
    yt_box.change(yt_save, [yt_box, yt_name, yt_who, teach_mode, parts, speed], [info, face, voice, move_dd])
    perform_btn.click(perform_ui, [move_dd, speed], [info, face, voice])
    reteach.click(lambda n: reteach_prepare(n, False), [move_dd], [move_name, teach_mode, info, face, voice]).then(
        None, None, None, js="()=>{window.armMove&&window.armMove();}")
    notexact.click(lambda n: reteach_prepare(n, True), [move_dd], [move_name, teach_mode, info, face, voice]).then(
        None, None, None, js="()=>{window.armMove&&window.armMove();}")
    undo.click(undo_move, [move_dd], [info, face, voice, move_dd])

    # ---- memory ----
    mem_go.click(memory_list, [mem_search], memory_box)
    mem_search.submit(memory_list, [mem_search], memory_box)
    try:
        acc.expand(lambda: memory_list(""), outputs=memory_box)
    except Exception:
        pass
    bk.click(make_backup, outputs=bkfile)
    rs.upload(restore_backup, [rs], [info, memory_box, face])
    clr.click(forget_all, outputs=[info, memory_box, face])
    clrv.click(forget_visual, outputs=[info, face, memory_box])

    # ---- voice + sounds (runs in the browser) and auto-return to the normal face ----
    voice.change(None, [voice, voice_on], None, queue=False, show_progress="hidden",
                 js="(v,on)=>{window.onVoice&&window.onVoice(v,on);}")
    if not _HEAD_OK:
        demo.load(None, None, None, js="()=>{" + JS_LIB + "}")
    if hasattr(gr, "Timer"):
        gr.Timer(1.5).tick(idle_tick, None, face)

if __name__ == "__main__":
    print("👉 Open the PUBLIC (…gradio.live) URL in a NEW browser tab. Camera and microphone are blocked inside Colab's frame.")
    _kw = dict(share=True, inline=False, debug=True, allowed_paths=[str(DATA_DIR)])
    if _LAUNCH_HAS_HEAD:
        demo.queue().launch(head=HEAD_JS, css=CSS, **_kw)
    else:
        demo.queue().launch(**_kw)

👉 Open the PUBLIC (…gradio.live) URL in a NEW browser tab. Camera and microphone are blocked inside Colab's frame.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://88f37005580542ee60.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://88f37005580542ee60.gradio.live
